# SonarNet-VN**Hệ thống giám sát tuân thủ quy định chống khai thác thuỷ sản bất hợp pháp, không khai báo và không theo quy định**Hợp nhất ảnh vệ tinh radar khẩu độ tổng hợp với tín hiệu giám sát hành trình tàu cá.Cuộc thi Sáng tạo trẻ Quốc gia trong lĩnh vực Trí tuệ nhân tạo năm 2026 — Bảng CChủ đề: **AI cho Phát triển kinh tế – xã hội**---## Bài toánTháng 10 năm 2017, Uỷ ban châu Âu áp dụng cảnh báo thẻ vàng đối với thuỷ sản Việt Nam vì chưa kiểm soát được hoạt động khai thác bất hợp pháp. Gần một thập kỷ sau, cảnh báo vẫn còn hiệu lực. Chính phủ đặt mục tiêu gỡ thẻ vàng trong năm 2026.Đây trước hết là bài toán kinh tế. Thuỷ sản là ngành xuất khẩu chủ lực, thị trường châu Âu có giá trị cao. Cảnh báo thẻ vàng làm tăng chi phí kiểm tra, kéo dài thông quan và giảm sức cạnh tranh của toàn ngành. Người chịu ảnh hưởng cuối cùng là hàng trăm nghìn ngư dân.Bốn nhóm khuyến nghị của Uỷ ban châu Âu tập trung vào quản lý đội tàu, thực thi pháp luật, truy xuất nguồn gốc và **giám sát**. Đề tài nhắm vào nhóm cuối.## Vì sao giám sát vẫn là điểm nghẽnViệt Nam có khoảng 80.300 tàu cá đăng ký, trong đó **99,6% tàu từ 15 mét trở lên đã lắp thiết bị giám sát hành trình**. Giai đoạn trang bị thiết bị về cơ bản đã hoàn tất.Điểm nghẽn nằm ở chỗ khác: **thiết bị được lắp nhưng không phải lúc nào cũng phát tín hiệu**. Khi tín hiệu biến mất, cơ quan quản lý không còn cách nào biết phương tiện đang ở đâu — đúng vào lúc cần biết nhất. Đây là giới hạn cố hữu của mọi phương thức giám sát chỉ dựa trên tín hiệu do chính phương tiện phát ra.## Hướng giải quyếtẢnh vệ tinh radar khẩu độ tổng hợp ghi nhận mọi vật thể phản xạ sóng radar trên mặt biển, kể cả khi nhiều mây và vào ban đêm, không phụ thuộc việc phương tiện có phát tín hiệu hay không.Đối chiếu tập phương tiện *nhìn thấy trên ảnh* với tập phương tiện *đang phát tín hiệu*, phần chênh lệch chính là các phương tiện mất kết nối. Hệ thống không kết luận nguyên nhân, mà cung cấp danh sách trường hợp cần rà soát kèm bằng chứng ảnh và toạ độ.## Kiến trúc bốn tầng| Tầng | Chức năng | Kỹ thuật ||---|---|---|| 1 | Thu nhận và tiền xử lý ảnh radar | Hiệu chuẩn bức xạ, khử nhiễu đốm, chỉnh hình học || 2 | Phát hiện phương tiện | YOLO trên hai GPU, hoặc Faster R-CNN dự phòng || 3 | Hợp nhất ảnh radar và tín hiệu | Làm trơn Rauch–Tung–Striebel, ghép cặp Hungarian || 4 | Phân tích hành vi và cảnh báo | Tăng cường gradient trên đặc trưng động học |## Ba trạng thái định danh| Trạng thái | Ý nghĩa ||---|---|| `AIS_OK` | Phát tín hiệu đầy đủ và trung thực || `AIS_MISMATCH` | Có phát tín hiệu nhưng kích thước khai báo sai lệch lớn || `DARK` | Không phát tín hiệu — cần rà soát |---## Trước khi chạyMở bảng **Session options** bên phải và đặt:| Thiết lập | Giá trị ||---|---|| Accelerator | **GPU T4 × 2** || Internet | **On** (khuyến nghị) |**Về thiết lập Internet.** Khi bật, hệ thống cài Ultralytics và tải trọng số YOLO đã huấn luyện trước, cho kết quả tốt nhất. Khi tắt, notebook **vẫn chạy đầy đủ mọi bước** bằng phương án dự phòng Faster R-CNN của Torchvision — thư viện này có sẵn trong ảnh Python của Kaggle.Sau đó bấm **Run All**. Toàn bộ quy trình chạy tuần tự từ đầu đến cuối, không cần can thiệp.

---# Bước 0 — Kiểm tra môi trườngXác nhận phần cứng và các thư viện sẵn có trước khi bắt đầu.

In [ ]:
import os, sys, platform, subprocessfrom pathlib import Pathprint("=" * 66)print("MÔI TRƯỜNG TÍNH TOÁN")print("=" * 66)print(f"Python  : {platform.python_version()}")print(f"Hệ điều hành: {platform.platform()}")try:    import torch    print(f"PyTorch : {torch.__version__}")    if torch.cuda.is_available():        n = torch.cuda.device_count()        print(f"CUDA    : {torch.version.cuda} | số GPU: {n}")        for i in range(n):            p = torch.cuda.get_device_properties(i)            print(f"   GPU {i}: {p.name} — {p.total_memory / 1024**3:.1f} GB")        if n >= 2:            print("\n   Đủ hai GPU: sẽ huấn luyện phân tán trên cả hai card.")        else:            print("\n   Chỉ có một GPU. Hệ thống vẫn chạy đầy đủ, thời gian dài hơn.")    else:        print("CUDA    : không khả dụng — sẽ chạy trên CPU (chậm hơn nhiều)")except Exception as exc:    print(f"Không kiểm tra được PyTorch: {exc}")print("\n" + "-" * 66)print("THƯ VIỆN")print("-" * 66)import importlib.utilfor name, role in [    ("numpy", "bắt buộc"), ("scipy", "bắt buộc"),    ("sklearn", "bắt buộc"), ("matplotlib", "bắt buộc"),    ("torchvision", "bắt buộc — phương án phát hiện dự phòng"),    ("ultralytics", "tuỳ chọn — phương án phát hiện chính"),    ("xgboost", "tuỳ chọn — phân loại hành vi"),    ("folium", "tuỳ chọn — bản đồ giám sát"),    ("cv2", "tuỳ chọn — đọc ghi ảnh nhanh hơn"),    ("pandas", "tuỳ chọn — kết xuất CSV"),]:    ok = importlib.util.find_spec(name) is not None    print(f"   {'có   ' if ok else 'thiếu'} {name:<14} {role}")

### Cài đặt thư viện tuỳ chọnÔ lệnh dưới đây cài Ultralytics nếu có kết nối mạng. Khi không cài được, hệ thống tự chuyển sang Faster R-CNN và **vẫn chạy đủ mọi bước** — không có bước nào bị bỏ qua.

In [ ]:
import importlib.util, subprocess, sysdef ensure(pkg: str, pip_name: str = None) -> bool:    if importlib.util.find_spec(pkg) is not None:        print(f"   {pkg}: đã có sẵn")        return True    name = pip_name or pkg    print(f"   {pkg}: đang cài đặt ...")    try:        subprocess.run(            [sys.executable, "-m", "pip", "install", "-q", name],            check=True, timeout=600,        )        ok = importlib.util.find_spec(pkg) is not None        print(f"   {pkg}: {'cài đặt xong' if ok else 'cài đặt không thành công'}")        return ok    except Exception as exc:        print(f"   {pkg}: không cài được ({type(exc).__name__}) — dùng phương án thay thế")        return Falseprint("Cài đặt thư viện tuỳ chọn")print("-" * 66)HAS_ULTRALYTICS = ensure("ultralytics")HAS_XGBOOST     = ensure("xgboost")HAS_FOLIUM      = ensure("folium")print("-" * 66)if HAS_ULTRALYTICS:    print("Phương án phát hiện: Ultralytics YOLO (khai thác cả hai GPU)")else:    print("Phương án phát hiện: Torchvision Faster R-CNN (dự phòng, không cần mạng)")

---# Bước 1 — Ghi mã nguồnCác ô lệnh dưới đây ghi toàn bộ gói `sonarnet` ra đĩa. Nội dung giữ nguyên như trong kho mã nguồn trên GitHub, nên notebook này và kho mã luôn nhất quán với nhau.Gói gồm chín nhóm mô-đun, tổng cộng 28 tệp.

In [ ]:
from pathlib import Pathimport sysWORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()SRC_DIR = WORK_DIR / "src"SRC_DIR.mkdir(parents=True, exist_ok=True)if str(SRC_DIR) not in sys.path:    sys.path.insert(0, str(SRC_DIR))print(f"Thư mục mã nguồn: {SRC_DIR}")

### Nền tảng — cấu hình và tiện ích

In [ ]:
# Ghi mã nguồn ra đĩa. Nội dung giữ nguyên như trong kho mã nguồn._FILES = {}_FILES['sonarnet/__init__.py'] = "\"\"\"SonarNet-VN — Hệ thống giám sát tuân thủ quy định chống khai thác IUU.\n\nHợp nhất ảnh vệ tinh radar khẩu độ tổng hợp với tín hiệu giám sát hành trình\nđể phát hiện phương tiện khai thác thuỷ sản mất kết nối thiết bị, phục vụ công\ntác gỡ cảnh báo thẻ vàng của Uỷ ban châu Âu và bảo vệ sinh kế ngư dân.\n\"\"\"\n\n__version__ = \"1.0.0\"\n\nfrom .config import CFG, RunConfig, describe  # noqa: F401\nfrom .utils import get_logger, probe_devices, set_seed, timed  # noqa: F401\n\n__all__ = [\n    \"CFG\",\n    \"RunConfig\",\n    \"describe\",\n    \"get_logger\",\n    \"probe_devices\",\n    \"set_seed\",\n    \"timed\",\n    \"__version__\",\n]\n"_FILES['sonarnet/config.py'] = "\"\"\"Cấu hình tập trung cho toàn bộ pipeline SonarNet-VN.\n\nMọi tham số điều khiển hành vi của hệ thống đều nằm ở đây. Notebook trên Kaggle\nchỉ cần sửa các trường trong ``CFG`` là thay đổi được toàn bộ quy trình.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom dataclasses import dataclass, field, asdict\nfrom pathlib import Path\nfrom typing import List, Optional\n\n\ndef _default_root() -> Path:\n    \"\"\"Thư mục làm việc: ưu tiên /kaggle/working khi chạy trên Kaggle.\"\"\"\n    if Path(\"/kaggle/working\").exists():\n        return Path(\"/kaggle/working/sonarnet_run\")\n    return Path.cwd() / \"sonarnet_run\"\n\n\n@dataclass\nclass GeoConfig:\n    \"\"\"Khung địa lý của vùng biển thí điểm.\n\n    Mặc định là một ô biển ngoài khơi Nam Trung Bộ. Toạ độ chỉ dùng để gắn\n    hệ quy chiếu cho ảnh mô phỏng và cho bản đồ trình diễn.\n    \"\"\"\n\n    lat_min: float = 9.20\n    lat_max: float = 10.10\n    lon_min: float = 108.40\n    lon_max: float = 109.40\n    # Độ phân giải mặt đất của Sentinel-1 GRD (mét trên mỗi điểm ảnh)\n    pixel_spacing_m: float = 10.0\n\n\n@dataclass\nclass DataConfig:\n    \"\"\"Tham số sinh và nạp dữ liệu.\"\"\"\n\n    # Kích thước một cảnh ảnh (điểm ảnh)\n    scene_size: int = 640\n    # Số cảnh cho từng tập\n    n_train_scenes: int = 260\n    n_val_scenes: int = 60\n    n_test_scenes: int = 60\n    # Số tàu trên mỗi cảnh\n    ships_per_scene: tuple = (3, 14)\n    # Tỉ lệ tàu chủ động ngắt AIS\n    dark_vessel_ratio: float = 0.28\n    # Tỉ lệ cảnh có vùng đất liền / đảo\n    land_probability: float = 0.22\n    # Bật bóng ma phương vị (azimuth ambiguity) - một giả tượng đặc trưng của SAR\n    azimuth_ambiguity: bool = True\n    # Nhiễu vị trí của bản ghi AIS (mét)\n    ais_position_noise_m: float = 45.0\n    # Khoảng thời gian giữa hai bản ghi AIS liên tiếp (giây)\n    ais_interval_s: tuple = (12, 180)\n    # Nửa cửa sổ thời gian lấy AIS quanh thời điểm chụp (giây)\n    ais_window_s: int = 900\n    # Thư mục dữ liệu Kaggle nếu người dùng có gắn bộ dữ liệu thật\n    kaggle_input_dir: str = \"/kaggle/input\"\n\n\n@dataclass\nclass DetectConfig:\n    \"\"\"Tham số huấn luyện mô hình phát hiện tàu.\"\"\"\n\n    # \"ultralytics\" | \"torchvision\" | \"auto\"\n    backend: str = \"auto\"\n    # Mô hình YOLO dùng khi backend là ultralytics\n    yolo_model: str = \"yolo11n.pt\"\n    imgsz: int = 640\n    epochs: int = 40\n    batch_size: int = 32\n    workers: int = 4\n    lr0: float = 0.01\n    patience: int = 15\n    # Danh sách GPU sử dụng. [0, 1] khai thác đủ hai card T4 trên Kaggle.\n    devices: List[int] = field(default_factory=lambda: [0, 1])\n    conf_threshold: float = 0.25\n    iou_threshold: float = 0.50\n\n\n@dataclass\nclass FusionConfig:\n    \"\"\"Tham số hợp nhất ảnh radar và tín hiệu AIS.\"\"\"\n\n    # Ngưỡng khoảng cách tối đa để chấp nhận một cặp ghép (mét)\n    max_match_distance_m: float = 500.0\n    # Độ lệch chuẩn gia tốc nhiễu của mô hình động học, mét trên giây bình phương.\n    # Giá trị được chọn theo quán tính thực tế của phương tiện đường thuỷ: trong\n    # một khoảng mười lăm phút, tàu biển gần như giữ nguyên hướng và tốc độ, biến\n    # động chủ yếu đến từ sóng, dòng chảy và thao tác bánh lái. Đặt tham số này\n    # quá lớn sẽ khiến mô hình động học mất tác dụng ràng buộc và bộ làm trơn\n    # thoái hoá về mức của phép nội suy hai điểm.\n    kalman_process_noise: float = 0.01\n    # Độ lệch chuẩn sai số vị trí của bản ghi AIS, mét\n    kalman_measurement_noise: float = 45.0\n    # Ngưỡng sai lệch kích thước để gán nhãn \"AIS không phù hợp\" (tỉ lệ)\n    size_mismatch_ratio: float = 0.55\n\n\n@dataclass\nclass BehaviorConfig:\n    \"\"\"Tham số phân loại hành vi hoạt động của phương tiện.\"\"\"\n\n    classes: List[str] = field(\n        default_factory=lambda: [\"qua_canh\", \"cau\", \"keo_luoi\", \"neo_dau\"]\n    )\n    # Độ dài chuỗi quỹ đạo dùng để trích đặc trưng (giờ)\n    track_hours: float = 18.0\n    # Bước thời gian lấy mẫu quỹ đạo (phút)\n    track_step_min: float = 10.0\n    n_tracks_per_class: int = 320\n    test_size: float = 0.25\n    # \"xgboost\" | \"sklearn\" | \"auto\"\n    backend: str = \"auto\"\n\n\n@dataclass\nclass RunConfig:\n    \"\"\"Cấu hình tổng thể của một lần chạy.\"\"\"\n\n    root: Path = field(default_factory=_default_root)\n    seed: int = 20260914\n    # Chế độ nhanh: giảm quy mô để chạy thử toàn tuyến trong vài phút\n    quick_mode: bool = False\n    # Cho phép dùng nhiều GPU\n    multi_gpu: bool = True\n\n    geo: GeoConfig = field(default_factory=GeoConfig)\n    data: DataConfig = field(default_factory=DataConfig)\n    detect: DetectConfig = field(default_factory=DetectConfig)\n    fusion: FusionConfig = field(default_factory=FusionConfig)\n    behavior: BehaviorConfig = field(default_factory=BehaviorConfig)\n\n    # ---- Đường dẫn dẫn xuất -------------------------------------------------\n    @property\n    def dir_data(self) -> Path:\n        return self.root / \"data\"\n\n    @property\n    def dir_yolo(self) -> Path:\n        return self.root / \"data\" / \"yolo\"\n\n    @property\n    def dir_runs(self) -> Path:\n        return self.root / \"runs\"\n\n    @property\n    def dir_results(self) -> Path:\n        return self.root / \"results\"\n\n    @property\n    def dir_figures(self) -> Path:\n        return self.root / \"results\" / \"figures\"\n\n    def make_dirs(self) -> None:\n        for d in (\n            self.root,\n            self.dir_data,\n            self.dir_yolo,\n            self.dir_runs,\n            self.dir_results,\n            self.dir_figures,\n        ):\n            d.mkdir(parents=True, exist_ok=True)\n\n    def apply_quick_mode(self) -> None:\n        \"\"\"Thu nhỏ quy mô để kiểm tra toàn tuyến nhanh.\"\"\"\n        self.quick_mode = True\n        self.data.n_train_scenes = 60\n        self.data.n_val_scenes = 20\n        self.data.n_test_scenes = 20\n        self.detect.epochs = 8\n        self.detect.batch_size = 16\n        self.behavior.n_tracks_per_class = 120\n\n    def to_json(self, path: Optional[Path] = None) -> str:\n        payload = asdict(self)\n        payload[\"root\"] = str(self.root)\n        text = json.dumps(payload, indent=2, ensure_ascii=False, default=str)\n        if path is not None:\n            Path(path).write_text(text, encoding=\"utf-8\")\n        return text\n\n\n# Đối tượng cấu hình dùng chung cho toàn pipeline\nCFG = RunConfig()\n\n\ndef describe(cfg: RunConfig = CFG) -> str:\n    \"\"\"Tóm tắt cấu hình dưới dạng văn bản ngắn để in ra notebook.\"\"\"\n    lines = [\n        f\"Thư mục làm việc      : {cfg.root}\",\n        f\"Hạt giống ngẫu nhiên  : {cfg.seed}\",\n        f\"Chế độ nhanh          : {'CÓ' if cfg.quick_mode else 'KHÔNG'}\",\n        f\"Số cảnh huấn luyện    : {cfg.data.n_train_scenes}\",\n        f\"Số cảnh kiểm định     : {cfg.data.n_val_scenes}\",\n        f\"Số cảnh kiểm tra      : {cfg.data.n_test_scenes}\",\n        f\"Kích thước cảnh       : {cfg.data.scene_size} x {cfg.data.scene_size} điểm ảnh\",\n        f\"Tỉ lệ tàu ngắt AIS    : {cfg.data.dark_vessel_ratio:.0%}\",\n        f\"Số chu kỳ huấn luyện  : {cfg.detect.epochs}\",\n        f\"GPU sử dụng           : {cfg.detect.devices}\",\n        f\"Ngưỡng ghép cặp       : {cfg.fusion.max_match_distance_m:.0f} m\",\n    ]\n    return \"\\n\".join(lines)\n"_FILES['sonarnet/utils.py'] = "\"\"\"Tiện ích dùng chung: hạt giống ngẫu nhiên, thiết bị tính toán, ghi nhật ký.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport os\nimport random\nimport subprocess\nimport sys\nimport time\nfrom contextlib import contextmanager\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional\n\nimport numpy as np\n\nLOG_FORMAT = \"%(asctime)s | %(levelname)-7s | %(name)-22s | %(message)s\"\n_CONFIGURED = False\n\n\ndef get_logger(name: str = \"sonarnet\") -> logging.Logger:\n    \"\"\"Trả về logger đã cấu hình sẵn, an toàn khi gọi nhiều lần.\"\"\"\n    global _CONFIGURED\n    if not _CONFIGURED:\n        handler = logging.StreamHandler(sys.stdout)\n        handler.setFormatter(logging.Formatter(LOG_FORMAT, datefmt=\"%H:%M:%S\"))\n        root = logging.getLogger(\"sonarnet\")\n        root.setLevel(logging.INFO)\n        root.handlers.clear()\n        root.addHandler(handler)\n        root.propagate = False\n        _CONFIGURED = True\n    return logging.getLogger(name if name.startswith(\"sonarnet\") else f\"sonarnet.{name}\")\n\n\nLOG = get_logger()\n\n\n# ---------------------------------------------------------------------------\n# Tính tái lập\n# ---------------------------------------------------------------------------\ndef set_seed(seed: int) -> np.random.Generator:\n    \"\"\"Đặt hạt giống cho toàn bộ thư viện và trả về bộ sinh số của NumPy.\"\"\"\n    random.seed(seed)\n    np.random.seed(seed % (2**32 - 1))\n    os.environ[\"PYTHONHASHSEED\"] = str(seed)\n    try:\n        import torch\n\n        torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic = False\n        torch.backends.cudnn.benchmark = True\n    except Exception:  # pragma: no cover - torch luôn có trên Kaggle\n        pass\n    return np.random.default_rng(seed)\n\n\n# ---------------------------------------------------------------------------\n# Thiết bị tính toán\n# ---------------------------------------------------------------------------\n@dataclass\nclass DeviceInfo:\n    n_gpu: int\n    names: List[str]\n    total_memory_gb: List[float]\n    cuda_available: bool\n    torch_version: str\n    kind: str = \"cpu\"   # \"cuda\" | \"mps\" | \"cpu\"\n\n    @property\n    def summary(self) -> str:\n        if self.kind == \"cpu\" or self.n_gpu == 0:\n            return \"Không phát hiện GPU. Pipeline sẽ chạy trên CPU (chậm hơn nhiều).\"\n        rows = [f\"PyTorch {self.torch_version} | Loại thiết bị: {self.kind} | Số GPU khả dụng: {self.n_gpu}\"]\n        for i, (nm, mem) in enumerate(zip(self.names, self.total_memory_gb)):\n            rows.append(f\"  GPU {i}: {nm} — {mem:.1f} GB\" if mem > 0 else f\"  GPU {i}: {nm}\")\n        if self.kind == \"cuda\" and self.n_gpu >= 2:\n            rows.append(\"  Cấu hình hai GPU hợp lệ: sẽ huấn luyện phân tán trên cả hai card.\")\n        if self.kind == \"mps\":\n            rows.append(\"  Apple Metal Performance Shaders — huấn luyện chạy trên GPU tích hợp của máy.\")\n        return \"\\n\".join(rows)\n\n\ndef probe_devices() -> DeviceInfo:\n    \"\"\"Kiểm tra GPU khả dụng: CUDA, MPS (Apple Metal), rồi mới đến CPU.\"\"\"\n    try:\n        import torch\n    except Exception:\n        return DeviceInfo(0, [], [], False, \"không rõ\", \"cpu\")\n\n    if torch.cuda.is_available():\n        n = torch.cuda.device_count()\n        names, mems = [], []\n        for i in range(n):\n            props = torch.cuda.get_device_properties(i)\n            names.append(props.name)\n            mems.append(props.total_memory / (1024**3))\n        return DeviceInfo(n, names, mems, True, torch.__version__, \"cuda\")\n\n    if hasattr(torch.backends, \"mps\") and torch.backends.mps.is_available():\n        return DeviceInfo(1, [\"Apple Metal (MPS)\"], [0.0], False, torch.__version__, \"mps\")\n\n    return DeviceInfo(0, [], [], False, torch.__version__, \"cpu\")\n\n\ndef torch_device(index: int = 0):\n    \"\"\"Trả về torch.device dựa trên phần cứng thực có.\"\"\"\n    import torch\n    info = probe_devices()\n    if info.kind == \"cuda\":\n        return torch.device(f\"cuda:{index}\")\n    if info.kind == \"mps\":\n        return torch.device(\"mps\")\n    return torch.device(\"cpu\")\n\n\ndef resolve_devices(requested: List[int], allow_multi: bool = True) -> List[int]:\n    \"\"\"Lọc danh sách GPU yêu cầu theo số card thực có.\"\"\"\n    info = probe_devices()\n    if info.n_gpu == 0:\n        return []\n    usable = [d for d in requested if d < info.n_gpu]\n    if not usable:\n        usable = [0]\n    if not allow_multi:\n        usable = usable[:1]\n    return usable\n\n\ndef gpu_utilisation() -> Optional[str]:\n    \"\"\"Đọc mức sử dụng GPU qua nvidia-smi, trả về None nếu không có.\"\"\"\n    try:\n        out = subprocess.run(\n            [\n                \"nvidia-smi\",\n                \"--query-gpu=index,name,utilization.gpu,memory.used,memory.total\",\n                \"--format=csv,noheader,nounits\",\n            ],\n            capture_output=True,\n            text=True,\n            timeout=20,\n        )\n        if out.returncode != 0:\n            return None\n        rows = []\n        for line in out.stdout.strip().splitlines():\n            idx, name, util, used, total = [x.strip() for x in line.split(\",\")]\n            rows.append(\n                f\"  GPU {idx} ({name}): tải {util}% | bộ nhớ {used}/{total} MiB\"\n            )\n        return \"\\n\".join(rows) if rows else None\n    except Exception:\n        return None\n\n\n# ---------------------------------------------------------------------------\n# Đo thời gian\n# ---------------------------------------------------------------------------\n@contextmanager\ndef timed(label: str, logger: Optional[logging.Logger] = None):\n    \"\"\"Đo thời gian một khối lệnh và ghi nhật ký.\"\"\"\n    log = logger or LOG\n    log.info(\"▶ %s ...\", label)\n    t0 = time.perf_counter()\n    try:\n        yield\n    finally:\n        dt = time.perf_counter() - t0\n        if dt < 90:\n            log.info(\"✔ %s — hoàn tất trong %.1f giây\", label, dt)\n        else:\n            log.info(\"✔ %s — hoàn tất trong %.1f phút\", label, dt / 60.0)\n\n\n# ---------------------------------------------------------------------------\n# Vào ra\n# ---------------------------------------------------------------------------\ndef save_json(obj: Any, path: Path) -> Path:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n\n    def _default(o):\n        if isinstance(o, (np.integer,)):\n            return int(o)\n        if isinstance(o, (np.floating,)):\n            return float(o)\n        if isinstance(o, np.ndarray):\n            return o.tolist()\n        if isinstance(o, Path):\n            return str(o)\n        return str(o)\n\n    path.write_text(\n        json.dumps(obj, indent=2, ensure_ascii=False, default=_default), encoding=\"utf-8\"\n    )\n    return path\n\n\ndef load_json(path: Path) -> Any:\n    return json.loads(Path(path).read_text(encoding=\"utf-8\"))\n\n\ndef package_available(name: str) -> bool:\n    \"\"\"Kiểm tra một gói Python có nạp được hay không.\"\"\"\n    import importlib.util\n\n    try:\n        return importlib.util.find_spec(name) is not None\n    except Exception:\n        return False\n\n\ndef format_table(rows: List[Dict[str, Any]], headers: Optional[List[str]] = None) -> str:\n    \"\"\"Định dạng danh sách bản ghi thành bảng văn bản căn cột.\"\"\"\n    if not rows:\n        return \"(không có dữ liệu)\"\n    headers = headers or list(rows[0].keys())\n    widths = {h: len(str(h)) for h in headers}\n    str_rows = []\n    for r in rows:\n        sr = {}\n        for h in headers:\n            v = r.get(h, \"\")\n            s = f\"{v:.4f}\" if isinstance(v, float) else str(v)\n            sr[h] = s\n            widths[h] = max(widths[h], len(s))\n        str_rows.append(sr)\n\n    sep = \"-+-\".join(\"-\" * widths[h] for h in headers)\n    out = [\" | \".join(str(h).ljust(widths[h]) for h in headers), sep]\n    for sr in str_rows:\n        out.append(\" | \".join(sr[h].ljust(widths[h]) for h in headers))\n    return \"\\n\".join(out)\n"for _rel, _content in _FILES.items():    _p = SRC_DIR / _rel    _p.parent.mkdir(parents=True, exist_ok=True)    _p.write_text(_content, encoding='utf-8')print(f'Đã ghi {len(_FILES)} tệp mã nguồn.')for _rel in _FILES:    print('   ', _rel)

### Tầng dữ liệu — hệ toạ độ và bộ dựng ảnh radar

In [ ]:
# Ghi mã nguồn ra đĩa. Nội dung giữ nguyên như trong kho mã nguồn._FILES = {}_FILES['sonarnet/data/__init__.py'] = "\"\"\"Lớp dữ liệu: mô phỏng cảnh ảnh radar, dòng AIS và quỹ đạo phương tiện.\"\"\"\n\nfrom .dataset import build_dataset, load_image, read_ais, read_scene_meta  # noqa: F401\nfrom .geo import SceneGeoReference, haversine_m  # noqa: F401\nfrom .sar_render import SARRenderer, ShipFootprint  # noqa: F401\nfrom .simulator import AIS_MISMATCH, AIS_OK, DARK, Scene, SceneSimulator, Vessel  # noqa: F401\nfrom .tracks import Track, generate_track, generate_track_dataset  # noqa: F401\n"_FILES['sonarnet/data/geo.py'] = "\"\"\"Chuyển đổi toạ độ địa lý và toạ độ điểm ảnh.\n\nVùng thí điểm có kích thước khoảng 100 km nên phép chiếu phẳng cục bộ cho sai\nsố không đáng kể so với ngưỡng ghép cặp 500 mét của hệ thống.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\nfrom typing import Tuple\n\nimport numpy as np\n\nEARTH_RADIUS_M = 6_371_008.8\n\n\ndef haversine_m(lat1, lon1, lat2, lon2):\n    \"\"\"Khoảng cách vòng lớn giữa hai điểm, tính bằng mét. Hỗ trợ mảng NumPy.\"\"\"\n    lat1, lon1, lat2, lon2 = map(np.asarray, (lat1, lon1, lat2, lon2))\n    p1, p2 = np.radians(lat1), np.radians(lat2)\n    dp = np.radians(lat2 - lat1)\n    dl = np.radians(lon2 - lon1)\n    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2\n    return 2 * EARTH_RADIUS_M * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))\n\n\ndef meters_per_degree(lat: float) -> Tuple[float, float]:\n    \"\"\"Số mét ứng với một độ vĩ và một độ kinh tại vĩ độ đã cho.\"\"\"\n    m_per_deg_lat = 111_132.92 - 559.82 * math.cos(2 * math.radians(lat))\n    m_per_deg_lon = 111_412.84 * math.cos(math.radians(lat))\n    return m_per_deg_lat, max(m_per_deg_lon, 1.0)\n\n\n@dataclass(frozen=True)\nclass SceneGeoReference:\n    \"\"\"Hệ quy chiếu của một cảnh ảnh: ánh xạ giữa điểm ảnh và toạ độ địa lý.\n\n    Gốc toạ độ điểm ảnh nằm ở góc trên bên trái, trục y hướng xuống — quy ước\n    chuẩn của ảnh raster.\n    \"\"\"\n\n    lat_top: float\n    lon_left: float\n    width: int\n    height: int\n    pixel_spacing_m: float\n\n    @property\n    def _scales(self) -> Tuple[float, float]:\n        m_lat, m_lon = meters_per_degree(self.lat_top)\n        return m_lat, m_lon\n\n    def pixel_to_lonlat(self, x, y) -> Tuple[np.ndarray, np.ndarray]:\n        m_lat, m_lon = self._scales\n        x = np.asarray(x, dtype=float)\n        y = np.asarray(y, dtype=float)\n        lon = self.lon_left + (x * self.pixel_spacing_m) / m_lon\n        lat = self.lat_top - (y * self.pixel_spacing_m) / m_lat\n        return lon, lat\n\n    def lonlat_to_pixel(self, lon, lat) -> Tuple[np.ndarray, np.ndarray]:\n        m_lat, m_lon = self._scales\n        lon = np.asarray(lon, dtype=float)\n        lat = np.asarray(lat, dtype=float)\n        x = (lon - self.lon_left) * m_lon / self.pixel_spacing_m\n        y = (self.lat_top - lat) * m_lat / self.pixel_spacing_m\n        return x, y\n\n    @property\n    def bounds(self) -> Tuple[float, float, float, float]:\n        \"\"\"Trả về (lon_min, lat_min, lon_max, lat_max) của cảnh.\"\"\"\n        lon_r, lat_b = self.pixel_to_lonlat(self.width, self.height)\n        return float(self.lon_left), float(lat_b), float(lon_r), float(self.lat_top)\n\n    @property\n    def center(self) -> Tuple[float, float]:\n        lon_c, lat_c = self.pixel_to_lonlat(self.width / 2, self.height / 2)\n        return float(lat_c), float(lon_c)\n"_FILES['sonarnet/data/sar_render.py'] = "\"\"\"Tổng hợp ảnh radar khẩu độ tổng hợp mô phỏng.\n\nẢnh được dựng theo các đặc trưng vật lý chính của ảnh SAR biển:\n\n* **Tán xạ nền biển** tuân theo phân bố Rayleigh, phản ánh tổng hợp ngẫu nhiên\n  của vô số tán xạ tử nhỏ trên mặt nước.\n* **Nhiễu đốm (speckle)** nhân tính, đặc trưng của mọi hệ radar kết hợp pha.\n* **Điều biến quy mô lớn** do gió và sóng lừng, tạo các vệt sáng tối trên mặt biển.\n* **Phương tiện** có hệ số tán xạ ngược cao hơn nền hàng chục lần, hình dạng\n  thuôn dài theo hướng mũi tàu, kèm một số tán xạ tử điểm rất sáng.\n* **Bóng ma phương vị** (azimuth ambiguity): bản sao mờ của mục tiêu sáng, lệch\n  theo hướng phương vị — một giả tượng thường gây dương tính giả trong thực tế.\n* **Vệt nước sau tàu** (wake): dải tối kéo dài phía sau phương tiện đang chạy.\n\nẢnh được lượng tử hoá về thang 8 bit sau khi nén theo hàm lô-ga-rít, đúng như\ncách sản phẩm Ground Range Detected được hiển thị trong thực tế.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import List, Optional, Tuple\n\nimport numpy as np\n\ntry:  # Lọc Gauss của SciPy cho chất lượng tốt hơn, có sẵn trên Kaggle\n    from scipy.ndimage import gaussian_filter as _gaussian_filter\n\n    _HAS_SCIPY = True\nexcept Exception:  # pragma: no cover\n    _HAS_SCIPY = False\n\n\ndef _smooth(arr: np.ndarray, sigma: float) -> np.ndarray:\n    \"\"\"Làm trơn Gauss, có phương án thay thế khi thiếu SciPy.\"\"\"\n    if sigma <= 0:\n        return arr\n    if _HAS_SCIPY:\n        return _gaussian_filter(arr, sigma=sigma, mode=\"reflect\")\n    # Phương án thay thế: tích chập tách được bằng nhân Gauss rời rạc\n    radius = max(1, int(3 * sigma))\n    k = np.exp(-0.5 * (np.arange(-radius, radius + 1) / sigma) ** 2)\n    k /= k.sum()\n    out = np.apply_along_axis(lambda m: np.convolve(m, k, mode=\"same\"), 0, arr)\n    out = np.apply_along_axis(lambda m: np.convolve(m, k, mode=\"same\"), 1, out)\n    return out\n\n\n@dataclass\nclass ShipFootprint:\n    \"\"\"Dấu vết của một phương tiện trên mặt phẳng ảnh.\"\"\"\n\n    cx: float          # tâm theo trục x, đơn vị điểm ảnh\n    cy: float          # tâm theo trục y, đơn vị điểm ảnh\n    length_px: float   # chiều dài thân tàu\n    width_px: float    # chiều rộng thân tàu\n    heading_deg: float # hướng mũi tàu, 0 độ là hướng đông, tăng ngược chiều kim đồng hồ\n    rcs: float         # cường độ tán xạ ngược tương đối\n    moving: bool       # có đang chạy hay không, quyết định việc vẽ vệt nước\n\n    def bbox(self, pad: float = 1.5) -> Tuple[float, float, float, float]:\n        \"\"\"Khung bao trục chuẩn của phương tiện, dạng (x1, y1, x2, y2).\"\"\"\n        th = np.radians(self.heading_deg)\n        hl, hw = self.length_px / 2.0, self.width_px / 2.0\n        # Bốn đỉnh của hình chữ nhật xoay\n        dx = np.array([hl, hl, -hl, -hl])\n        dy = np.array([hw, -hw, hw, -hw])\n        xs = self.cx + dx * np.cos(th) - dy * np.sin(th)\n        ys = self.cy + dx * np.sin(th) + dy * np.cos(th)\n        return (\n            float(xs.min() - pad),\n            float(ys.min() - pad),\n            float(xs.max() + pad),\n            float(ys.max() + pad),\n        )\n\n\nclass SARRenderer:\n    \"\"\"Bộ dựng ảnh SAR mô phỏng cho một cảnh biển.\"\"\"\n\n    def __init__(\n        self,\n        size: int = 640,\n        sea_scale: float = 0.24,\n        wind_strength: float = 0.30,\n        speckle_looks: int = 3,\n        azimuth_ambiguity: bool = True,\n    ) -> None:\n        self.size = size\n        self.sea_scale = sea_scale\n        self.wind_strength = wind_strength\n        self.speckle_looks = max(1, speckle_looks)\n        self.azimuth_ambiguity = azimuth_ambiguity\n\n    # -- Nền biển ----------------------------------------------------------\n    def _sea_background(self, rng: np.random.Generator) -> np.ndarray:\n        n = self.size\n        # Tán xạ nền Rayleigh\n        base = rng.rayleigh(scale=self.sea_scale, size=(n, n))\n        # Điều biến quy mô lớn do gió và sóng lừng\n        coarse = _smooth(rng.standard_normal((n, n)), sigma=n / 14.0)\n        coarse = coarse / (np.abs(coarse).max() + 1e-9)\n        # Vệt gió có hướng ưu thế\n        angle = rng.uniform(0, np.pi)\n        yy, xx = np.mgrid[0:n, 0:n].astype(np.float32)\n        streak_phase = (xx * np.cos(angle) + yy * np.sin(angle)) / rng.uniform(45, 130)\n        streaks = 0.5 * np.sin(streak_phase + rng.uniform(0, 2 * np.pi))\n        modulation = 1.0 + self.wind_strength * (coarse + 0.45 * streaks)\n        return base * np.clip(modulation, 0.25, 2.2)\n\n    # -- Vùng đất liền -----------------------------------------------------\n    def _add_land(self, img: np.ndarray, rng: np.random.Generator) -> np.ndarray:\n        \"\"\"Chèn một vùng đất liền hoặc đảo với tán xạ mạnh và kết cấu thô.\"\"\"\n        n = self.size\n        mask = np.zeros((n, n), dtype=np.float32)\n        # Dựng biên đất bằng một hàm sóng ngẫu nhiên dọc một cạnh ảnh\n        edge = rng.integers(0, 4)\n        depth = rng.uniform(0.12, 0.34) * n\n        coords = np.arange(n)\n        wobble = np.zeros(n)\n        for _ in range(3):\n            wobble += rng.uniform(0.05, 0.22) * depth * np.sin(\n                2 * np.pi * coords / rng.uniform(60, 320) + rng.uniform(0, 6.28)\n            )\n        boundary = np.clip(depth + wobble, 4, n - 4)\n        yy, xx = np.mgrid[0:n, 0:n]\n        if edge == 0:\n            mask[yy < boundary[None, :]] = 1.0\n        elif edge == 1:\n            mask[yy > (n - boundary[None, :])] = 1.0\n        elif edge == 2:\n            mask[xx < boundary[:, None]] = 1.0\n        else:\n            mask[xx > (n - boundary[:, None])] = 1.0\n\n        mask = _smooth(mask, sigma=1.6)\n        land_tex = rng.rayleigh(scale=self.sea_scale * 3.6, size=(n, n))\n        land_tex *= 1.0 + 0.8 * _smooth(rng.standard_normal((n, n)), sigma=2.2)\n        return img * (1 - mask) + land_tex * mask, mask\n\n    # -- Phương tiện -------------------------------------------------------\n    def _draw_ship(\n        self, img: np.ndarray, ship: ShipFootprint, rng: np.random.Generator\n    ) -> None:\n        n = self.size\n        pad = int(max(ship.length_px, ship.width_px)) + 12\n        x0 = int(max(0, ship.cx - pad))\n        x1 = int(min(n, ship.cx + pad))\n        y0 = int(max(0, ship.cy - pad))\n        y1 = int(min(n, ship.cy + pad))\n        if x1 <= x0 or y1 <= y0:\n            return\n\n        yy, xx = np.mgrid[y0:y1, x0:x1].astype(np.float32)\n        th = np.radians(ship.heading_deg)\n        dx = xx - ship.cx\n        dy = yy - ship.cy\n        # Quay về hệ toạ độ gắn với thân tàu\n        u = dx * np.cos(th) + dy * np.sin(th)     # dọc thân\n        v = -dx * np.sin(th) + dy * np.cos(th)    # ngang thân\n\n        hl, hw = ship.length_px / 2.0, ship.width_px / 2.0\n        # Thân tàu: siêu ê-líp cho cạnh sắc hơn hình ê-líp thường\n        body = np.exp(-(((u / hl) ** 4 + (v / hw) ** 4)) * 1.6)\n\n        # Tán xạ tử điểm: cấu trúc thượng tầng, cần cẩu, góc phản xạ ba mặt\n        n_scat = rng.integers(2, 6)\n        scat = np.zeros_like(body)\n        for _ in range(n_scat):\n            su = rng.uniform(-hl * 0.85, hl * 0.85)\n            sv = rng.uniform(-hw * 0.6, hw * 0.6)\n            amp = rng.uniform(0.8, 2.4)\n            sig = rng.uniform(0.7, 1.7)\n            scat += amp * np.exp(-(((u - su) ** 2 + (v - sv) ** 2) / (2 * sig**2)))\n\n        patch = ship.rcs * (body + 0.55 * scat)\n        img[y0:y1, x0:x1] += patch\n\n        # Vệt nước sau tàu: dải tối do mặt biển bị san phẳng\n        if ship.moving and rng.random() < 0.55:\n            wake_len = ship.length_px * rng.uniform(3.5, 9.0)\n            wake_wid = ship.width_px * rng.uniform(0.8, 1.6)\n            u_w = u + wake_len / 2.0\n            wake = np.exp(-((u_w / (wake_len / 2.0)) ** 6) - (v / wake_wid) ** 2)\n            wake *= (u < 0).astype(np.float32)\n            img[y0:y1, x0:x1] *= 1.0 - 0.45 * wake\n\n        # Bóng ma phương vị: bản sao mờ lệch theo trục dọc ảnh\n        if self.azimuth_ambiguity and rng.random() < 0.16:\n            shift = int(rng.choice([-1, 1]) * rng.uniform(0.06, 0.13) * n)\n            gy0, gy1 = y0 + shift, y1 + shift\n            if 0 <= gy0 and gy1 <= n:\n                img[gy0:gy1, x0:x1] += patch * rng.uniform(0.05, 0.13)\n\n    # -- Giao diện chính ---------------------------------------------------\n    def render(\n        self,\n        ships: List[ShipFootprint],\n        rng: np.random.Generator,\n        with_land: bool = False,\n    ) -> Tuple[np.ndarray, Optional[np.ndarray]]:\n        \"\"\"Dựng một cảnh ảnh SAR 8 bit.\n\n        Trả về ảnh dạng ``uint8`` và mặt nạ đất liền (hoặc ``None``).\n        \"\"\"\n        img = self._sea_background(rng)\n        land_mask = None\n        if with_land:\n            img, land_mask = self._add_land(img, rng)\n\n        for ship in ships:\n            self._draw_ship(img, ship, rng)\n\n        # Nhiễu đốm nhân tính, mô hình đa nhìn (multi-look) phân bố Gamma\n        L = self.speckle_looks\n        speckle = rng.gamma(shape=L, scale=1.0 / L, size=img.shape)\n        img = img * speckle\n\n        # Nén lô-ga-rít về thang decibel rồi lượng tử hoá 8 bit\n        img = np.clip(img, 1e-4, None)\n        db = 20.0 * np.log10(img)\n        lo, hi = np.percentile(db, [1.0, 99.6])\n        if hi - lo < 1e-6:\n            hi = lo + 1.0\n        out = np.clip((db - lo) / (hi - lo), 0.0, 1.0)\n        return (out * 255.0).astype(np.uint8), land_mask\n"for _rel, _content in _FILES.items():    _p = SRC_DIR / _rel    _p.parent.mkdir(parents=True, exist_ok=True)    _p.write_text(_content, encoding='utf-8')print(f'Đã ghi {len(_FILES)} tệp mã nguồn.')for _rel in _FILES:    print('   ', _rel)

### Tầng dữ liệu — mô phỏng cảnh ảnh, dòng AIS và quỹ đạo

In [ ]:
# Ghi mã nguồn ra đĩa. Nội dung giữ nguyên như trong kho mã nguồn._FILES = {}_FILES['sonarnet/data/simulator.py'] = "\"\"\"Mô phỏng liên kết giữa cảnh ảnh SAR và dòng tín hiệu AIS.\n\nĐây là thành phần cho phép toàn bộ pipeline chạy được từ đầu đến cuối mà không\ncần khoá truy cập Copernicus hay Global Fishing Watch. Mỗi cảnh mô phỏng gồm ba\nlớp thông tin gắn chặt với nhau:\n\n1. Danh sách phương tiện thật sự có mặt trong vùng ảnh, kèm toạ độ địa lý,\n   kích thước, hướng và tốc độ.\n2. Ảnh SAR dựng từ chính danh sách đó.\n3. Dòng bản ghi AIS phát ra bởi các phương tiện *có* định danh, kèm nhiễu vị trí\n   và nhịp phát không đều như trong thực tế.\n\nNhờ cấu trúc này, mọi chỉ tiêu của tầng hợp nhất đều có nhãn đối chứng chính xác.\n\nBa trạng thái định danh được mô phỏng, tương ứng với phân loại mà hệ thống cần\nnhận ra:\n\n* ``AIS_OK``       — phát tín hiệu đầy đủ và trung thực;\n* ``AIS_MISMATCH`` — có phát tín hiệu nhưng kích thước khai báo sai lệch lớn so\n  với kích thước quan sát được trên ảnh radar;\n* ``DARK``         — không phát tín hiệu tại thời điểm chụp.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Dict, List, Optional, Tuple\n\nimport numpy as np\n\nfrom .geo import SceneGeoReference\nfrom .sar_render import SARRenderer, ShipFootprint\n\n# Ba trạng thái định danh\nAIS_OK = \"AIS_OK\"\nAIS_MISMATCH = \"AIS_MISMATCH\"\nDARK = \"DARK\"\nIDENTITY_STATES = (AIS_OK, AIS_MISMATCH, DARK)\n\n# Bốn nhóm hành vi hoạt động\nBEHAVIOURS = (\"qua_canh\", \"cau\", \"keo_luoi\", \"neo_dau\")\n\n# Đặc tả các lớp phương tiện: (tên, dài tối thiểu, dài tối đa, tỉ lệ rộng/dài)\nVESSEL_CLASSES = [\n    (\"tau_ca_nho\", 60.0, 110.0, 0.26),\n    (\"tau_ca_von_sat\", 110.0, 190.0, 0.22),\n    (\"tau_hau_can\", 170.0, 260.0, 0.18),\n    (\"tau_van_tai\", 220.0, 330.0, 0.15),\n]\n\n# Dải tốc độ đặc trưng cho từng nhóm hành vi, đơn vị hải lý trên giờ\nBEHAVIOUR_SPEED = {\n    \"qua_canh\": (9.5, 17.0),\n    \"cau\": (1.8, 4.5),\n    \"keo_luoi\": (3.0, 6.2),\n    \"neo_dau\": (0.0, 0.9),\n}\n\n\n@dataclass\nclass Vessel:\n    \"\"\"Một phương tiện trong cảnh, kèm toàn bộ thông tin đối chứng.\"\"\"\n\n    mmsi: int\n    lat: float\n    lon: float\n    length_m: float\n    width_m: float\n    heading_deg: float\n    speed_kn: float\n    vessel_class: str\n    behaviour: str\n    identity_state: str\n    # Kích thước khai báo qua AIS; khác length_m ở các trường hợp AIS_MISMATCH\n    declared_length_m: float\n    # Toạ độ điểm ảnh và khung bao trên ảnh\n    cx: float = 0.0\n    cy: float = 0.0\n    bbox: Tuple[float, float, float, float] = (0.0, 0.0, 0.0, 0.0)\n\n    @property\n    def has_ais(self) -> bool:\n        return self.identity_state != DARK\n\n\n@dataclass\nclass Scene:\n    \"\"\"Một cảnh ảnh hoàn chỉnh cùng dữ liệu AIS đi kèm.\"\"\"\n\n    scene_id: str\n    image: np.ndarray\n    georef: SceneGeoReference\n    vessels: List[Vessel]\n    ais_records: List[Dict]\n    capture_time_s: float\n    land_mask: Optional[np.ndarray] = None\n    meta: Dict = field(default_factory=dict)\n\n    @property\n    def boxes(self) -> np.ndarray:\n        \"\"\"Khung bao đối chứng, dạng mảng (N, 4) theo thứ tự x1, y1, x2, y2.\"\"\"\n        if not self.vessels:\n            return np.zeros((0, 4), dtype=np.float32)\n        return np.array([v.bbox for v in self.vessels], dtype=np.float32)\n\n    def count_by_state(self) -> Dict[str, int]:\n        out = {s: 0 for s in IDENTITY_STATES}\n        for v in self.vessels:\n            out[v.identity_state] += 1\n        return out\n\n\nclass SceneSimulator:\n    \"\"\"Sinh các cảnh mô phỏng theo cấu hình đã cho.\"\"\"\n\n    def __init__(self, cfg) -> None:\n        self.cfg = cfg\n        self.renderer = SARRenderer(\n            size=cfg.data.scene_size,\n            azimuth_ambiguity=cfg.data.azimuth_ambiguity,\n        )\n        self._mmsi_counter = 574_000_000  # Dải MMSI của Việt Nam bắt đầu bằng 574\n\n    # -- Trợ giúp ----------------------------------------------------------\n    def _next_mmsi(self, rng: np.random.Generator) -> int:\n        self._mmsi_counter += int(rng.integers(7, 900))\n        return self._mmsi_counter\n\n    def _sample_vessel_class(self, rng: np.random.Generator) -> Tuple[str, float, float]:\n        idx = rng.choice(len(VESSEL_CLASSES), p=[0.42, 0.30, 0.17, 0.11])\n        name, lo, hi, ratio = VESSEL_CLASSES[idx]\n        length = float(rng.uniform(lo, hi))\n        width = float(length * ratio * rng.uniform(0.85, 1.15))\n        return name, length, width\n\n    def _sample_identity_state(self, rng: np.random.Generator) -> str:\n        p_dark = self.cfg.data.dark_vessel_ratio\n        p_mismatch = 0.10\n        r = rng.random()\n        if r < p_dark:\n            return DARK\n        if r < p_dark + p_mismatch:\n            return AIS_MISMATCH\n        return AIS_OK\n\n    # -- Sinh phương tiện --------------------------------------------------\n    def _make_vessels(\n        self, georef: SceneGeoReference, rng: np.random.Generator,\n        land_mask: Optional[np.ndarray],\n    ) -> List[Vessel]:\n        lo, hi = self.cfg.data.ships_per_scene\n        n = int(rng.integers(lo, hi + 1))\n        spacing = self.cfg.geo.pixel_spacing_m\n        size = self.cfg.data.scene_size\n        margin = 26\n\n        vessels: List[Vessel] = []\n        placed: List[Tuple[float, float, float]] = []  # (cx, cy, bán kính chiếm chỗ)\n\n        attempts = 0\n        while len(vessels) < n and attempts < n * 60:\n            attempts += 1\n            cx = float(rng.uniform(margin, size - margin))\n            cy = float(rng.uniform(margin, size - margin))\n\n            # Không đặt phương tiện lên vùng đất liền\n            if land_mask is not None and land_mask[int(cy), int(cx)] > 0.25:\n                continue\n\n            cls, length_m, width_m = self._sample_vessel_class(rng)\n            length_px = length_m / spacing\n            width_px = width_m / spacing\n            radius = 0.5 * length_px + 8.0\n\n            # Tránh chồng lấn để nhãn đối chứng không nhập nhằng\n            if any(\n                (cx - px) ** 2 + (cy - py) ** 2 < (radius + pr) ** 2\n                for px, py, pr in placed\n            ):\n                continue\n\n            behaviour = str(rng.choice(BEHAVIOURS, p=[0.34, 0.27, 0.23, 0.16]))\n            s_lo, s_hi = BEHAVIOUR_SPEED[behaviour]\n            speed = float(rng.uniform(s_lo, s_hi))\n            heading = float(rng.uniform(0, 360))\n            state = self._sample_identity_state(rng)\n\n            declared = length_m\n            if state == AIS_MISMATCH:\n                # Khai báo sai lệch mạnh theo một trong hai hướng\n                factor = rng.choice([rng.uniform(0.22, 0.42), rng.uniform(2.1, 3.4)])\n                declared = float(length_m * factor)\n\n            lon, lat = georef.pixel_to_lonlat(cx, cy)\n            footprint = ShipFootprint(\n                cx=cx, cy=cy,\n                length_px=length_px, width_px=width_px,\n                heading_deg=heading,\n                rcs=float(rng.uniform(2.6, 7.5)),\n                moving=speed > 3.0,\n            )\n            v = Vessel(\n                mmsi=self._next_mmsi(rng),\n                lat=float(lat), lon=float(lon),\n                length_m=length_m, width_m=width_m,\n                heading_deg=heading, speed_kn=speed,\n                vessel_class=cls, behaviour=behaviour,\n                identity_state=state, declared_length_m=declared,\n                cx=cx, cy=cy, bbox=footprint.bbox(),\n            )\n            vessels.append(v)\n            placed.append((cx, cy, radius))\n            # Lưu tạm footprint để dựng ảnh\n            v_meta = getattr(self, \"_footprints\", None)\n            if v_meta is None:\n                self._footprints = {}\n            self._footprints[v.mmsi] = footprint\n\n        return vessels\n\n    # -- Sinh bản ghi AIS --------------------------------------------------\n    def _make_ais(\n        self, vessels: List[Vessel], capture_time_s: float, rng: np.random.Generator\n    ) -> List[Dict]:\n        \"\"\"Sinh dòng bản ghi AIS quanh thời điểm chụp ảnh.\n\n        Quỹ đạo thật của mỗi phương tiện được tích phân từ một chuỗi gia tốc\n        nhiễu trắng biên độ nhỏ, thay vì giả định vận tốc không đổi tuyệt đối.\n        Đây là điểm quan trọng: trên biển, tàu luôn đổi tốc và đổi hướng nhẹ do\n        sóng, dòng chảy và thao tác bánh lái. Một mô phỏng tuyến tính hoàn hảo sẽ\n        khiến mọi phép so sánh giữa các phương pháp nội suy mất ý nghĩa.\n\n        Quỹ đạo được neo sao cho vị trí tại đúng thời điểm chụp ảnh trùng khớp\n        với vị trí đối chứng của phương tiện. Trên nền quỹ đạo đó, hệ thống lấy\n        mẫu tại các mốc thời gian không đều và cộng thêm nhiễu đo của thiết bị.\n\n        Phương tiện ở trạng thái ``DARK`` hoàn toàn không xuất hiện trong dòng\n        dữ liệu này — đúng như tình huống tàu chủ động ngắt thiết bị.\n        \"\"\"\n        window = self.cfg.data.ais_window_s\n        noise_m = self.cfg.data.ais_position_noise_m\n        int_lo, int_hi = self.cfg.data.ais_interval_s\n        # Biên độ gia tốc nhiễu, mét trên giây bình phương. Giá trị nhỏ phản ánh\n        # quán tính lớn của phương tiện đường thuỷ.\n        accel_sigma = 0.012\n        step = 15.0  # bước tích phân quỹ đạo, giây\n\n        records: List[Dict] = []\n\n        for v in vessels:\n            if not v.has_ais:\n                continue\n\n            speed_ms = v.speed_kn * 0.514444\n            course = np.radians(v.heading_deg)\n            vx0 = speed_ms * np.cos(course)   # thành phần đông\n            vy0 = speed_ms * np.sin(course)   # thành phần bắc\n\n            # ---- Tích phân quỹ đạo thật trên lưới thời gian đều ----------\n            grid = np.arange(-window - 60.0, window + 60.0 + step, step)\n            n = len(grid)\n            ax = rng.normal(0.0, accel_sigma, size=n)\n            ay = rng.normal(0.0, accel_sigma, size=n)\n            # Vận tốc là tích phân của gia tốc, lấy mốc không tại thời điểm chụp\n            i0 = int(np.argmin(np.abs(grid)))\n            vx = vx0 + np.cumsum(ax) * step\n            vy = vy0 + np.cumsum(ay) * step\n            vx -= vx[i0] - vx0\n            vy -= vy[i0] - vy0\n            # Vị trí là tích phân của vận tốc, neo về gốc tại thời điểm chụp\n            px = np.cumsum(vx) * step\n            py = np.cumsum(vy) * step\n            px -= px[i0]\n            py -= py[i0]\n\n            m_lat = 111_132.0\n            m_lon = 111_320.0 * np.cos(np.radians(v.lat))\n\n            # ---- Lấy mẫu tại các mốc phát tín hiệu không đều -------------\n            t = capture_time_s - window\n            times = []\n            while t <= capture_time_s + window:\n                times.append(t)\n                t += float(rng.uniform(int_lo, int_hi))\n\n            # Một phần phương tiện có khoảng trống ngắn trong dòng tín hiệu,\n            # phản ánh mất sóng do nhiễu hoặc che khuất\n            if rng.random() < 0.22 and len(times) > 6:\n                g0 = int(rng.integers(1, len(times) - 4))\n                g1 = min(len(times), g0 + int(rng.integers(2, 5)))\n                times = times[:g0] + times[g1:]\n\n            for ts in times:\n                dt = ts - capture_time_s\n                east = float(np.interp(dt, grid, px)) + rng.normal(0.0, noise_m)\n                north = float(np.interp(dt, grid, py)) + rng.normal(0.0, noise_m)\n                sog = float(np.hypot(\n                    np.interp(dt, grid, vx), np.interp(dt, grid, vy)\n                )) / 0.514444\n                records.append(\n                    {\n                        \"mmsi\": v.mmsi,\n                        \"timestamp\": float(ts),\n                        \"lat\": float(v.lat + north / m_lat),\n                        \"lon\": float(v.lon + east / m_lon),\n                        \"sog_kn\": float(max(0.0, sog + rng.normal(0, 0.2))),\n                        \"cog_deg\": float((v.heading_deg + rng.normal(0, 3.5)) % 360.0),\n                        \"declared_length_m\": float(v.declared_length_m),\n                    }\n                )\n\n        records.sort(key=lambda r: r[\"timestamp\"])\n        return records\n\n    # -- Giao diện chính ---------------------------------------------------\n    def generate_scene(self, scene_id: str, rng: np.random.Generator) -> Scene:\n        size = self.cfg.data.scene_size\n        spacing = self.cfg.geo.pixel_spacing_m\n        g = self.cfg.geo\n\n        # Chọn ngẫu nhiên một ô con trong vùng thí điểm\n        span_deg_lat = (size * spacing) / 111_132.0\n        lat_top = float(rng.uniform(g.lat_min + span_deg_lat, g.lat_max))\n        span_deg_lon = (size * spacing) / (111_320.0 * np.cos(np.radians(lat_top)))\n        lon_left = float(rng.uniform(g.lon_min, g.lon_max - span_deg_lon))\n\n        georef = SceneGeoReference(\n            lat_top=lat_top, lon_left=lon_left,\n            width=size, height=size, pixel_spacing_m=spacing,\n        )\n\n        # Vùng đất liền được quyết định trước để tránh đặt phương tiện lên bờ\n        with_land = bool(rng.random() < self.cfg.data.land_probability)\n        land_mask = None\n        if with_land:\n            probe = self.renderer._sea_background(rng)\n            _, land_mask = self.renderer._add_land(probe, rng)\n\n        self._footprints = {}\n        vessels = self._make_vessels(georef, rng, land_mask)\n        footprints = [self._footprints[v.mmsi] for v in vessels]\n\n        image, rendered_mask = self.renderer.render(\n            footprints, rng, with_land=with_land\n        )\n        if rendered_mask is not None:\n            land_mask = rendered_mask\n\n        capture_time_s = float(rng.uniform(0, 86_400))\n        ais = self._make_ais(vessels, capture_time_s, rng)\n\n        return Scene(\n            scene_id=scene_id,\n            image=image,\n            georef=georef,\n            vessels=vessels,\n            ais_records=ais,\n            capture_time_s=capture_time_s,\n            land_mask=land_mask,\n            meta={\n                \"with_land\": with_land,\n                \"n_vessels\": len(vessels),\n                \"n_ais_records\": len(ais),\n            },\n        )\n\n    def generate_split(\n        self, n_scenes: int, prefix: str, rng: np.random.Generator\n    ) -> List[Scene]:\n        \"\"\"Sinh một tập cảnh với định danh liên tiếp.\"\"\"\n        return [\n            self.generate_scene(f\"{prefix}_{i:05d}\", rng) for i in range(n_scenes)\n        ]\n"_FILES['sonarnet/data/tracks.py'] = "\"\"\"Sinh quỹ đạo phương tiện theo bốn nhóm hành vi hoạt động.\n\nMỗi nhóm hành vi có dấu hiệu động học riêng biệt, phản ánh đúng thực tế khai\nthác trên biển:\n\n* ``qua_canh``  — di chuyển quá cảnh: tốc độ cao, hướng gần như không đổi.\n* ``cau``       — câu: tốc độ thấp, hướng đổi liên tục theo kiểu bước ngẫu nhiên,\n  xen kẽ nhiều lần dừng khi thu và thả câu.\n* ``keo_luoi``  — kéo lưới: tốc độ trung bình ổn định, chạy theo các đoạn thẳng\n  dài rồi quay đầu gấp, tạo thành đường đi hình răng lược.\n* ``neo_dau``   — neo đậu hoặc tụ tập: gần như đứng yên, chỉ trôi quanh một điểm\n  theo dòng triều.\n\nCác đặc trưng trích từ quỹ đạo là đầu vào cho bộ phân loại hành vi ở tầng bốn.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, List, Tuple\n\nimport numpy as np\n\nfrom .geo import meters_per_degree\n\nKNOT_TO_MS = 0.514444\n\n\n@dataclass\nclass Track:\n    \"\"\"Một quỹ đạo phương tiện đã lấy mẫu theo thời gian.\"\"\"\n\n    mmsi: int\n    behaviour: str\n    times_s: np.ndarray      # (T,) mốc thời gian, giây\n    lats: np.ndarray         # (T,) vĩ độ\n    lons: np.ndarray         # (T,) kinh độ\n    speeds_kn: np.ndarray    # (T,) tốc độ trên mặt nước\n    courses_deg: np.ndarray  # (T,) hướng đi\n\n    def __len__(self) -> int:\n        return len(self.times_s)\n\n\ndef _step(lat, lon, speed_kn, course_deg, dt_s):\n    \"\"\"Dịch chuyển một bước theo tốc độ và hướng đã cho.\"\"\"\n    m_lat, m_lon = meters_per_degree(lat)\n    dist = speed_kn * KNOT_TO_MS * dt_s\n    rad = np.radians(course_deg)\n    # Quy ước: 0 độ là hướng bắc, tăng theo chiều kim đồng hồ\n    north = dist * np.cos(rad)\n    east = dist * np.sin(rad)\n    return lat + north / m_lat, lon + east / m_lon\n\n\ndef _gen_transit(n, dt, lat0, lon0, rng) -> Tuple[np.ndarray, ...]:\n    speed = rng.uniform(10.0, 17.0)\n    course = rng.uniform(0, 360)\n    lats, lons, sp, co = [lat0], [lon0], [], []\n    for _ in range(n):\n        # Dao động nhỏ quanh giá trị danh nghĩa do sóng và bánh lái\n        s = max(0.0, speed + rng.normal(0, 0.55))\n        c = course + rng.normal(0, 2.2)\n        course = 0.97 * course + 0.03 * c  # trôi hướng rất chậm\n        sp.append(s)\n        co.append(c % 360)\n        la, lo = _step(lats[-1], lons[-1], s, c, dt)\n        lats.append(la)\n        lons.append(lo)\n    return np.array(lats[:-1]), np.array(lons[:-1]), np.array(sp), np.array(co)\n\n\ndef _gen_fishing(n, dt, lat0, lon0, rng) -> Tuple[np.ndarray, ...]:\n    course = rng.uniform(0, 360)\n    lats, lons, sp, co = [lat0], [lon0], [], []\n    for _ in range(n):\n        # Bước ngẫu nhiên mạnh về hướng, đặc trưng của thao tác thả và thu câu\n        course = (course + rng.normal(0, 34.0)) % 360\n        if rng.random() < 0.18:\n            s = rng.uniform(0.0, 0.6)          # dừng để thu câu\n        else:\n            s = max(0.0, rng.uniform(1.8, 4.5) + rng.normal(0, 0.4))\n        sp.append(s)\n        co.append(course)\n        la, lo = _step(lats[-1], lons[-1], s, course, dt)\n        lats.append(la)\n        lons.append(lo)\n    return np.array(lats[:-1]), np.array(lons[:-1]), np.array(sp), np.array(co)\n\n\ndef _gen_trawling(n, dt, lat0, lon0, rng) -> Tuple[np.ndarray, ...]:\n    base_course = rng.uniform(0, 360)\n    leg_len = int(rng.integers(8, 18))   # số bước của một đoạn thẳng\n    speed = rng.uniform(3.2, 6.0)\n    lats, lons, sp, co = [lat0], [lon0], [], []\n    course = base_course\n    since_turn = 0\n    flip = 1\n    for _ in range(n):\n        if since_turn >= leg_len:\n            # Quay đầu gần 180 độ rồi dịch ngang sang luống kế tiếp\n            course = (base_course + (180 if flip > 0 else 0) + rng.normal(0, 6)) % 360\n            flip *= -1\n            since_turn = 0\n            leg_len = int(rng.integers(8, 18))\n        s = max(0.0, speed + rng.normal(0, 0.3))\n        c = course + rng.normal(0, 3.0)\n        sp.append(s)\n        co.append(c % 360)\n        la, lo = _step(lats[-1], lons[-1], s, c, dt)\n        lats.append(la)\n        lons.append(lo)\n        since_turn += 1\n    return np.array(lats[:-1]), np.array(lons[:-1]), np.array(sp), np.array(co)\n\n\ndef _gen_loitering(n, dt, lat0, lon0, rng) -> Tuple[np.ndarray, ...]:\n    # Trôi quanh điểm neo theo một vòng triều khép kín\n    radius_m = rng.uniform(60, 260)\n    period = rng.uniform(6, 14) * 3600.0\n    phase = rng.uniform(0, 2 * np.pi)\n    m_lat, m_lon = meters_per_degree(lat0)\n    lats, lons, sp, co = [], [], [], []\n    prev = None\n    for i in range(n):\n        t = i * dt\n        ang = 2 * np.pi * t / period + phase\n        la = lat0 + (radius_m * np.sin(ang)) / m_lat + rng.normal(0, 8) / m_lat\n        lo = lon0 + (radius_m * np.cos(ang)) / m_lon + rng.normal(0, 8) / m_lon\n        lats.append(la)\n        lons.append(lo)\n        if prev is None:\n            sp.append(rng.uniform(0.0, 0.4))\n            co.append(rng.uniform(0, 360))\n        else:\n            dn = (la - prev[0]) * m_lat\n            de = (lo - prev[1]) * m_lon\n            dist = float(np.hypot(dn, de))\n            sp.append(dist / dt / KNOT_TO_MS)\n            co.append(float(np.degrees(np.arctan2(de, dn)) % 360))\n        prev = (la, lo)\n    return np.array(lats), np.array(lons), np.array(sp), np.array(co)\n\n\n_GENERATORS = {\n    \"qua_canh\": _gen_transit,\n    \"cau\": _gen_fishing,\n    \"keo_luoi\": _gen_trawling,\n    \"neo_dau\": _gen_loitering,\n}\n\n\ndef generate_track(\n    behaviour: str, mmsi: int, cfg, rng: np.random.Generator\n) -> Track:\n    \"\"\"Sinh một quỹ đạo thuộc nhóm hành vi đã cho.\"\"\"\n    dt = cfg.behavior.track_step_min * 60.0\n    n = int(cfg.behavior.track_hours * 3600.0 / dt)\n    g = cfg.geo\n    lat0 = float(rng.uniform(g.lat_min, g.lat_max))\n    lon0 = float(rng.uniform(g.lon_min, g.lon_max))\n\n    lats, lons, sp, co = _GENERATORS[behaviour](n, dt, lat0, lon0, rng)\n    times = np.arange(n, dtype=float) * dt\n    return Track(\n        mmsi=mmsi, behaviour=behaviour,\n        times_s=times, lats=lats, lons=lons,\n        speeds_kn=sp, courses_deg=co,\n    )\n\n\ndef generate_track_dataset(cfg, rng: np.random.Generator) -> List[Track]:\n    \"\"\"Sinh toàn bộ tập quỹ đạo dùng để huấn luyện bộ phân loại hành vi.\"\"\"\n    tracks: List[Track] = []\n    mmsi = 574_900_000\n    for behaviour in cfg.behavior.classes:\n        for _ in range(cfg.behavior.n_tracks_per_class):\n            mmsi += int(rng.integers(3, 400))\n            tracks.append(generate_track(behaviour, mmsi, cfg, rng))\n    rng.shuffle(tracks)\n    return tracks\n"_FILES['sonarnet/data/dataset.py'] = "\"\"\"Ghi các cảnh mô phỏng ra đĩa theo định dạng chuẩn cho huấn luyện và đánh giá.\n\nBố cục thư mục tuân theo quy ước của Ultralytics YOLO để có thể huấn luyện trực\ntiếp, đồng thời lưu kèm siêu dữ liệu địa lý và dòng AIS phục vụ tầng hợp nhất::\n\n    data/yolo/\n      ├── images/{train,val,test}/*.png\n      ├── labels/{train,val,test}/*.txt\n      ├── scenes/{train,val,test}.jsonl     ← siêu dữ liệu phương tiện\n      ├── ais/{train,val,test}.jsonl        ← dòng bản ghi AIS\n      └── data.yaml\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Optional\n\nimport numpy as np\n\nfrom ..utils import get_logger\nfrom .geo import SceneGeoReference\nfrom .simulator import Scene, Vessel\n\nLOG = get_logger(\"data.dataset\")\n\n# Một lớp đối tượng duy nhất: phương tiện trên biển\nCLASS_NAMES = [\"phuong_tien\"]\n\n\ndef _save_image(img: np.ndarray, path: Path) -> None:\n    \"\"\"Ghi ảnh xám 8 bit, ưu tiên OpenCV rồi tới Pillow.\"\"\"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    try:\n        import cv2\n\n        cv2.imwrite(str(path), img)\n        return\n    except Exception:\n        pass\n    from PIL import Image\n\n    Image.fromarray(img).save(path)\n\n\ndef load_image(path: Path) -> np.ndarray:\n    try:\n        import cv2\n\n        arr = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)\n        if arr is not None:\n            return arr\n    except Exception:\n        pass\n    from PIL import Image\n\n    return np.array(Image.open(path).convert(\"L\"))\n\n\ndef boxes_to_yolo(boxes: np.ndarray, size: int) -> List[str]:\n    \"\"\"Chuyển khung bao (x1, y1, x2, y2) sang dòng nhãn YOLO đã chuẩn hoá.\"\"\"\n    lines: List[str] = []\n    for x1, y1, x2, y2 in boxes:\n        x1 = max(0.0, min(float(x1), size - 1.0))\n        y1 = max(0.0, min(float(y1), size - 1.0))\n        x2 = max(0.0, min(float(x2), size - 1.0))\n        y2 = max(0.0, min(float(y2), size - 1.0))\n        w, h = x2 - x1, y2 - y1\n        if w <= 1.0 or h <= 1.0:\n            continue\n        xc = (x1 + x2) / 2.0 / size\n        yc = (y1 + y2) / 2.0 / size\n        lines.append(f\"0 {xc:.6f} {yc:.6f} {w / size:.6f} {h / size:.6f}\")\n    return lines\n\n\ndef _georef_to_dict(g: SceneGeoReference) -> Dict:\n    return {\n        \"lat_top\": g.lat_top,\n        \"lon_left\": g.lon_left,\n        \"width\": g.width,\n        \"height\": g.height,\n        \"pixel_spacing_m\": g.pixel_spacing_m,\n    }\n\n\ndef georef_from_dict(d: Dict) -> SceneGeoReference:\n    return SceneGeoReference(\n        lat_top=d[\"lat_top\"], lon_left=d[\"lon_left\"],\n        width=d[\"width\"], height=d[\"height\"],\n        pixel_spacing_m=d[\"pixel_spacing_m\"],\n    )\n\n\ndef write_split(\n    scenes: Iterable[Scene], split: str, root: Path, size: int\n) -> Dict[str, int]:\n    \"\"\"Ghi một tập cảnh ra đĩa, trả về thống kê tóm tắt.\"\"\"\n    root = Path(root)\n    img_dir = root / \"images\" / split\n    lbl_dir = root / \"labels\" / split\n    img_dir.mkdir(parents=True, exist_ok=True)\n    lbl_dir.mkdir(parents=True, exist_ok=True)\n    (root / \"scenes\").mkdir(parents=True, exist_ok=True)\n    (root / \"ais\").mkdir(parents=True, exist_ok=True)\n\n    scene_meta_path = root / \"scenes\" / f\"{split}.jsonl\"\n    ais_path = root / \"ais\" / f\"{split}.jsonl\"\n\n    n_scene = n_vessel = n_box = n_ais = 0\n    state_counts: Dict[str, int] = {}\n\n    with scene_meta_path.open(\"w\", encoding=\"utf-8\") as f_scene, \\\n         ais_path.open(\"w\", encoding=\"utf-8\") as f_ais:\n\n        for sc in scenes:\n            _save_image(sc.image, img_dir / f\"{sc.scene_id}.png\")\n\n            lines = boxes_to_yolo(sc.boxes, size)\n            (lbl_dir / f\"{sc.scene_id}.txt\").write_text(\n                \"\\n\".join(lines), encoding=\"utf-8\"\n            )\n            n_box += len(lines)\n\n            f_scene.write(\n                json.dumps(\n                    {\n                        \"scene_id\": sc.scene_id,\n                        \"capture_time_s\": sc.capture_time_s,\n                        \"georef\": _georef_to_dict(sc.georef),\n                        \"meta\": sc.meta,\n                        \"vessels\": [asdict(v) for v in sc.vessels],\n                    },\n                    ensure_ascii=False,\n                )\n                + \"\\n\"\n            )\n\n            for rec in sc.ais_records:\n                rec = dict(rec)\n                rec[\"scene_id\"] = sc.scene_id\n                f_ais.write(json.dumps(rec, ensure_ascii=False) + \"\\n\")\n            n_ais += len(sc.ais_records)\n\n            for v in sc.vessels:\n                state_counts[v.identity_state] = state_counts.get(v.identity_state, 0) + 1\n            n_vessel += len(sc.vessels)\n            n_scene += 1\n\n    LOG.info(\n        \"Tập %-5s: %d cảnh | %d phương tiện | %d khung bao | %d bản ghi AIS\",\n        split, n_scene, n_vessel, n_box, n_ais,\n    )\n    return {\n        \"split\": split,\n        \"n_scenes\": n_scene,\n        \"n_vessels\": n_vessel,\n        \"n_boxes\": n_box,\n        \"n_ais_records\": n_ais,\n        **{f\"n_{k}\": v for k, v in state_counts.items()},\n    }\n\n\ndef write_data_yaml(root: Path) -> Path:\n    \"\"\"Sinh tệp mô tả dữ liệu cho Ultralytics.\"\"\"\n    root = Path(root).resolve()\n    content = (\n        f\"# Bộ dữ liệu phát hiện phương tiện trên ảnh SAR — SonarNet-VN\\n\"\n        f\"path: {root}\\n\"\n        f\"train: images/train\\n\"\n        f\"val: images/val\\n\"\n        f\"test: images/test\\n\"\n        f\"nc: {len(CLASS_NAMES)}\\n\"\n        f\"names:\\n\"\n        + \"\".join(f\"  {i}: {n}\\n\" for i, n in enumerate(CLASS_NAMES))\n    )\n    path = root / \"data.yaml\"\n    path.write_text(content, encoding=\"utf-8\")\n    LOG.info(\"Đã ghi mô tả dữ liệu: %s\", path)\n    return path\n\n\ndef read_scene_meta(root: Path, split: str) -> List[Dict]:\n    \"\"\"Đọc lại siêu dữ liệu phương tiện của một tập.\"\"\"\n    path = Path(root) / \"scenes\" / f\"{split}.jsonl\"\n    if not path.exists():\n        return []\n    return [json.loads(line) for line in path.read_text(encoding=\"utf-8\").splitlines() if line.strip()]\n\n\ndef read_ais(root: Path, split: str) -> Dict[str, List[Dict]]:\n    \"\"\"Đọc dòng AIS, nhóm theo định danh cảnh.\"\"\"\n    path = Path(root) / \"ais\" / f\"{split}.jsonl\"\n    out: Dict[str, List[Dict]] = {}\n    if not path.exists():\n        return out\n    for line in path.read_text(encoding=\"utf-8\").splitlines():\n        if not line.strip():\n            continue\n        rec = json.loads(line)\n        out.setdefault(rec[\"scene_id\"], []).append(rec)\n    return out\n\n\ndef build_dataset(cfg, rng: np.random.Generator) -> Dict:\n    \"\"\"Sinh và ghi toàn bộ ba tập dữ liệu. Trả về bản tóm tắt.\"\"\"\n    from .simulator import SceneSimulator\n\n    sim = SceneSimulator(cfg)\n    root = cfg.dir_yolo\n    root.mkdir(parents=True, exist_ok=True)\n\n    summary = {\"splits\": []}\n    plan = [\n        (\"train\", cfg.data.n_train_scenes),\n        (\"val\", cfg.data.n_val_scenes),\n        (\"test\", cfg.data.n_test_scenes),\n    ]\n    for split, n in plan:\n        scenes = sim.generate_split(n, split, rng)\n        summary[\"splits\"].append(\n            write_split(scenes, split, root, cfg.data.scene_size)\n        )\n\n    summary[\"data_yaml\"] = str(write_data_yaml(root))\n    summary[\"root\"] = str(root)\n    return summary\n"for _rel, _content in _FILES.items():    _p = SRC_DIR / _rel    _p.parent.mkdir(parents=True, exist_ok=True)    _p.write_text(_content, encoding='utf-8')print(f'Đã ghi {len(_FILES)} tệp mã nguồn.')for _rel in _FILES:    print('   ', _rel)

### Tầng phát hiện phương tiện

In [ ]:
# Ghi mã nguồn ra đĩa. Nội dung giữ nguyên như trong kho mã nguồn._FILES = {}_FILES['sonarnet/detect/__init__.py'] = "\"\"\"Tầng phát hiện phương tiện trên ảnh radar.\"\"\"\n\nfrom .interface import BaseDetector, DetectionOutput, build_detector, select_backend  # noqa: F401\n"_FILES['sonarnet/detect/interface.py'] = "\"\"\"Giao diện thống nhất cho các bộ phát hiện phương tiện.\n\nHệ thống hỗ trợ hai phương án thực thi để bảo đảm chạy được trong mọi điều kiện\ncủa môi trường Kaggle:\n\n* **Ultralytics YOLO** — phương án chính, đúng như mô tả trong đề xuất. Khai thác\n  được cả hai card T4 thông qua huấn luyện phân tán.\n* **Torchvision Faster R-CNN** — phương án dự phòng, dùng khi không cài đặt được\n  gói bổ sung. Torchvision luôn có sẵn trong ảnh Python của Kaggle nên phương án\n  này không cần kết nối mạng.\n\nCả hai đều phơi bày cùng một giao diện nên các tầng phía sau không cần biết\nphương án nào đang được dùng.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom abc import ABC, abstractmethod\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\n\n@dataclass\nclass DetectionOutput:\n    \"\"\"Kết quả phát hiện trên một ảnh.\"\"\"\n\n    image_id: str\n    boxes: np.ndarray   # (N, 4) theo thứ tự x1, y1, x2, y2\n    scores: np.ndarray  # (N,)\n\n    def filter_by_score(self, threshold: float) -> \"DetectionOutput\":\n        keep = self.scores >= threshold\n        return DetectionOutput(self.image_id, self.boxes[keep], self.scores[keep])\n\n\nclass BaseDetector(ABC):\n    \"\"\"Hợp đồng chung cho mọi bộ phát hiện.\"\"\"\n\n    name: str = \"base\"\n    devices: List[int]\n\n    @abstractmethod\n    def train(self, data_yaml: Path, cfg) -> Dict:\n        \"\"\"Huấn luyện mô hình. Trả về từ điển thông tin của lần chạy.\"\"\"\n\n    @abstractmethod\n    def predict(\n        self, image_paths: Sequence[Path], conf: float = 0.25\n    ) -> List[DetectionOutput]:\n        \"\"\"Suy luận trên danh sách ảnh.\"\"\"\n\n    @abstractmethod\n    def load(self, weights: Path) -> \"BaseDetector\":\n        \"\"\"Nạp trọng số đã huấn luyện.\"\"\"\n\n\ndef select_backend(preference: str = \"auto\") -> str:\n    \"\"\"Chọn phương án thực thi phù hợp với môi trường hiện tại.\"\"\"\n    from ..utils import package_available\n\n    if preference in (\"ultralytics\", \"torchvision\"):\n        return preference\n    if package_available(\"ultralytics\"):\n        return \"ultralytics\"\n    return \"torchvision\"\n\n\ndef build_detector(cfg, backend: Optional[str] = None) -> BaseDetector:\n    \"\"\"Khởi tạo bộ phát hiện theo cấu hình.\"\"\"\n    from ..utils import get_logger, resolve_devices\n\n    log = get_logger(\"detect\")\n    chosen = select_backend(backend or cfg.detect.backend)\n    devices = resolve_devices(cfg.detect.devices, allow_multi=cfg.multi_gpu)\n\n    if chosen == \"ultralytics\":\n        from .yolo_backend import YOLODetector\n\n        log.info(\"Phương án phát hiện: Ultralytics YOLO | GPU: %s\", devices or \"CPU\")\n        return YOLODetector(devices=devices, cfg=cfg)\n\n    from .torchvision_backend import TorchvisionDetector\n\n    log.info(\n        \"Phương án phát hiện: Torchvision Faster R-CNN | GPU: %s\",\n        devices[:1] or \"CPU\",\n    )\n    return TorchvisionDetector(devices=devices[:1], cfg=cfg)\n"_FILES['sonarnet/detect/yolo_backend.py'] = "\"\"\"Bộ phát hiện dựa trên Ultralytics YOLO, khai thác đồng thời hai GPU T4.\n\nHuấn luyện phân tán được kích hoạt bằng cách truyền danh sách nhiều thiết bị cho\ntham số ``device``. Ultralytics tự khởi tạo tiến trình phân tán ở phía sau; để\nquá trình này ổn định trên Kaggle, việc huấn luyện nên được gọi từ một tiến trình\ncon độc lập thay vì gọi trực tiếp trong nhân của notebook. Tệp\n``scripts/train_detector.py`` đảm nhiệm vai trò đó.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence\n\nimport numpy as np\n\nfrom ..utils import get_logger\nfrom .interface import BaseDetector, DetectionOutput\n\nLOG = get_logger(\"detect.yolo\")\n\n\ndef resolve_model_spec(preferred: str = \"yolo11n.pt\") -> str:\n    \"\"\"Chọn điểm khởi tạo mô hình phù hợp với điều kiện kết nối mạng.\n\n    Trọng số đã huấn luyện trước cần tải về từ máy chủ của Ultralytics. Khi\n    notebook chạy ở chế độ ngắt mạng, hệ thống chuyển sang khởi tạo mô hình từ\n    tệp mô tả kiến trúc — quá trình huấn luyện vẫn diễn ra bình thường, chỉ cần\n    thêm một số chu kỳ để hội tụ.\n    \"\"\"\n    local = Path(preferred)\n    if local.exists():\n        return str(local)\n\n    try:\n        import urllib.request\n\n        urllib.request.urlopen(\"https://github.com\", timeout=6)\n        return preferred\n    except Exception:\n        fallback = preferred.replace(\".pt\", \".yaml\")\n        LOG.warning(\n            \"Không có kết nối mạng: khởi tạo mô hình từ kiến trúc %s thay vì \"\n            \"trọng số huấn luyện trước.\", fallback,\n        )\n        return fallback\n\n\nclass YOLODetector(BaseDetector):\n    name = \"ultralytics\"\n\n    def __init__(self, devices: List[int], cfg) -> None:\n        self.devices = list(devices)\n        self.cfg = cfg\n        self.model = None\n        self.weights_path: Optional[Path] = None\n\n    # -- Huấn luyện --------------------------------------------------------\n    def train(self, data_yaml: Path, cfg=None) -> Dict:\n        \"\"\"Huấn luyện trực tiếp trong tiến trình hiện tại.\"\"\"\n        from ultralytics import YOLO\n\n        cfg = cfg or self.cfg\n        spec = resolve_model_spec(cfg.detect.yolo_model)\n        self.model = YOLO(spec)\n\n        # Ưu tiên: CUDA (T4×2 trên Kaggle) → MPS (Apple Silicon) → CPU\n        from ..utils import probe_devices\n        info = probe_devices()\n        if info.kind == \"cuda\":\n            device_arg = self.devices if len(self.devices) > 1 else (\n                self.devices[0] if self.devices else \"cpu\"\n            )\n        elif info.kind == \"mps\":\n            device_arg = \"mps\"\n        else:\n            device_arg = \"cpu\"\n        LOG.info(\"Bắt đầu huấn luyện YOLO trên thiết bị: %s (%s)\", device_arg, info.kind)\n\n        results = self.model.train(\n            data=str(data_yaml),\n            epochs=cfg.detect.epochs,\n            imgsz=cfg.detect.imgsz,\n            batch=cfg.detect.batch_size,\n            workers=cfg.detect.workers,\n            lr0=cfg.detect.lr0,\n            patience=cfg.detect.patience,\n            device=device_arg,\n            project=str(cfg.dir_runs),\n            name=\"yolo_sar\",\n            exist_ok=True,\n            seed=cfg.seed,\n            pretrained=spec.endswith(\".pt\"),\n            verbose=True,\n            plots=True,\n            # Ảnh radar là ảnh cường độ đơn kênh: tắt các phép tăng cường màu\n            hsv_h=0.0, hsv_s=0.0, hsv_v=0.25,\n            degrees=180.0,   # hướng tàu là bất kỳ nên xoay toàn dải là hợp lý\n            fliplr=0.5, flipud=0.5,\n            mosaic=0.6, mixup=0.0, translate=0.08, scale=0.35,\n        )\n\n        best = Path(cfg.dir_runs) / \"yolo_sar\" / \"weights\" / \"best.pt\"\n        self.weights_path = best if best.exists() else None\n        return {\n            \"backend\": self.name,\n            \"weights\": str(best),\n            \"devices\": self.devices,\n            \"epochs\": cfg.detect.epochs,\n        }\n\n    # -- Nạp trọng số ------------------------------------------------------\n    def load(self, weights: Path) -> \"YOLODetector\":\n        from ultralytics import YOLO\n\n        self.model = YOLO(str(weights))\n        self.weights_path = Path(weights)\n        LOG.info(\"Đã nạp trọng số: %s\", weights)\n        return self\n\n    # -- Suy luận ----------------------------------------------------------\n    def predict(\n        self, image_paths: Sequence[Path], conf: float = 0.25\n    ) -> List[DetectionOutput]:\n        if self.model is None:\n            raise RuntimeError(\"Chưa nạp mô hình. Gọi train() hoặc load() trước.\")\n\n        from ..utils import probe_devices\n        info = probe_devices()\n        if info.kind == \"cuda\":\n            device = self.devices[0] if self.devices else \"cpu\"\n        elif info.kind == \"mps\":\n            device = \"mps\"\n        else:\n            device = \"cpu\"\n        outs: List[DetectionOutput] = []\n        batch = 32\n        paths = [Path(p) for p in image_paths]\n\n        for i in range(0, len(paths), batch):\n            chunk = paths[i : i + batch]\n            results = self.model.predict(\n                [str(p) for p in chunk],\n                conf=conf,\n                imgsz=self.cfg.detect.imgsz,\n                device=device,\n                verbose=False,\n            )\n            for p, r in zip(chunk, results):\n                if r.boxes is None or len(r.boxes) == 0:\n                    outs.append(\n                        DetectionOutput(p.stem, np.zeros((0, 4), np.float32), np.zeros((0,), np.float32))\n                    )\n                else:\n                    outs.append(\n                        DetectionOutput(\n                            p.stem,\n                            r.boxes.xyxy.cpu().numpy().astype(np.float32),\n                            r.boxes.conf.cpu().numpy().astype(np.float32),\n                        )\n                    )\n        return outs\n\n    # -- Suy luận phân tán trên hai GPU ------------------------------------\n    def predict_sharded(\n        self, image_paths: Sequence[Path], conf: float = 0.25\n    ) -> List[DetectionOutput]:\n        \"\"\"Chia đôi khối lượng suy luận cho hai GPU nhằm rút ngắn thời gian.\n\n        Khi chỉ có một GPU, hàm này tương đương với ``predict``.\n        \"\"\"\n        if len(self.devices) < 2:\n            return self.predict(image_paths, conf)\n\n        import threading\n\n        paths = [Path(p) for p in image_paths]\n        mid = len(paths) // 2\n        shards = [paths[:mid], paths[mid:]]\n        results: Dict[int, List[DetectionOutput]] = {}\n\n        def worker(idx: int, subset: Sequence[Path], dev: int) -> None:\n            from ultralytics import YOLO\n\n            local = YOLO(str(self.weights_path)) if self.weights_path else self.model\n            outs: List[DetectionOutput] = []\n            for i in range(0, len(subset), 32):\n                chunk = subset[i : i + 32]\n                rs = local.predict(\n                    [str(p) for p in chunk], conf=conf,\n                    imgsz=self.cfg.detect.imgsz, device=dev, verbose=False,\n                )\n                for p, r in zip(chunk, rs):\n                    if r.boxes is None or len(r.boxes) == 0:\n                        outs.append(DetectionOutput(p.stem, np.zeros((0, 4), np.float32), np.zeros((0,), np.float32)))\n                    else:\n                        outs.append(\n                            DetectionOutput(\n                                p.stem,\n                                r.boxes.xyxy.cpu().numpy().astype(np.float32),\n                                r.boxes.conf.cpu().numpy().astype(np.float32),\n                            )\n                        )\n            results[idx] = outs\n\n        threads = [\n            threading.Thread(target=worker, args=(i, shards[i], self.devices[i]))\n            for i in range(2)\n        ]\n        for t in threads:\n            t.start()\n        for t in threads:\n            t.join()\n\n        return results.get(0, []) + results.get(1, [])\n"_FILES['sonarnet/detect/torchvision_backend.py'] = "\"\"\"Bộ phát hiện dự phòng dựa trên Faster R-CNN của Torchvision.\n\nPhương án này tồn tại để bảo đảm pipeline chạy được ngay cả khi notebook bị ngắt\nkết nối mạng và không cài đặt được gói bổ sung, vì Torchvision luôn có sẵn trong\nảnh Python của Kaggle. Mô hình được khởi tạo từ đầu, không dùng trọng số huấn\nluyện trước, nên hoàn toàn không phụ thuộc vào việc tải tệp từ Internet.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom ..utils import get_logger\nfrom .interface import BaseDetector, DetectionOutput\n\nLOG = get_logger(\"detect.torchvision\")\n\n\nclass YoloFormatDataset:\n    \"\"\"Đọc bộ dữ liệu bố trí theo quy ước YOLO và trả về định dạng Torchvision.\"\"\"\n\n    def __init__(self, root: Path, split: str, size: int) -> None:\n        import torch  # noqa: F401\n\n        self.img_dir = Path(root) / \"images\" / split\n        self.lbl_dir = Path(root) / \"labels\" / split\n        self.size = size\n        self.files = sorted(self.img_dir.glob(\"*.png\"))\n        if not self.files:\n            raise FileNotFoundError(f\"Không tìm thấy ảnh trong {self.img_dir}\")\n\n    def __len__(self) -> int:\n        return len(self.files)\n\n    def __getitem__(self, idx: int):\n        import torch\n\n        from ..data.dataset import load_image\n\n        path = self.files[idx]\n        img = load_image(path).astype(np.float32) / 255.0\n        # Nhân bản kênh xám thành ba kênh cho tương thích với mạng nền tiêu chuẩn\n        tensor = torch.from_numpy(img).unsqueeze(0).repeat(3, 1, 1)\n\n        lbl_path = self.lbl_dir / f\"{path.stem}.txt\"\n        boxes: List[List[float]] = []\n        if lbl_path.exists():\n            for line in lbl_path.read_text(encoding=\"utf-8\").splitlines():\n                parts = line.split()\n                if len(parts) != 5:\n                    continue\n                _, xc, yc, w, h = map(float, parts)\n                x1 = (xc - w / 2) * self.size\n                y1 = (yc - h / 2) * self.size\n                x2 = (xc + w / 2) * self.size\n                y2 = (yc + h / 2) * self.size\n                if x2 - x1 > 1 and y2 - y1 > 1:\n                    boxes.append([x1, y1, x2, y2])\n\n        if boxes:\n            boxes_t = torch.tensor(boxes, dtype=torch.float32)\n            labels_t = torch.ones((len(boxes),), dtype=torch.int64)\n        else:\n            boxes_t = torch.zeros((0, 4), dtype=torch.float32)\n            labels_t = torch.zeros((0,), dtype=torch.int64)\n\n        target = {\"boxes\": boxes_t, \"labels\": labels_t,\n                  \"image_id\": torch.tensor([idx])}\n        return tensor, target, path.stem\n\n\ndef _collate(batch):\n    imgs, targets, ids = zip(*batch)\n    return list(imgs), list(targets), list(ids)\n\n\nclass TorchvisionDetector(BaseDetector):\n    name = \"torchvision\"\n\n    def __init__(self, devices: List[int], cfg) -> None:\n        self.devices = list(devices)\n        self.cfg = cfg\n        self.model = None\n        self.weights_path: Optional[Path] = None\n\n    def _device(self):  # dùng probe_devices để hỗ trợ CUDA / MPS / CPU\n        from ..utils import torch_device\n        return torch_device(self.devices[0] if self.devices else 0)\n\n    def _device_cuda_only(self):\n        import torch\n\n        return torch.device(f\"cuda:{self.devices[0]}\") if self.devices else torch.device(\"cpu\")\n\n    def _build_model(self):\n        import torchvision\n        from torchvision.models.detection.faster_rcnn import FastRCNNPredictor\n        from torchvision.models.detection.rpn import AnchorGenerator\n\n        # Kích thước neo nhỏ vì phương tiện chỉ chiếm vài chục điểm ảnh\n        anchor_gen = AnchorGenerator(\n            sizes=((8,), (16,), (32,), (64,), (128,)),\n            aspect_ratios=((0.5, 1.0, 2.0),) * 5,\n        )\n        model = torchvision.models.detection.fasterrcnn_resnet50_fpn(\n            weights=None,\n            weights_backbone=None,\n            num_classes=2,            # nền và phương tiện\n            rpn_anchor_generator=anchor_gen,\n            min_size=self.cfg.detect.imgsz,\n            max_size=self.cfg.detect.imgsz,\n            box_detections_per_img=64,\n        )\n        return model\n\n    # -- Huấn luyện --------------------------------------------------------\n    def train(self, data_yaml: Path, cfg=None) -> Dict:\n        import torch\n        from torch.utils.data import DataLoader\n\n        cfg = cfg or self.cfg\n        root = Path(data_yaml).parent\n        device = self._device()\n\n        train_ds = YoloFormatDataset(root, \"train\", cfg.data.scene_size)\n        val_ds = YoloFormatDataset(root, \"val\", cfg.data.scene_size)\n\n        # Faster R-CNN tiêu tốn bộ nhớ hơn YOLO nên giảm kích thước lô\n        batch = max(2, cfg.detect.batch_size // 8)\n        use_amp = device.type == \"cuda\"\n        train_dl = DataLoader(\n            train_ds, batch_size=batch, shuffle=True,\n            num_workers=2, collate_fn=_collate, pin_memory=(device.type == \"cuda\"),\n        )\n\n        self.model = self._build_model().to(device)\n        params = [p for p in self.model.parameters() if p.requires_grad]\n        optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=5e-4)\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(\n            optimizer, T_max=max(1, cfg.detect.epochs)\n        )\n        scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n\n        LOG.info(\n            \"Huấn luyện Faster R-CNN: %d chu kỳ, lô %d, thiết bị %s\",\n            cfg.detect.epochs, batch, device,\n        )\n\n        for epoch in range(cfg.detect.epochs):\n            self.model.train()\n            running = 0.0\n            for imgs, targets, _ in train_dl:\n                imgs = [i.to(device, non_blocking=True) for i in imgs]\n                targets = [\n                    {k: v.to(device) for k, v in t.items()} for t in targets\n                ]\n                optimizer.zero_grad(set_to_none=True)\n                with torch.amp.autocast(\"cuda\", enabled=use_amp):\n                    loss_dict = self.model(imgs, targets)\n                    loss = sum(loss_dict.values())\n                scaler.scale(loss).backward()\n                scaler.step(optimizer)\n                scaler.update()\n                running += float(loss.detach())\n            scheduler.step()\n            LOG.info(\n                \"  chu kỳ %2d/%d — hàm mất mát trung bình %.4f\",\n                epoch + 1, cfg.detect.epochs, running / max(1, len(train_dl)),\n            )\n\n        out_dir = Path(cfg.dir_runs) / \"frcnn_sar\" / \"weights\"\n        out_dir.mkdir(parents=True, exist_ok=True)\n        self.weights_path = out_dir / \"best.pt\"\n        torch.save(self.model.state_dict(), self.weights_path)\n        LOG.info(\"Đã lưu trọng số: %s\", self.weights_path)\n\n        return {\n            \"backend\": self.name,\n            \"weights\": str(self.weights_path),\n            \"devices\": self.devices,\n            \"epochs\": cfg.detect.epochs,\n        }\n\n    # -- Nạp trọng số ------------------------------------------------------\n    def load(self, weights: Path) -> \"TorchvisionDetector\":\n        import torch\n\n        self.model = self._build_model()\n        state = torch.load(str(weights), map_location=\"cpu\")\n        self.model.load_state_dict(state)\n        self.model.to(self._device()).eval()\n        self.weights_path = Path(weights)\n        return self\n\n    # -- Suy luận ----------------------------------------------------------\n    def predict(\n        self, image_paths: Sequence[Path], conf: float = 0.25\n    ) -> List[DetectionOutput]:\n        import torch\n\n        from ..data.dataset import load_image\n\n        if self.model is None:\n            raise RuntimeError(\"Chưa nạp mô hình. Gọi train() hoặc load() trước.\")\n\n        device = self._device()\n        self.model.eval()\n        outs: List[DetectionOutput] = []\n        paths = [Path(p) for p in image_paths]\n        batch = 8\n\n        with torch.no_grad():\n            for i in range(0, len(paths), batch):\n                chunk = paths[i : i + batch]\n                tensors = []\n                for p in chunk:\n                    arr = load_image(p).astype(np.float32) / 255.0\n                    t = torch.from_numpy(arr).unsqueeze(0).repeat(3, 1, 1)\n                    tensors.append(t.to(device))\n                preds = self.model(tensors)\n                for p, pr in zip(chunk, preds):\n                    boxes = pr[\"boxes\"].cpu().numpy().astype(np.float32)\n                    scores = pr[\"scores\"].cpu().numpy().astype(np.float32)\n                    keep = scores >= conf\n                    outs.append(DetectionOutput(p.stem, boxes[keep], scores[keep]))\n        return outs\n\n    def predict_sharded(\n        self, image_paths: Sequence[Path], conf: float = 0.25\n    ) -> List[DetectionOutput]:\n        return self.predict(image_paths, conf)\n"for _rel, _content in _FILES.items():    _p = SRC_DIR / _rel    _p.parent.mkdir(parents=True, exist_ok=True)    _p.write_text(_content, encoding='utf-8')print(f'Đã ghi {len(_FILES)} tệp mã nguồn.')for _rel in _FILES:    print('   ', _rel)

### Tầng hợp nhất ảnh radar và tín hiệu AIS

In [ ]:
# Ghi mã nguồn ra đĩa. Nội dung giữ nguyên như trong kho mã nguồn._FILES = {}_FILES['sonarnet/fusion/__init__.py'] = "\"\"\"Tầng hợp nhất ảnh radar và tín hiệu nhận dạng tự động.\"\"\"\n\nfrom .kalman import ConstantVelocityKalman, KalmanEstimate  # noqa: F401\nfrom .matching import Detection, FusionResult, SARAISFusion, detections_from_boxes  # noqa: F401\n"_FILES['sonarnet/fusion/kalman.py'] = "\"\"\"Nội suy quỹ đạo AIS về đúng thời điểm chụp ảnh radar.\n\nBản ghi AIS được phát với nhịp không đều và kèm nhiễu vị trí. Để đối chiếu với\nmột cảnh ảnh radar chụp tại thời điểm ``T``, hệ thống cần ước lượng vị trí của\ntừng phương tiện đúng tại thời điểm đó.\n\nĐiểm mấu chốt về phương pháp\n----------------------------\nThời điểm chụp ảnh nằm **ở giữa** chuỗi bản ghi AIS: hệ thống có cả quan trắc\ntrước lẫn quan trắc sau thời điểm đó. Vì vậy bài toán ở đây là *làm trơn*\n(smoothing) chứ không phải *lọc* (filtering).\n\nMột bộ lọc Kalman tiến thuần tuý chỉ khai thác quan trắc trong quá khứ rồi ngoại\nsuy tới thời điểm mục tiêu. Cách làm đó bỏ phí toàn bộ thông tin phía sau và để\nsai số mô hình tích luỹ theo khoảng ngoại suy — trong thực nghiệm nó còn kém hơn\ncả phép nội suy tuyến tính đơn giản.\n\nMô-đun này cài đặt bộ làm trơn **Rauch–Tung–Striebel**: chạy lọc tiến qua toàn\nbộ chuỗi, sau đó chạy một lượt truy hồi ngược để phân bổ lại thông tin từ tương\nlai về quá khứ. Kết quả là ước lượng tại thời điểm mục tiêu sử dụng **mọi** quan\ntrắc có được, đồng thời vẫn tôn trọng mô hình động học của phương tiện. Nhờ đó\nnhiễu đo được trung bình hoá qua nhiều bản ghi thay vì truyền thẳng vào kết quả\nnhư ở phép nội suy tuyến tính.\n\nVector trạng thái gồm bốn thành phần ``[đông, bắc, vận tốc đông, vận tốc bắc]``\ntính theo mét trong hệ quy chiếu phẳng cục bộ.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom ..data.geo import meters_per_degree\n\n# Độ lệch chuẩn tiên nghiệm của vận tốc, tính theo mét trên giây.\n# Giá trị 10 m/s tương ứng khoảng 20 hải lý trên giờ, bao phủ hầu hết\n# phương tiện đánh bắt và vận tải hoạt động trong vùng biển thí điểm.\nVELOCITY_PRIOR_MS = 10.0\n\n\n@dataclass\nclass KalmanEstimate:\n    \"\"\"Kết quả nội suy tại một thời điểm.\"\"\"\n\n    lat: float\n    lon: float\n    speed_kn: float\n    course_deg: float\n    # Bán kính bất định một xích-ma của vị trí, tính bằng mét\n    position_sigma_m: float\n    # Khoảng cách thời gian tới bản ghi AIS gần nhất, tính bằng giây\n    gap_to_nearest_s: float\n    n_observations: int\n\n\nclass ConstantVelocityKalman:\n    \"\"\"Bộ làm trơn Kalman với mô hình vận tốc không đổi.\n\n    Tham số ``process_noise`` là độ lệch chuẩn của gia tốc nhiễu trắng, đơn vị\n    mét trên giây bình phương; nó quyết định mức độ hệ thống cho phép phương tiện\n    đổi hướng và đổi tốc giữa hai quan trắc. Tham số ``measurement_noise`` là độ\n    lệch chuẩn sai số vị trí của bản ghi AIS, đơn vị mét.\n    \"\"\"\n\n    def __init__(self, process_noise: float = 0.35, measurement_noise: float = 45.0):\n        self.q = float(process_noise)\n        self.r = float(measurement_noise)\n\n    # -- Ma trận mô hình ---------------------------------------------------\n    @staticmethod\n    def _transition(dt: float) -> np.ndarray:\n        F = np.eye(4)\n        F[0, 2] = dt\n        F[1, 3] = dt\n        return F\n\n    def _process_cov(self, dt: float) -> np.ndarray:\n        \"\"\"Hiệp phương sai nhiễu quá trình cho mô hình gia tốc trắng rời rạc.\"\"\"\n        q = self.q**2\n        dt2 = dt * dt\n        dt3 = dt2 * dt / 2.0\n        dt4 = dt2 * dt2 / 4.0\n        Q = np.zeros((4, 4))\n        Q[0, 0] = Q[1, 1] = dt4 * q\n        Q[2, 2] = Q[3, 3] = dt2 * q\n        Q[0, 2] = Q[2, 0] = Q[1, 3] = Q[3, 1] = dt3 * q\n        # Thêm một lượng rất nhỏ trên đường chéo để bảo đảm khả nghịch\n        return Q + np.eye(4) * 1e-9\n\n    # -- Làm trơn ----------------------------------------------------------\n    def smooth_to_time(\n        self, records: Sequence[dict], target_time_s: float\n    ) -> Optional[KalmanEstimate]:\n        \"\"\"Ước lượng trạng thái của một phương tiện tại ``target_time_s``.\n\n        ``records`` là các bản ghi AIS của cùng một MMSI, mỗi bản ghi có các khoá\n        ``timestamp``, ``lat``, ``lon``. Danh sách không cần sắp xếp trước.\n\n        Thuật toán gồm ba giai đoạn: dựng trục thời gian chứa cả các mốc quan\n        trắc lẫn thời điểm mục tiêu; chạy lọc Kalman tiến trên trục đó; chạy\n        truy hồi Rauch–Tung–Striebel ngược để thu được ước lượng làm trơn.\n        \"\"\"\n        if not records:\n            return None\n\n        recs = sorted(records, key=lambda r: float(r[\"timestamp\"]))\n\n        # Hệ quy chiếu phẳng cục bộ, lấy gốc tại quan trắc đầu tiên\n        lat_ref = float(recs[0][\"lat\"])\n        lon_ref = float(recs[0][\"lon\"])\n        m_lat, m_lon = meters_per_degree(lat_ref)\n\n        def to_local(lat: float, lon: float) -> Tuple[float, float]:\n            return ((lon - lon_ref) * m_lon, (lat - lat_ref) * m_lat)\n\n        # ---- Giai đoạn 1: dựng trục thời gian ---------------------------\n        obs_times = [float(r[\"timestamp\"]) for r in recs]\n        timeline = sorted(set(obs_times + [float(target_time_s)]))\n        index_of = {t: i for i, t in enumerate(timeline)}\n        target_idx = index_of[float(target_time_s)]\n\n        # Nhóm quan trắc theo mốc thời gian, phòng trường hợp trùng nhãn thời gian\n        obs_at: Dict[int, List[np.ndarray]] = {}\n        for r in recs:\n            i = index_of[float(r[\"timestamp\"])]\n            e, n = to_local(float(r[\"lat\"]), float(r[\"lon\"]))\n            obs_at.setdefault(i, []).append(np.array([e, n], dtype=float))\n\n        N = len(timeline)\n        H = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])\n        R = np.eye(2) * (self.r**2)\n\n        # ---- Giai đoạn 2: lọc tiến ---------------------------------------\n        x_pred = [np.zeros(4) for _ in range(N)]\n        P_pred = [np.eye(4) for _ in range(N)]\n        x_filt = [np.zeros(4) for _ in range(N)]\n        P_filt = [np.eye(4) for _ in range(N)]\n        F_list = [np.eye(4) for _ in range(N)]\n\n        # Khởi tạo tại mốc đầu tiên của trục thời gian\n        first_obs = obs_at.get(0)\n        if first_obs is not None:\n            x0 = np.array([first_obs[0][0], first_obs[0][1], 0.0, 0.0])\n            P0 = np.diag([self.r**2, self.r**2,\n                          VELOCITY_PRIOR_MS**2, VELOCITY_PRIOR_MS**2])\n        else:\n            # Mốc đầu là thời điểm mục tiêu, nằm trước mọi quan trắc.\n            # Khởi tạo lỏng; lượt truy hồi ngược sẽ hiệu chỉnh lại.\n            e0, n0 = to_local(float(recs[0][\"lat\"]), float(recs[0][\"lon\"]))\n            x0 = np.array([e0, n0, 0.0, 0.0])\n            P0 = np.diag([(5.0 * self.r) ** 2, (5.0 * self.r) ** 2,\n                          VELOCITY_PRIOR_MS**2, VELOCITY_PRIOR_MS**2])\n\n        x_pred[0], P_pred[0] = x0.copy(), P0.copy()\n        x, P = x0.copy(), P0.copy()\n        if first_obs is not None:\n            for z in first_obs:\n                y = z - H @ x\n                S = H @ P @ H.T + R\n                K = P @ H.T @ np.linalg.inv(S)\n                x = x + K @ y\n                P = (np.eye(4) - K @ H) @ P\n        x_filt[0], P_filt[0] = x.copy(), P.copy()\n\n        for k in range(1, N):\n            dt = timeline[k] - timeline[k - 1]\n            F = self._transition(dt)\n            F_list[k] = F\n            x = F @ x_filt[k - 1]\n            P = F @ P_filt[k - 1] @ F.T + self._process_cov(dt)\n            x_pred[k], P_pred[k] = x.copy(), P.copy()\n\n            for z in obs_at.get(k, []):\n                y = z - H @ x\n                S = H @ P @ H.T + R\n                K = P @ H.T @ np.linalg.inv(S)\n                x = x + K @ y\n                P = (np.eye(4) - K @ H) @ P\n            x_filt[k], P_filt[k] = x.copy(), P.copy()\n\n        # ---- Giai đoạn 3: truy hồi Rauch–Tung–Striebel --------------------\n        x_smooth = [xf.copy() for xf in x_filt]\n        P_smooth = [Pf.copy() for Pf in P_filt]\n\n        for k in range(N - 2, -1, -1):\n            try:\n                P_pred_inv = np.linalg.inv(P_pred[k + 1])\n            except np.linalg.LinAlgError:  # pragma: no cover\n                P_pred_inv = np.linalg.pinv(P_pred[k + 1])\n            C = P_filt[k] @ F_list[k + 1].T @ P_pred_inv\n            x_smooth[k] = x_filt[k] + C @ (x_smooth[k + 1] - x_pred[k + 1])\n            P_smooth[k] = P_filt[k] + C @ (P_smooth[k + 1] - P_pred[k + 1]) @ C.T\n\n        xs = x_smooth[target_idx]\n        Ps = P_smooth[target_idx]\n\n        lat = lat_ref + xs[1] / m_lat\n        lon = lon_ref + xs[0] / m_lon\n        speed_ms = float(np.hypot(xs[2], xs[3]))\n        course = float(np.degrees(np.arctan2(xs[2], xs[3])) % 360.0)\n        sigma = float(np.sqrt(max((Ps[0, 0] + Ps[1, 1]) / 2.0, 0.0)))\n        gap = float(min(abs(t - float(target_time_s)) for t in obs_times))\n\n        return KalmanEstimate(\n            lat=float(lat), lon=float(lon),\n            speed_kn=speed_ms / 0.514444,\n            course_deg=course,\n            position_sigma_m=sigma,\n            gap_to_nearest_s=gap,\n            n_observations=len(recs),\n        )\n\n\ndef linear_interpolate_to_time(\n    records: Sequence[dict], target_time_s: float\n) -> Optional[KalmanEstimate]:\n    \"\"\"Nội suy tuyến tính — phương án cơ sở để so sánh với bộ làm trơn Kalman.\n\n    Phép nội suy này chỉ dùng hai bản ghi lân cận thời điểm mục tiêu, nên toàn bộ\n    nhiễu đo của hai điểm mút truyền thẳng vào kết quả mà không được trung bình\n    hoá qua chuỗi quan trắc.\n    \"\"\"\n    if not records:\n        return None\n\n    recs = sorted(records, key=lambda r: float(r[\"timestamp\"]))\n    times = np.array([float(r[\"timestamp\"]) for r in recs], dtype=float)\n    lats = np.array([float(r[\"lat\"]) for r in recs], dtype=float)\n    lons = np.array([float(r[\"lon\"]) for r in recs], dtype=float)\n\n    lat = float(np.interp(target_time_s, times, lats))\n    lon = float(np.interp(target_time_s, times, lons))\n    gap = float(np.min(np.abs(times - target_time_s)))\n\n    if len(recs) >= 2:\n        i = int(np.clip(np.searchsorted(times, target_time_s), 1, len(recs) - 1))\n        dt = max(times[i] - times[i - 1], 1e-3)\n        m_lat, m_lon = meters_per_degree(lats[i])\n        de = (lons[i] - lons[i - 1]) * m_lon\n        dn = (lats[i] - lats[i - 1]) * m_lat\n        speed = float(np.hypot(de, dn) / dt / 0.514444)\n        course = float(np.degrees(np.arctan2(de, dn)) % 360.0)\n    else:\n        speed, course = 0.0, 0.0\n\n    return KalmanEstimate(\n        lat=lat, lon=lon, speed_kn=speed, course_deg=course,\n        position_sigma_m=float(\"nan\"), gap_to_nearest_s=gap,\n        n_observations=len(recs),\n    )\n"_FILES['sonarnet/fusion/matching.py'] = "\"\"\"Hợp nhất phát hiện trên ảnh radar với tín hiệu AIS.\n\nQuy trình gồm ba bước, đúng như mô tả trong đề xuất:\n\n1. **Nội suy quỹ đạo AIS** về đúng thời điểm chụp ảnh bằng bộ lọc Kalman.\n2. **Ghép cặp tối ưu toàn cục** giữa tập phát hiện và tập vị trí AIS bằng thuật\n   toán Hungarian, với ma trận chi phí là khoảng cách địa lý và ngưỡng chặn trên.\n3. **Phân loại trạng thái định danh** của từng phát hiện thành một trong ba mức:\n   phát tín hiệu trung thực, phát tín hiệu nhưng kích thước khai báo sai lệch,\n   hoặc không phát tín hiệu.\n\nBước thứ ba là điểm mấu chốt: một phương tiện xuất hiện rõ trên ảnh radar mà\nkhông có bản ghi AIS tương ứng chính là dấu hiệu đặc trưng của hành vi chủ động\nngắt định danh.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom ..data.geo import haversine_m\nfrom ..data.simulator import AIS_MISMATCH, AIS_OK, DARK\nfrom .kalman import ConstantVelocityKalman, KalmanEstimate, linear_interpolate_to_time\n\ntry:\n    from scipy.optimize import linear_sum_assignment as _hungarian\n\n    _HAS_SCIPY = True\nexcept Exception:  # pragma: no cover\n    _HAS_SCIPY = False\n\n\ndef _greedy_assignment(cost: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n    \"\"\"Ghép cặp tham lam — phương án thay thế khi thiếu SciPy.\"\"\"\n    cost = cost.copy()\n    rows, cols = [], []\n    n, m = cost.shape\n    for _ in range(min(n, m)):\n        idx = int(np.argmin(cost))\n        r, c = divmod(idx, m)\n        if not np.isfinite(cost[r, c]):\n            break\n        rows.append(r)\n        cols.append(c)\n        cost[r, :] = np.inf\n        cost[:, c] = np.inf\n    return np.array(rows, dtype=int), np.array(cols, dtype=int)\n\n\ndef solve_assignment(cost: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n    \"\"\"Giải bài toán gán tối ưu, ưu tiên thuật toán Hungarian của SciPy.\"\"\"\n    if cost.size == 0:\n        return np.array([], dtype=int), np.array([], dtype=int)\n    if _HAS_SCIPY:\n        # Thay vô cực bằng một chi phí hữu hạn rất lớn để SciPy xử lý được\n        big = np.nanmax(cost[np.isfinite(cost)]) if np.any(np.isfinite(cost)) else 1.0\n        safe = np.where(np.isfinite(cost), cost, big * 1e6 + 1e6)\n        r, c = _hungarian(safe)\n        return r, c\n    return _greedy_assignment(cost)\n\n\n@dataclass\nclass Detection:\n    \"\"\"Một phát hiện trên ảnh radar, đã quy về toạ độ địa lý.\"\"\"\n\n    det_id: int\n    lat: float\n    lon: float\n    length_m: float          # chiều dài ước lượng từ khung bao\n    width_m: float\n    bbox: Tuple[float, float, float, float]\n    score: float = 1.0\n\n\n@dataclass\nclass FusionResult:\n    \"\"\"Kết quả hợp nhất cho một cảnh.\"\"\"\n\n    scene_id: str\n    # Trạng thái dự đoán cho từng phát hiện, cùng thứ tự với danh sách đầu vào\n    states: List[str] = field(default_factory=list)\n    matched_mmsi: List[Optional[int]] = field(default_factory=list)\n    match_distance_m: List[float] = field(default_factory=list)\n    # Các MMSI có phát AIS nhưng không ghép được với phát hiện nào\n    unmatched_mmsi: List[int] = field(default_factory=list)\n    n_detections: int = 0\n    n_ais_vessels: int = 0\n\n    def summary(self) -> Dict[str, int]:\n        out = {s: 0 for s in (AIS_OK, AIS_MISMATCH, DARK)}\n        for s in self.states:\n            out[s] = out.get(s, 0) + 1\n        return out\n\n\nclass SARAISFusion:\n    \"\"\"Bộ hợp nhất ảnh radar và tín hiệu AIS.\"\"\"\n\n    def __init__(self, cfg, use_kalman: bool = True) -> None:\n        self.cfg = cfg\n        self.use_kalman = use_kalman\n        self.kf = ConstantVelocityKalman(\n            process_noise=cfg.fusion.kalman_process_noise,\n            measurement_noise=cfg.fusion.kalman_measurement_noise,\n        )\n\n    # -- Bước 1: nội suy --------------------------------------------------\n    def interpolate_ais(\n        self, ais_records: Sequence[dict], capture_time_s: float\n    ) -> Dict[int, KalmanEstimate]:\n        \"\"\"Ước lượng vị trí của mọi phương tiện có AIS tại thời điểm chụp.\"\"\"\n        by_mmsi: Dict[int, List[dict]] = {}\n        for rec in ais_records:\n            by_mmsi.setdefault(int(rec[\"mmsi\"]), []).append(rec)\n\n        out: Dict[int, KalmanEstimate] = {}\n        for mmsi, recs in by_mmsi.items():\n            est = (\n                self.kf.smooth_to_time(recs, capture_time_s)\n                if self.use_kalman\n                else linear_interpolate_to_time(recs, capture_time_s)\n            )\n            if est is not None:\n                est_declared = float(recs[0].get(\"declared_length_m\", float(\"nan\")))\n                out[mmsi] = est\n                setattr(out[mmsi], \"declared_length_m\", est_declared)\n        return out\n\n    # -- Bước 2: ghép cặp -------------------------------------------------\n    def match(\n        self,\n        detections: Sequence[Detection],\n        ais_estimates: Dict[int, KalmanEstimate],\n    ) -> Tuple[Dict[int, int], Dict[int, float]]:\n        \"\"\"Ghép phát hiện với vị trí AIS. Trả về ánh xạ chỉ số phát hiện → MMSI.\"\"\"\n        if not detections or not ais_estimates:\n            return {}, {}\n\n        mmsis = list(ais_estimates.keys())\n        det_lat = np.array([d.lat for d in detections])\n        det_lon = np.array([d.lon for d in detections])\n        ais_lat = np.array([ais_estimates[m].lat for m in mmsis])\n        ais_lon = np.array([ais_estimates[m].lon for m in mmsis])\n\n        # Ma trận chi phí: khoảng cách địa lý giữa mọi cặp\n        cost = haversine_m(\n            det_lat[:, None], det_lon[:, None], ais_lat[None, :], ais_lon[None, :]\n        )\n        thr = self.cfg.fusion.max_match_distance_m\n        cost_masked = np.where(cost <= thr, cost, np.inf)\n\n        rows, cols = solve_assignment(cost_masked)\n\n        det_to_mmsi: Dict[int, int] = {}\n        distances: Dict[int, float] = {}\n        for r, c in zip(rows, cols):\n            d = float(cost[r, c])\n            if d <= thr:\n                det_to_mmsi[int(r)] = int(mmsis[int(c)])\n                distances[int(r)] = d\n        return det_to_mmsi, distances\n\n    # -- Bước 3: phân loại trạng thái -------------------------------------\n    def classify_states(\n        self,\n        detections: Sequence[Detection],\n        det_to_mmsi: Dict[int, int],\n        ais_estimates: Dict[int, KalmanEstimate],\n    ) -> List[str]:\n        \"\"\"Gán một trong ba trạng thái định danh cho từng phát hiện.\"\"\"\n        ratio_thr = self.cfg.fusion.size_mismatch_ratio\n        states: List[str] = []\n        for i, det in enumerate(detections):\n            mmsi = det_to_mmsi.get(i)\n            if mmsi is None:\n                states.append(DARK)\n                continue\n            declared = getattr(ais_estimates[mmsi], \"declared_length_m\", float(\"nan\"))\n            if not np.isfinite(declared) or declared <= 0 or det.length_m <= 0:\n                states.append(AIS_OK)\n                continue\n            # Sai lệch tương đối giữa kích thước khai báo và kích thước quan sát\n            rel = abs(declared - det.length_m) / max(declared, det.length_m)\n            states.append(AIS_MISMATCH if rel >= ratio_thr else AIS_OK)\n        return states\n\n    # -- Giao diện tổng hợp -----------------------------------------------\n    def run(\n        self,\n        scene_id: str,\n        detections: Sequence[Detection],\n        ais_records: Sequence[dict],\n        capture_time_s: float,\n    ) -> FusionResult:\n        est = self.interpolate_ais(ais_records, capture_time_s)\n        det_to_mmsi, dists = self.match(detections, est)\n        states = self.classify_states(detections, det_to_mmsi, est)\n\n        matched = set(det_to_mmsi.values())\n        return FusionResult(\n            scene_id=scene_id,\n            states=states,\n            matched_mmsi=[det_to_mmsi.get(i) for i in range(len(detections))],\n            match_distance_m=[dists.get(i, float(\"nan\")) for i in range(len(detections))],\n            unmatched_mmsi=[m for m in est.keys() if m not in matched],\n            n_detections=len(detections),\n            n_ais_vessels=len(est),\n        )\n\n\ndef detections_from_boxes(\n    boxes: np.ndarray, scores: np.ndarray, georef, pixel_spacing_m: float\n) -> List[Detection]:\n    \"\"\"Chuyển khung bao trên ảnh thành các phát hiện có toạ độ địa lý.\"\"\"\n    dets: List[Detection] = []\n    for i, (b, s) in enumerate(zip(boxes, scores)):\n        x1, y1, x2, y2 = [float(v) for v in b]\n        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0\n        lon, lat = georef.pixel_to_lonlat(cx, cy)\n        w_px, h_px = abs(x2 - x1), abs(y2 - y1)\n        # Chiều dài lấy theo cạnh lớn của khung bao, đã trừ phần đệm khi gán nhãn\n        length_m = max(w_px, h_px) * pixel_spacing_m\n        width_m = min(w_px, h_px) * pixel_spacing_m\n        dets.append(\n            Detection(\n                det_id=i, lat=float(lat), lon=float(lon),\n                length_m=length_m, width_m=width_m,\n                bbox=(x1, y1, x2, y2), score=float(s),\n            )\n        )\n    return dets\n"for _rel, _content in _FILES.items():    _p = SRC_DIR / _rel    _p.parent.mkdir(parents=True, exist_ok=True)    _p.write_text(_content, encoding='utf-8')print(f'Đã ghi {len(_FILES)} tệp mã nguồn.')for _rel in _FILES:    print('   ', _rel)

### Tầng phân loại hành vi hoạt động

In [ ]:
# Ghi mã nguồn ra đĩa. Nội dung giữ nguyên như trong kho mã nguồn._FILES = {}_FILES['sonarnet/behavior/__init__.py'] = "\"\"\"Tầng phân loại hành vi hoạt động của phương tiện.\"\"\"\n\nfrom .classifier import BehaviorClassifier, BehaviorReport, train_and_evaluate  # noqa: F401\nfrom .features import FEATURE_NAMES, extract_features, features_matrix  # noqa: F401\n"_FILES['sonarnet/behavior/features.py'] = "\"\"\"Trích xuất đặc trưng động học từ quỹ đạo phương tiện.\n\nBộ đặc trưng được thiết kế để phân biệt bốn nhóm hành vi khai thác dựa trên các\ndấu hiệu vật lý bền vững, không phụ thuộc vào vị trí địa lý tuyệt đối:\n\n* **Thống kê tốc độ** — phân biệt quá cảnh (nhanh, ổn định) với neo đậu (gần như\n  đứng yên) và các hoạt động khai thác (trung bình, biến động).\n* **Thống kê đổi hướng** — câu có bước ngẫu nhiên mạnh về hướng, trong khi kéo\n  lưới giữ hướng ổn định trên từng luống rồi quay đầu gấp.\n* **Tỉ lệ dừng** — tỉ lệ thời gian tốc độ dưới ngưỡng, rất cao ở nhóm neo đậu.\n* **Độ thẳng của đường đi** — tỉ số giữa khoảng cách hai đầu và tổng quãng đường,\n  gần 1 với quá cảnh và rất nhỏ với neo đậu.\n* **Bán kính hồi chuyển** — đo mức độ trải rộng không gian của quỹ đạo.\n* **Đặc trưng phổ của chuỗi hướng** — nắm bắt tính tuần hoàn của mẫu răng lược\n  đặc trưng cho hoạt động kéo lưới.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Dict, List, Sequence\n\nimport numpy as np\n\nfrom ..data.geo import meters_per_degree\nfrom ..data.tracks import KNOT_TO_MS, Track\n\n# Ngưỡng coi là đang dừng, tính theo hải lý trên giờ\nSTOP_SPEED_KN = 0.8\n\nFEATURE_NAMES: List[str] = [\n    \"speed_mean\", \"speed_std\", \"speed_median\", \"speed_p10\", \"speed_p90\",\n    \"speed_max\", \"speed_cv\",\n    \"turn_abs_mean\", \"turn_abs_std\", \"turn_abs_p90\", \"turn_sign_changes\",\n    \"stop_fraction\", \"moving_fraction\",\n    \"straightness\", \"radius_gyration_m\", \"path_length_m\", \"net_displacement_m\",\n    \"bbox_diag_m\", \"area_coverage_ratio\",\n    \"accel_abs_mean\", \"accel_abs_std\",\n    \"course_autocorr_lag1\", \"course_spectral_peak\", \"course_spectral_ratio\",\n    \"speed_autocorr_lag1\", \"n_points\",\n]\n\n\ndef _angular_difference(a: np.ndarray) -> np.ndarray:\n    \"\"\"Chênh lệch hướng liên tiếp, quy về khoảng [-180, 180] độ.\"\"\"\n    d = np.diff(a)\n    return (d + 180.0) % 360.0 - 180.0\n\n\ndef _local_xy(lats: np.ndarray, lons: np.ndarray) -> tuple:\n    \"\"\"Quy đổi kinh vĩ độ về toạ độ phẳng cục bộ, đơn vị mét.\"\"\"\n    lat0 = float(np.mean(lats))\n    m_lat, m_lon = meters_per_degree(lat0)\n    x = (lons - float(np.mean(lons))) * m_lon\n    y = (lats - lat0) * m_lat\n    return x, y\n\n\ndef extract_features(track: Track) -> Dict[str, float]:\n    \"\"\"Tính toàn bộ đặc trưng cho một quỹ đạo.\"\"\"\n    sp = np.asarray(track.speeds_kn, dtype=float)\n    co = np.asarray(track.courses_deg, dtype=float)\n    t = np.asarray(track.times_s, dtype=float)\n    x, y = _local_xy(np.asarray(track.lats), np.asarray(track.lons))\n\n    n = len(sp)\n    dt = float(np.median(np.diff(t))) if n > 1 else 1.0\n\n    # -- Tốc độ ------------------------------------------------------------\n    speed_mean = float(np.mean(sp))\n    speed_std = float(np.std(sp))\n    f: Dict[str, float] = {\n        \"speed_mean\": speed_mean,\n        \"speed_std\": speed_std,\n        \"speed_median\": float(np.median(sp)),\n        \"speed_p10\": float(np.percentile(sp, 10)),\n        \"speed_p90\": float(np.percentile(sp, 90)),\n        \"speed_max\": float(np.max(sp)),\n        \"speed_cv\": float(speed_std / (speed_mean + 1e-6)),\n    }\n\n    # -- Đổi hướng ---------------------------------------------------------\n    turns = _angular_difference(co) if n > 1 else np.zeros(1)\n    abs_turns = np.abs(turns)\n    signs = np.sign(turns)\n    f.update(\n        {\n            \"turn_abs_mean\": float(np.mean(abs_turns)),\n            \"turn_abs_std\": float(np.std(abs_turns)),\n            \"turn_abs_p90\": float(np.percentile(abs_turns, 90)),\n            \"turn_sign_changes\": float(\n                np.mean(np.abs(np.diff(signs)) > 0) if len(signs) > 1 else 0.0\n            ),\n        }\n    )\n\n    # -- Dừng và di chuyển -------------------------------------------------\n    stop_frac = float(np.mean(sp < STOP_SPEED_KN))\n    f[\"stop_fraction\"] = stop_frac\n    f[\"moving_fraction\"] = 1.0 - stop_frac\n\n    # -- Hình học đường đi --------------------------------------------------\n    seg = np.hypot(np.diff(x), np.diff(y)) if n > 1 else np.zeros(1)\n    path_len = float(np.sum(seg))\n    net_disp = float(np.hypot(x[-1] - x[0], y[-1] - y[0])) if n > 1 else 0.0\n    f[\"path_length_m\"] = path_len\n    f[\"net_displacement_m\"] = net_disp\n    f[\"straightness\"] = float(net_disp / (path_len + 1e-6))\n\n    cx, cy = float(np.mean(x)), float(np.mean(y))\n    f[\"radius_gyration_m\"] = float(np.sqrt(np.mean((x - cx) ** 2 + (y - cy) ** 2)))\n\n    w = float(np.ptp(x))\n    h = float(np.ptp(y))\n    f[\"bbox_diag_m\"] = float(np.hypot(w, h))\n    f[\"area_coverage_ratio\"] = float(path_len / (np.hypot(w, h) + 1e-6))\n\n    # -- Gia tốc -----------------------------------------------------------\n    acc = np.diff(sp * KNOT_TO_MS) / dt if n > 1 else np.zeros(1)\n    f[\"accel_abs_mean\"] = float(np.mean(np.abs(acc)))\n    f[\"accel_abs_std\"] = float(np.std(np.abs(acc)))\n\n    # -- Tự tương quan và phổ ----------------------------------------------\n    def _autocorr1(v: np.ndarray) -> float:\n        if len(v) < 3:\n            return 0.0\n        v = v - np.mean(v)\n        denom = float(np.dot(v, v))\n        if denom < 1e-9:\n            return 0.0\n        return float(np.dot(v[:-1], v[1:]) / denom)\n\n    f[\"course_autocorr_lag1\"] = _autocorr1(turns)\n    f[\"speed_autocorr_lag1\"] = _autocorr1(sp)\n\n    # Phổ của chuỗi đổi hướng: mẫu răng lược của kéo lưới tạo đỉnh phổ rõ rệt\n    if len(turns) >= 8:\n        sig = turns - np.mean(turns)\n        spec = np.abs(np.fft.rfft(sig)) ** 2\n        spec = spec[1:]  # bỏ thành phần một chiều\n        if spec.size and float(np.sum(spec)) > 1e-12:\n            peak = int(np.argmax(spec))\n            f[\"course_spectral_peak\"] = float(peak / len(spec))\n            f[\"course_spectral_ratio\"] = float(spec[peak] / np.sum(spec))\n        else:\n            f[\"course_spectral_peak\"] = 0.0\n            f[\"course_spectral_ratio\"] = 0.0\n    else:\n        f[\"course_spectral_peak\"] = 0.0\n        f[\"course_spectral_ratio\"] = 0.0\n\n    f[\"n_points\"] = float(n)\n    return f\n\n\ndef features_matrix(tracks: Sequence[Track]) -> tuple:\n    \"\"\"Xây ma trận đặc trưng và vector nhãn từ danh sách quỹ đạo.\"\"\"\n    rows = [extract_features(tr) for tr in tracks]\n    X = np.array([[r[name] for name in FEATURE_NAMES] for r in rows], dtype=np.float32)\n    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)\n    y = np.array([tr.behaviour for tr in tracks])\n    return X, y\n"_FILES['sonarnet/behavior/classifier.py'] = "\"\"\"Bộ phân loại hành vi hoạt động của phương tiện.\n\nSử dụng thuật toán tăng cường gradient. Ưu tiên XGBoost nếu có sẵn trong môi\ntrường, nếu không sẽ tự động chuyển sang ``HistGradientBoostingClassifier`` của\nscikit-learn — thư viện luôn có mặt trong ảnh Python của Kaggle. Cả hai lựa chọn\nđều cho chất lượng tương đương trên bài toán bảng số quy mô này.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom ..utils import get_logger, package_available\n\nLOG = get_logger(\"behavior.classifier\")\n\n\n@dataclass\nclass BehaviorReport:\n    \"\"\"Kết quả đánh giá bộ phân loại hành vi.\"\"\"\n\n    backend: str\n    classes: List[str]\n    accuracy: float\n    macro_f1: float\n    per_class_f1: Dict[str, float]\n    confusion_matrix: np.ndarray\n    feature_importance: Optional[Dict[str, float]] = None\n    n_train: int = 0\n    n_test: int = 0\n\n    def to_dict(self) -> Dict:\n        return {\n            \"backend\": self.backend,\n            \"classes\": self.classes,\n            \"accuracy\": float(self.accuracy),\n            \"macro_f1\": float(self.macro_f1),\n            \"per_class_f1\": {k: float(v) for k, v in self.per_class_f1.items()},\n            \"confusion_matrix\": self.confusion_matrix.tolist(),\n            \"n_train\": self.n_train,\n            \"n_test\": self.n_test,\n            \"top_features\": (\n                dict(sorted(self.feature_importance.items(), key=lambda kv: -kv[1])[:12])\n                if self.feature_importance\n                else None\n            ),\n        }\n\n\nclass BehaviorClassifier:\n    \"\"\"Bao bọc thống nhất cho hai thư viện tăng cường gradient.\"\"\"\n\n    def __init__(self, backend: str = \"auto\", seed: int = 0) -> None:\n        if backend == \"auto\":\n            backend = \"xgboost\" if package_available(\"xgboost\") else \"sklearn\"\n        self.backend = backend\n        self.seed = seed\n        self.model = None\n        self.classes_: List[str] = []\n        self._label_to_idx: Dict[str, int] = {}\n\n    # -- Huấn luyện --------------------------------------------------------\n    def fit(self, X: np.ndarray, y: np.ndarray) -> \"BehaviorClassifier\":\n        self.classes_ = sorted(set(map(str, y)))\n        self._label_to_idx = {c: i for i, c in enumerate(self.classes_)}\n        y_idx = np.array([self._label_to_idx[str(v)] for v in y], dtype=int)\n\n        if self.backend == \"xgboost\":\n            import xgboost as xgb\n\n            self.model = xgb.XGBClassifier(\n                n_estimators=450,\n                max_depth=6,\n                learning_rate=0.06,\n                subsample=0.85,\n                colsample_bytree=0.85,\n                reg_lambda=1.2,\n                objective=\"multi:softprob\",\n                num_class=len(self.classes_),\n                tree_method=\"hist\",\n                random_state=self.seed,\n                n_jobs=-1,\n                eval_metric=\"mlogloss\",\n            )\n        else:\n            from sklearn.ensemble import HistGradientBoostingClassifier\n\n            self.model = HistGradientBoostingClassifier(\n                max_iter=450,\n                max_depth=8,\n                learning_rate=0.07,\n                l2_regularization=1.0,\n                random_state=self.seed,\n            )\n        self.model.fit(X, y_idx)\n        return self\n\n    # -- Suy luận ----------------------------------------------------------\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        idx = self.model.predict(X)\n        return np.array([self.classes_[int(i)] for i in idx])\n\n    def predict_proba(self, X: np.ndarray) -> np.ndarray:\n        return self.model.predict_proba(X)\n\n    # -- Độ quan trọng đặc trưng ------------------------------------------\n    def feature_importance(self, names: Sequence[str]) -> Optional[Dict[str, float]]:\n        try:\n            if self.backend == \"xgboost\":\n                imp = np.asarray(self.model.feature_importances_, dtype=float)\n            else:\n                from sklearn.inspection import permutation_importance  # noqa: F401\n                return None  # bỏ qua để tránh chi phí tính toán lớn\n            total = float(imp.sum()) or 1.0\n            return {n: float(v / total) for n, v in zip(names, imp)}\n        except Exception:\n            return None\n\n    # -- Lưu và nạp --------------------------------------------------------\n    def save(self, path: Path) -> Path:\n        import pickle\n\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        with path.open(\"wb\") as f:\n            pickle.dump(\n                {\n                    \"backend\": self.backend,\n                    \"classes\": self.classes_,\n                    \"model\": self.model,\n                },\n                f,\n            )\n        return path\n\n\ndef macro_f1_score(y_true: np.ndarray, y_pred: np.ndarray, classes: List[str]) -> Tuple[float, Dict[str, float]]:\n    \"\"\"Tính F1 vĩ mô và F1 theo từng lớp, không phụ thuộc scikit-learn.\"\"\"\n    per_class: Dict[str, float] = {}\n    for c in classes:\n        tp = int(np.sum((y_true == c) & (y_pred == c)))\n        fp = int(np.sum((y_true != c) & (y_pred == c)))\n        fn = int(np.sum((y_true == c) & (y_pred != c)))\n        prec = tp / (tp + fp) if (tp + fp) else 0.0\n        rec = tp / (tp + fn) if (tp + fn) else 0.0\n        per_class[c] = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0\n    return float(np.mean(list(per_class.values()))), per_class\n\n\ndef confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, classes: List[str]) -> np.ndarray:\n    idx = {c: i for i, c in enumerate(classes)}\n    cm = np.zeros((len(classes), len(classes)), dtype=int)\n    for t, p in zip(y_true, y_pred):\n        cm[idx[str(t)], idx[str(p)]] += 1\n    return cm\n\n\ndef train_and_evaluate(\n    X: np.ndarray,\n    y: np.ndarray,\n    feature_names: Sequence[str],\n    cfg,\n) -> Tuple[BehaviorClassifier, BehaviorReport]:\n    \"\"\"Chia tập, huấn luyện và đánh giá bộ phân loại hành vi.\"\"\"\n    from sklearn.model_selection import train_test_split\n\n    X_tr, X_te, y_tr, y_te = train_test_split(\n        X, y,\n        test_size=cfg.behavior.test_size,\n        random_state=cfg.seed,\n        stratify=y,\n    )\n\n    clf = BehaviorClassifier(backend=cfg.behavior.backend, seed=cfg.seed)\n    clf.fit(X_tr, y_tr)\n    LOG.info(\n        \"Đã huấn luyện bộ phân loại hành vi bằng %s trên %d mẫu.\",\n        clf.backend, len(X_tr),\n    )\n\n    y_pred = clf.predict(X_te)\n    acc = float(np.mean(y_pred == y_te))\n    mf1, per_class = macro_f1_score(y_te, y_pred, clf.classes_)\n    cm = confusion_matrix(y_te, y_pred, clf.classes_)\n\n    report = BehaviorReport(\n        backend=clf.backend,\n        classes=clf.classes_,\n        accuracy=acc,\n        macro_f1=mf1,\n        per_class_f1=per_class,\n        confusion_matrix=cm,\n        feature_importance=clf.feature_importance(feature_names),\n        n_train=len(X_tr),\n        n_test=len(X_te),\n    )\n    LOG.info(\"Độ chính xác %.4f | F1 vĩ mô %.4f\", acc, mf1)\n    return clf, report\n"for _rel, _content in _FILES.items():    _p = SRC_DIR / _rel    _p.parent.mkdir(parents=True, exist_ok=True)    _p.write_text(_content, encoding='utf-8')print(f'Đã ghi {len(_FILES)} tệp mã nguồn.')for _rel in _FILES:    print('   ', _rel)

### Chỉ tiêu đánh giá và phân tích đóng góp thành phần

In [ ]:
# Ghi mã nguồn ra đĩa. Nội dung giữ nguyên như trong kho mã nguồn._FILES = {}_FILES['sonarnet/evaluation/__init__.py'] = "\"\"\"Chỉ tiêu đánh giá và phân tích đóng góp thành phần.\"\"\"\n\nfrom .ablation import AblationRow, format_ablation_table, run_ablation  # noqa: F401\nfrom .detection_metrics import DetectionMetrics, box_iou, evaluate_multi_iou  # noqa: F401\nfrom .fusion_metrics import FusionMetrics, evaluate_fusion  # noqa: F401\n"_FILES['sonarnet/evaluation/detection_metrics.py'] = "\"\"\"Chỉ tiêu đánh giá tầng phát hiện phương tiện.\n\nToàn bộ phép tính được cài đặt trực tiếp, không phụ thuộc ``pycocotools``, nhằm\nbảo đảm chạy được trong mọi môi trường. Quy ước tính toán tuân theo chuẩn\nPASCAL VOC 2010 trở về sau: đường cong chính xác – độ nhạy được nội suy đơn điệu\nrồi lấy tích phân toàn phần.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, List, Sequence, Tuple\n\nimport numpy as np\n\n\ndef box_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n    \"\"\"Ma trận IoU giữa hai tập khung bao, dạng (Na, Nb).\"\"\"\n    if a.size == 0 or b.size == 0:\n        return np.zeros((len(a), len(b)), dtype=np.float32)\n\n    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)\n    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)\n\n    lt = np.maximum(a[:, None, :2], b[None, :, :2])\n    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])\n    wh = np.clip(rb - lt, 0, None)\n    inter = wh[..., 0] * wh[..., 1]\n\n    union = area_a[:, None] + area_b[None, :] - inter\n    return (inter / np.clip(union, 1e-9, None)).astype(np.float32)\n\n\n@dataclass\nclass DetectionMetrics:\n    \"\"\"Tập chỉ tiêu của tầng phát hiện.\"\"\"\n\n    ap50: float\n    ap50_95: float\n    precision: float\n    recall: float\n    f1: float\n    n_gt: int\n    n_pred: int\n    n_tp: int\n    n_fp: int\n    n_fn: int\n    pr_curve: Tuple[np.ndarray, np.ndarray]  # (recall, precision)\n\n    def to_dict(self) -> Dict:\n        return {\n            \"mAP@0.5\": round(float(self.ap50), 4),\n            \"mAP@0.5:0.95\": round(float(self.ap50_95), 4),\n            \"precision\": round(float(self.precision), 4),\n            \"recall\": round(float(self.recall), 4),\n            \"f1\": round(float(self.f1), 4),\n            \"n_ground_truth\": self.n_gt,\n            \"n_predictions\": self.n_pred,\n            \"true_positive\": self.n_tp,\n            \"false_positive\": self.n_fp,\n            \"false_negative\": self.n_fn,\n        }\n\n\ndef _average_precision(recall: np.ndarray, precision: np.ndarray) -> float:\n    \"\"\"Diện tích dưới đường cong chính xác – độ nhạy sau khi nội suy đơn điệu.\"\"\"\n    if recall.size == 0:\n        return 0.0\n    mrec = np.concatenate(([0.0], recall, [1.0]))\n    mpre = np.concatenate(([1.0], precision, [0.0]))\n    # Làm cho đường chính xác đơn điệu không tăng\n    for i in range(mpre.size - 2, -1, -1):\n        mpre[i] = max(mpre[i], mpre[i + 1])\n    idx = np.where(mrec[1:] != mrec[:-1])[0]\n    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))\n\n\ndef evaluate_detections(\n    predictions: Dict[str, Tuple[np.ndarray, np.ndarray]],\n    ground_truth: Dict[str, np.ndarray],\n    iou_threshold: float = 0.5,\n    score_threshold: float = 0.25,\n) -> DetectionMetrics:\n    \"\"\"Đánh giá tầng phát hiện trên toàn bộ tập ảnh.\n\n    ``predictions`` ánh xạ định danh ảnh sang cặp (khung bao, điểm tin cậy).\n    ``ground_truth`` ánh xạ định danh ảnh sang khung bao đối chứng.\n    \"\"\"\n    records: List[Tuple[float, int]] = []  # (điểm tin cậy, là dương tính thật)\n    n_gt_total = 0\n\n    for image_id, gt_boxes in ground_truth.items():\n        gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)\n        n_gt_total += len(gt_boxes)\n\n        pred_boxes, pred_scores = predictions.get(\n            image_id, (np.zeros((0, 4), np.float32), np.zeros((0,), np.float32))\n        )\n        pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)\n        pred_scores = np.asarray(pred_scores, dtype=np.float32).reshape(-1)\n\n        if pred_boxes.size == 0:\n            continue\n\n        order = np.argsort(-pred_scores)\n        pred_boxes, pred_scores = pred_boxes[order], pred_scores[order]\n\n        ious = box_iou(pred_boxes, gt_boxes)\n        taken = np.zeros(len(gt_boxes), dtype=bool)\n\n        for i in range(len(pred_boxes)):\n            if gt_boxes.size == 0:\n                records.append((float(pred_scores[i]), 0))\n                continue\n            j = int(np.argmax(np.where(taken, -1.0, ious[i])))\n            if ious[i, j] >= iou_threshold and not taken[j]:\n                taken[j] = True\n                records.append((float(pred_scores[i]), 1))\n            else:\n                records.append((float(pred_scores[i]), 0))\n\n    if not records or n_gt_total == 0:\n        return DetectionMetrics(\n            0.0, 0.0, 0.0, 0.0, 0.0, n_gt_total, 0, 0, 0, n_gt_total,\n            (np.array([]), np.array([])),\n        )\n\n    records.sort(key=lambda r: -r[0])\n    tps = np.array([r[1] for r in records], dtype=np.float32)\n    scores = np.array([r[0] for r in records], dtype=np.float32)\n\n    cum_tp = np.cumsum(tps)\n    cum_fp = np.cumsum(1.0 - tps)\n    recall = cum_tp / n_gt_total\n    precision = cum_tp / np.clip(cum_tp + cum_fp, 1e-9, None)\n\n    ap50 = _average_precision(recall, precision)\n\n    # Chỉ tiêu tại ngưỡng tin cậy làm việc\n    keep = scores >= score_threshold\n    n_tp = int(np.sum(tps[keep]))\n    n_fp = int(np.sum(1.0 - tps[keep]))\n    n_fn = int(n_gt_total - n_tp)\n    p = n_tp / max(n_tp + n_fp, 1)\n    r = n_tp / max(n_gt_total, 1)\n    f1 = 2 * p * r / max(p + r, 1e-9)\n\n    return DetectionMetrics(\n        ap50=ap50,\n        ap50_95=ap50,  # được ghi đè khi gọi evaluate_multi_iou\n        precision=p, recall=r, f1=f1,\n        n_gt=n_gt_total, n_pred=int(len(records)),\n        n_tp=n_tp, n_fp=n_fp, n_fn=n_fn,\n        pr_curve=(recall, precision),\n    )\n\n\ndef evaluate_multi_iou(\n    predictions: Dict[str, Tuple[np.ndarray, np.ndarray]],\n    ground_truth: Dict[str, np.ndarray],\n    score_threshold: float = 0.25,\n) -> DetectionMetrics:\n    \"\"\"Tính đầy đủ cả mAP@0.5 và mAP trung bình trên dải ngưỡng 0,5 đến 0,95.\"\"\"\n    base = evaluate_detections(\n        predictions, ground_truth, iou_threshold=0.5, score_threshold=score_threshold\n    )\n    aps = []\n    for thr in np.arange(0.5, 1.0, 0.05):\n        m = evaluate_detections(\n            predictions, ground_truth, iou_threshold=float(thr),\n            score_threshold=score_threshold,\n        )\n        aps.append(m.ap50)\n    base.ap50_95 = float(np.mean(aps)) if aps else 0.0\n    return base\n\n\ndef match_predictions_to_truth(\n    pred_boxes: np.ndarray,\n    gt_boxes: np.ndarray,\n    iou_threshold: float = 0.5,\n) -> Dict[int, int]:\n    \"\"\"Ghép từng phát hiện với phương tiện đối chứng tương ứng.\n\n    Trả về ánh xạ chỉ số phát hiện sang chỉ số đối chứng. Kết quả này là cầu nối\n    để đánh giá tầng hợp nhất trên chính các phát hiện của mô hình.\n    \"\"\"\n    if len(pred_boxes) == 0 or len(gt_boxes) == 0:\n        return {}\n    ious = box_iou(\n        np.asarray(pred_boxes, np.float32), np.asarray(gt_boxes, np.float32)\n    )\n    mapping: Dict[int, int] = {}\n    taken = set()\n    # Duyệt theo thứ tự IoU giảm dần để ưu tiên các cặp khớp tốt nhất\n    pairs = [\n        (float(ious[i, j]), i, j)\n        for i in range(ious.shape[0])\n        for j in range(ious.shape[1])\n        if ious[i, j] >= iou_threshold\n    ]\n    pairs.sort(reverse=True)\n    for _, i, j in pairs:\n        if i in mapping or j in taken:\n            continue\n        mapping[i] = j\n        taken.add(j)\n    return mapping\n"_FILES['sonarnet/evaluation/fusion_metrics.py'] = "\"\"\"Chỉ tiêu đánh giá tầng hợp nhất ảnh radar và tín hiệu AIS.\n\nBa nhóm chỉ tiêu được tính:\n\n1. **Chất lượng ghép cặp** — tỉ lệ phương tiện có AIS được ghép đúng với chính\n   nó, và sai số vị trí của phép ghép.\n2. **Chất lượng phát hiện phương tiện ngắt định danh** — độ chính xác và độ nhạy\n   đối với lớp ``DARK``. Đây là chỉ tiêu quan trọng nhất của toàn hệ thống.\n3. **Chất lượng phân loại ba trạng thái** — ma trận nhầm lẫn đầy đủ.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom ..data.simulator import AIS_MISMATCH, AIS_OK, DARK, IDENTITY_STATES\n\n\n@dataclass\nclass FusionMetrics:\n    \"\"\"Tập chỉ tiêu của tầng hợp nhất.\"\"\"\n\n    # Ghép cặp\n    match_accuracy: float\n    mean_match_error_m: float\n    median_match_error_m: float\n    p90_match_error_m: float\n    # Phát hiện phương tiện ngắt định danh\n    dark_precision: float\n    dark_recall: float\n    dark_f1: float\n    # Phân loại ba trạng thái\n    state_accuracy: float\n    state_macro_f1: float\n    confusion_matrix: np.ndarray\n    per_state_f1: Dict[str, float] = field(default_factory=dict)\n    n_evaluated: int = 0\n\n    def to_dict(self) -> Dict:\n        return {\n            \"match_accuracy\": round(float(self.match_accuracy), 4),\n            \"mean_match_error_m\": round(float(self.mean_match_error_m), 2),\n            \"median_match_error_m\": round(float(self.median_match_error_m), 2),\n            \"p90_match_error_m\": round(float(self.p90_match_error_m), 2),\n            \"dark_precision\": round(float(self.dark_precision), 4),\n            \"dark_recall\": round(float(self.dark_recall), 4),\n            \"dark_f1\": round(float(self.dark_f1), 4),\n            \"state_accuracy\": round(float(self.state_accuracy), 4),\n            \"state_macro_f1\": round(float(self.state_macro_f1), 4),\n            \"confusion_matrix\": self.confusion_matrix.tolist(),\n            \"confusion_labels\": list(IDENTITY_STATES),\n            \"per_state_f1\": {k: round(float(v), 4) for k, v in self.per_state_f1.items()},\n            \"n_evaluated\": self.n_evaluated,\n        }\n\n\ndef _prf(tp: int, fp: int, fn: int) -> Tuple[float, float, float]:\n    p = tp / (tp + fp) if (tp + fp) else 0.0\n    r = tp / (tp + fn) if (tp + fn) else 0.0\n    f = 2 * p * r / (p + r) if (p + r) else 0.0\n    return p, r, f\n\n\ndef evaluate_fusion(\n    true_states: Sequence[str],\n    pred_states: Sequence[str],\n    true_mmsi: Sequence[Optional[int]],\n    pred_mmsi: Sequence[Optional[int]],\n    match_errors_m: Sequence[float],\n) -> FusionMetrics:\n    \"\"\"Tính toàn bộ chỉ tiêu của tầng hợp nhất.\n\n    Bốn dãy đầu vào có cùng độ dài và cùng thứ tự, mỗi phần tử ứng với một phát\n    hiện đã được ghép với một phương tiện đối chứng.\n    \"\"\"\n    true_states = [str(s) for s in true_states]\n    pred_states = [str(s) for s in pred_states]\n    n = len(true_states)\n\n    if n == 0:\n        return FusionMetrics(\n            0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,\n            np.zeros((3, 3), dtype=int), {}, 0,\n        )\n\n    # -- Ghép cặp ----------------------------------------------------------\n    # Chỉ xét các phương tiện thực sự có phát AIS\n    correct, total_with_ais = 0, 0\n    for ts, tm, pm in zip(true_states, true_mmsi, pred_mmsi):\n        if ts == DARK:\n            continue\n        total_with_ais += 1\n        if tm is not None and pm is not None and int(tm) == int(pm):\n            correct += 1\n    match_acc = correct / total_with_ais if total_with_ais else 0.0\n\n    errs = np.array(\n        [e for e in match_errors_m if e is not None and np.isfinite(e)], dtype=float\n    )\n    mean_e = float(np.mean(errs)) if errs.size else float(\"nan\")\n    med_e = float(np.median(errs)) if errs.size else float(\"nan\")\n    p90_e = float(np.percentile(errs, 90)) if errs.size else float(\"nan\")\n\n    # -- Phát hiện phương tiện ngắt định danh -------------------------------\n    tp = sum(1 for t, p in zip(true_states, pred_states) if t == DARK and p == DARK)\n    fp = sum(1 for t, p in zip(true_states, pred_states) if t != DARK and p == DARK)\n    fn = sum(1 for t, p in zip(true_states, pred_states) if t == DARK and p != DARK)\n    d_p, d_r, d_f = _prf(tp, fp, fn)\n\n    # -- Phân loại ba trạng thái -------------------------------------------\n    idx = {s: i for i, s in enumerate(IDENTITY_STATES)}\n    cm = np.zeros((len(IDENTITY_STATES), len(IDENTITY_STATES)), dtype=int)\n    for t, p in zip(true_states, pred_states):\n        if t in idx and p in idx:\n            cm[idx[t], idx[p]] += 1\n\n    acc = float(np.trace(cm) / max(cm.sum(), 1))\n    per_state: Dict[str, float] = {}\n    for s in IDENTITY_STATES:\n        i = idx[s]\n        s_tp = int(cm[i, i])\n        s_fp = int(cm[:, i].sum() - s_tp)\n        s_fn = int(cm[i, :].sum() - s_tp)\n        per_state[s] = _prf(s_tp, s_fp, s_fn)[2]\n    macro_f1 = float(np.mean(list(per_state.values())))\n\n    return FusionMetrics(\n        match_accuracy=match_acc,\n        mean_match_error_m=mean_e,\n        median_match_error_m=med_e,\n        p90_match_error_m=p90_e,\n        dark_precision=d_p, dark_recall=d_r, dark_f1=d_f,\n        state_accuracy=acc, state_macro_f1=macro_f1,\n        confusion_matrix=cm, per_state_f1=per_state,\n        n_evaluated=n,\n    )\n\n\ndef compare_interpolation_methods(\n    scenes_meta: Sequence[Dict],\n    ais_by_scene: Dict[str, List[Dict]],\n    cfg,\n) -> Dict[str, Dict[str, float]]:\n    \"\"\"So sánh bộ lọc Kalman với nội suy tuyến tính.\n\n    Chỉ tiêu là sai số giữa vị trí nội suy và vị trí thật của phương tiện tại\n    đúng thời điểm chụp ảnh. Phép so sánh này lượng hoá đóng góp riêng của bước\n    nội suy trong toàn bộ tầng hợp nhất.\n    \"\"\"\n    from ..data.geo import haversine_m\n    from ..fusion.kalman import ConstantVelocityKalman, linear_interpolate_to_time\n\n    kf = ConstantVelocityKalman(\n        process_noise=cfg.fusion.kalman_process_noise,\n        measurement_noise=cfg.fusion.kalman_measurement_noise,\n    )\n    errors = {\"kalman\": [], \"linear\": []}\n\n    for meta in scenes_meta:\n        sid = meta[\"scene_id\"]\n        t_cap = float(meta[\"capture_time_s\"])\n        recs = ais_by_scene.get(sid, [])\n        by_mmsi: Dict[int, List[Dict]] = {}\n        for r in recs:\n            by_mmsi.setdefault(int(r[\"mmsi\"]), []).append(r)\n\n        for v in meta[\"vessels\"]:\n            if v[\"identity_state\"] == DARK:\n                continue\n            rs = by_mmsi.get(int(v[\"mmsi\"]))\n            if not rs:\n                continue\n            for name, fn in (\n                (\"kalman\", lambda rr: kf.smooth_to_time(rr, t_cap)),\n                (\"linear\", lambda rr: linear_interpolate_to_time(rr, t_cap)),\n            ):\n                est = fn(rs)\n                if est is None:\n                    continue\n                err = float(\n                    haversine_m(v[\"lat\"], v[\"lon\"], est.lat, est.lon)\n                )\n                errors[name].append(err)\n\n    out: Dict[str, Dict[str, float]] = {}\n    for name, vals in errors.items():\n        arr = np.array(vals, dtype=float)\n        out[name] = {\n            \"mean_error_m\": float(np.mean(arr)) if arr.size else float(\"nan\"),\n            \"median_error_m\": float(np.median(arr)) if arr.size else float(\"nan\"),\n            \"p90_error_m\": float(np.percentile(arr, 90)) if arr.size else float(\"nan\"),\n            \"n\": int(arr.size),\n        }\n    return out\n"_FILES['sonarnet/evaluation/ablation.py'] = "\"\"\"Phân tích đóng góp của từng thành phần trong hệ thống.\n\nBốn cấu hình được so sánh, đúng theo kế hoạch nêu trong đề xuất:\n\n======  ==========================================================\nCấu hình  Nội dung\n======  ==========================================================\nA        Chỉ dùng ảnh radar. Hệ thống phát hiện được phương tiện\n         nhưng không có căn cứ nào để xác định trạng thái định danh.\nB        Chỉ dùng dữ liệu AIS. Mọi phương tiện chủ động ngắt tín\n         hiệu đều vô hình đối với hệ thống.\nC        Hợp nhất hai nguồn, nội suy tuyến tính, chưa kiểm tra\n         kích thước khai báo.\nD        Hệ thống đầy đủ: hợp nhất có bộ lọc Kalman và có kiểm tra\n         kích thước khai báo.\n======  ==========================================================\n\nPhép so sánh này cho phép trả lời câu hỏi trọng tâm: mỗi thành phần đóng góp\nbao nhiêu vào năng lực phát hiện phương tiện ngắt định danh.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport copy\nfrom dataclasses import dataclass\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom ..data.simulator import AIS_MISMATCH, AIS_OK, DARK\nfrom ..utils import get_logger\nfrom .fusion_metrics import FusionMetrics, evaluate_fusion\n\nLOG = get_logger(\"evaluation.ablation\")\n\nCONFIG_LABELS = {\n    \"A\": \"Chỉ ảnh radar\",\n    \"B\": \"Chỉ dữ liệu AIS\",\n    \"C\": \"Hợp nhất, nội suy tuyến tính\",\n    \"D\": \"Hệ thống đầy đủ\",\n}\n\n\n@dataclass\nclass AblationRow:\n    \"\"\"Một dòng kết quả của bảng phân tích đóng góp.\"\"\"\n\n    config: str\n    label: str\n    vessel_recall: float\n    dark_precision: float\n    dark_recall: float\n    dark_f1: float\n    state_accuracy: float\n    state_macro_f1: float\n    mean_match_error_m: float\n\n    def to_dict(self) -> Dict:\n        return {\n            \"Cấu hình\": self.config,\n            \"Nội dung\": self.label,\n            \"Độ nhạy phát hiện\": round(self.vessel_recall, 4),\n            \"Chính xác lớp DARK\": round(self.dark_precision, 4),\n            \"Độ nhạy lớp DARK\": round(self.dark_recall, 4),\n            \"F1 lớp DARK\": round(self.dark_f1, 4),\n            \"Chính xác 3 trạng thái\": round(self.state_accuracy, 4),\n            \"F1 vĩ mô 3 trạng thái\": round(self.state_macro_f1, 4),\n            \"Sai số ghép cặp (m)\": (\n                round(self.mean_match_error_m, 1)\n                if np.isfinite(self.mean_match_error_m) else None\n            ),\n        }\n\n\ndef _vessel_recall(records: Sequence[Dict]) -> float:\n    \"\"\"Tỉ lệ phương tiện đối chứng được hệ thống nhìn thấy.\"\"\"\n    if not records:\n        return 0.0\n    seen = sum(1 for r in records if r.get(\"detected\", False))\n    return seen / len(records)\n\n\ndef run_ablation(\n    fusion_runner,\n    cfg,\n    scenes_meta: Sequence[Dict],\n    ais_by_scene: Dict[str, List[Dict]],\n    detections_by_scene: Dict[str, Tuple[np.ndarray, np.ndarray]],\n) -> Tuple[List[AblationRow], Dict[str, FusionMetrics]]:\n    \"\"\"Chạy bốn cấu hình và tổng hợp kết quả.\n\n    ``fusion_runner`` là hàm nhận ``(cfg, scenes_meta, ais, detections, options)``\n    và trả về bản ghi chi tiết theo từng phát hiện đã ghép với đối chứng.\n    \"\"\"\n    rows: List[AblationRow] = []\n    metrics_map: Dict[str, FusionMetrics] = {}\n\n    # ---- Cấu hình D: hệ thống đầy đủ -------------------------------------\n    recs_d = fusion_runner(\n        cfg, scenes_meta, ais_by_scene, detections_by_scene,\n        use_kalman=True, use_size_check=True,\n    )\n    m_d = _metrics_from_records(recs_d)\n    metrics_map[\"D\"] = m_d\n    rows.append(\n        AblationRow(\n            \"D\", CONFIG_LABELS[\"D\"], _vessel_recall(recs_d),\n            m_d.dark_precision, m_d.dark_recall, m_d.dark_f1,\n            m_d.state_accuracy, m_d.state_macro_f1, m_d.mean_match_error_m,\n        )\n    )\n\n    # ---- Cấu hình C: nội suy tuyến tính, không kiểm tra kích thước --------\n    recs_c = fusion_runner(\n        cfg, scenes_meta, ais_by_scene, detections_by_scene,\n        use_kalman=False, use_size_check=False,\n    )\n    m_c = _metrics_from_records(recs_c)\n    metrics_map[\"C\"] = m_c\n    rows.append(\n        AblationRow(\n            \"C\", CONFIG_LABELS[\"C\"], _vessel_recall(recs_c),\n            m_c.dark_precision, m_c.dark_recall, m_c.dark_f1,\n            m_c.state_accuracy, m_c.state_macro_f1, m_c.mean_match_error_m,\n        )\n    )\n\n    # ---- Cấu hình A: chỉ ảnh radar ---------------------------------------\n    # Không có AIS nên hệ thống không thể xác nhận định danh của bất kỳ\n    # phương tiện nào; mọi phát hiện đều phải bị coi là chưa xác định.\n    recs_a = [dict(r) for r in recs_d]\n    for r in recs_a:\n        r[\"pred_state\"] = DARK\n        r[\"pred_mmsi\"] = None\n        r[\"match_error_m\"] = float(\"nan\")\n    m_a = _metrics_from_records(recs_a)\n    metrics_map[\"A\"] = m_a\n    rows.append(\n        AblationRow(\n            \"A\", CONFIG_LABELS[\"A\"], _vessel_recall(recs_a),\n            m_a.dark_precision, m_a.dark_recall, m_a.dark_f1,\n            m_a.state_accuracy, m_a.state_macro_f1, float(\"nan\"),\n        )\n    )\n\n    # ---- Cấu hình B: chỉ dữ liệu AIS -------------------------------------\n    # Phương tiện không phát tín hiệu hoàn toàn không xuất hiện trong hệ thống.\n    recs_b = []\n    for r in recs_d:\n        rb = dict(r)\n        if rb[\"true_state\"] == DARK:\n            rb[\"detected\"] = False\n            rb[\"pred_state\"] = None      # hệ thống không biết phương tiện này tồn tại\n        else:\n            rb[\"detected\"] = True\n            rb[\"pred_state\"] = rb[\"true_state\"]\n            rb[\"pred_mmsi\"] = rb[\"true_mmsi\"]\n            rb[\"match_error_m\"] = 0.0\n        recs_b.append(rb)\n    m_b = _metrics_from_records(recs_b)\n    metrics_map[\"B\"] = m_b\n    rows.append(\n        AblationRow(\n            \"B\", CONFIG_LABELS[\"B\"], _vessel_recall(recs_b),\n            m_b.dark_precision, m_b.dark_recall, m_b.dark_f1,\n            m_b.state_accuracy, m_b.state_macro_f1, m_b.mean_match_error_m,\n        )\n    )\n\n    rows.sort(key=lambda r: r.config)\n    return rows, metrics_map\n\n\ndef _metrics_from_records(records: Sequence[Dict]) -> FusionMetrics:\n    \"\"\"Quy đổi danh sách bản ghi chi tiết thành tập chỉ tiêu.\n\n    Phương tiện mà hệ thống hoàn toàn không nhìn thấy — do mô hình phát hiện bỏ\n    sót, hoặc do cấu hình không có nguồn dữ liệu tương ứng — mang ``pred_state``\n    bằng ``None``. Những trường hợp này không tham gia ma trận nhầm lẫn nhưng\n    vẫn được tính là bỏ sót đối với lớp thật của chúng, để chỉ tiêu độ nhạy\n    phản ánh đúng năng lực thực tế của toàn hệ thống.\n    \"\"\"\n    visible = [r for r in records if r.get(\"pred_state\") is not None]\n    invisible = [r for r in records if r.get(\"pred_state\") is None]\n\n    true_states = [r[\"true_state\"] for r in visible]\n    pred_states = [r[\"pred_state\"] for r in visible]\n    true_mmsi = [r.get(\"true_mmsi\") for r in visible]\n    pred_mmsi = [r.get(\"pred_mmsi\") for r in visible]\n    errs = [r.get(\"match_error_m\", float(\"nan\")) for r in visible]\n\n    # Bổ sung các phương tiện ngắt định danh bị bỏ sót hoàn toàn\n    for r in invisible:\n        if r[\"true_state\"] != DARK:\n            continue\n        true_states.append(DARK)\n        pred_states.append(AIS_OK)  # nhãn giữ chỗ, chắc chắn khác DARK\n        true_mmsi.append(None)\n        pred_mmsi.append(None)\n        errs.append(float(\"nan\"))\n\n    return evaluate_fusion(true_states, pred_states, true_mmsi, pred_mmsi, errs)\n\n\ndef format_ablation_table(rows: Sequence[AblationRow]) -> str:\n    \"\"\"Kết xuất bảng phân tích đóng góp dưới dạng văn bản căn cột.\"\"\"\n    from ..utils import format_table\n\n    return format_table([r.to_dict() for r in rows])\n"for _rel, _content in _FILES.items():    _p = SRC_DIR / _rel    _p.parent.mkdir(parents=True, exist_ok=True)    _p.write_text(_content, encoding='utf-8')print(f'Đã ghi {len(_FILES)} tệp mã nguồn.')for _rel in _FILES:    print('   ', _rel)

### Kết xuất hình minh hoạ và bản đồ giám sát

In [ ]:
# Ghi mã nguồn ra đĩa. Nội dung giữ nguyên như trong kho mã nguồn._FILES = {}_FILES['sonarnet/viz/__init__.py'] = "\"\"\"Kết xuất hình minh hoạ và bảng điều khiển bản đồ.\"\"\"\n\nfrom .dashboard import build_dashboard  # noqa: F401\n"_FILES['sonarnet/viz/figures.py'] = "\"\"\"Kết xuất hình minh hoạ phục vụ báo cáo kỹ thuật và video trình diễn.\n\nBảng màu được chọn thống nhất với bộ tài liệu đề xuất: nền trung tính, các mức\nnhấn bằng tông pastel, chữ màu mực đậm. Phông chữ mặc định của Matplotlib hỗ trợ\nđầy đủ tiếng Việt nên không cần cài đặt thêm.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom ..utils import get_logger\n\nLOG = get_logger(\"viz.figures\")\n\n# Bảng màu thống nhất toàn dự án\nNAVY = \"#1C3557\"\nSLATE = \"#3F576F\"\nBLUE = \"#5B8DB8\"\nBLUE_SOFT = \"#D6E5F3\"\nMINT = \"#7FB29A\"\nMINT_SOFT = \"#E4F0EA\"\nCREAM = \"#DFC58A\"\nBLUSH = \"#D98C7A\"\nGREY = \"#8A97A3\"\n\nSTATE_COLORS = {\n    \"AIS_OK\": MINT,\n    \"AIS_MISMATCH\": CREAM,\n    \"DARK\": BLUSH,\n}\nSTATE_LABELS = {\n    \"AIS_OK\": \"Phát tín hiệu trung thực\",\n    \"AIS_MISMATCH\": \"Kích thước khai báo sai lệch\",\n    \"DARK\": \"Không phát tín hiệu\",\n}\n\n\ndef _setup():\n    import matplotlib\n\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n\n    plt.rcParams.update(\n        {\n            \"figure.dpi\": 130,\n            \"savefig.dpi\": 160,\n            \"savefig.bbox\": \"tight\",\n            \"font.size\": 10,\n            \"axes.titlesize\": 12,\n            \"axes.titleweight\": \"bold\",\n            \"axes.labelcolor\": SLATE,\n            \"axes.edgecolor\": GREY,\n            \"axes.titlecolor\": NAVY,\n            \"xtick.color\": SLATE,\n            \"ytick.color\": SLATE,\n            \"axes.grid\": True,\n            \"grid.color\": \"#E3E9EF\",\n            \"grid.linewidth\": 0.7,\n            \"axes.axisbelow\": True,\n        }\n    )\n    return plt\n\n\n# ---------------------------------------------------------------------------\ndef plot_scene_with_boxes(\n    image: np.ndarray,\n    gt_boxes: np.ndarray,\n    gt_states: Sequence[str],\n    pred_boxes: Optional[np.ndarray],\n    out_path: Path,\n    title: str = \"Cảnh ảnh radar mô phỏng\",\n) -> Path:\n    \"\"\"Vẽ một cảnh ảnh kèm khung bao đối chứng và khung bao dự đoán.\"\"\"\n    plt = _setup()\n    from matplotlib.patches import Patch, Rectangle\n\n    fig, ax = plt.subplots(figsize=(7.2, 7.2))\n    ax.imshow(image, cmap=\"gray\", interpolation=\"nearest\")\n    ax.grid(False)\n\n    for box, state in zip(gt_boxes, gt_states):\n        x1, y1, x2, y2 = box\n        ax.add_patch(\n            Rectangle(\n                (x1, y1), x2 - x1, y2 - y1,\n                fill=False, edgecolor=STATE_COLORS.get(state, MINT),\n                linewidth=1.7,\n            )\n        )\n\n    if pred_boxes is not None and len(pred_boxes):\n        for x1, y1, x2, y2 in pred_boxes:\n            ax.add_patch(\n                Rectangle(\n                    (x1, y1), x2 - x1, y2 - y1,\n                    fill=False, edgecolor=\"#FFFFFF\",\n                    linewidth=0.9, linestyle=\"--\",\n                )\n            )\n\n    handles = [\n        Patch(facecolor=\"none\", edgecolor=STATE_COLORS[s], label=STATE_LABELS[s])\n        for s in STATE_COLORS\n    ]\n    if pred_boxes is not None and len(pred_boxes):\n        handles.append(\n            Patch(facecolor=\"none\", edgecolor=\"#FFFFFF\", linestyle=\"--\",\n                  label=\"Phát hiện của mô hình\")\n        )\n    ax.legend(handles=handles, loc=\"upper right\", framealpha=0.85, fontsize=8)\n    ax.set_title(title)\n    ax.set_xlabel(\"Điểm ảnh theo trục ngang\")\n    ax.set_ylabel(\"Điểm ảnh theo trục dọc\")\n\n    out_path = Path(out_path)\n    fig.savefig(out_path)\n    plt.close(fig)\n    return out_path\n\n\n# ---------------------------------------------------------------------------\ndef plot_pr_curve(recall: np.ndarray, precision: np.ndarray, ap: float, out_path: Path) -> Path:\n    \"\"\"Đường cong chính xác – độ nhạy của tầng phát hiện.\"\"\"\n    plt = _setup()\n    fig, ax = plt.subplots(figsize=(5.6, 4.4))\n    ax.plot(recall, precision, color=NAVY, linewidth=2.0)\n    ax.fill_between(recall, precision, color=BLUE_SOFT, alpha=0.75)\n    ax.set_xlim(0, 1)\n    ax.set_ylim(0, 1.02)\n    ax.set_xlabel(\"Độ nhạy (Recall)\")\n    ax.set_ylabel(\"Độ chính xác (Precision)\")\n    ax.set_title(f\"Đường cong chính xác – độ nhạy   (mAP@0.5 = {ap:.3f})\")\n    fig.savefig(out_path)\n    plt.close(fig)\n    return Path(out_path)\n\n\n# ---------------------------------------------------------------------------\ndef plot_confusion(\n    cm: np.ndarray,\n    labels: Sequence[str],\n    out_path: Path,\n    title: str = \"Ma trận nhầm lẫn\",\n    display_labels: Optional[Sequence[str]] = None,\n) -> Path:\n    \"\"\"Ma trận nhầm lẫn dạng bản đồ nhiệt, có ghi số tuyệt đối và tỉ lệ.\"\"\"\n    plt = _setup()\n    from matplotlib.colors import LinearSegmentedColormap\n\n    cmap = LinearSegmentedColormap.from_list(\"pastel_blue\", [\"#FFFFFF\", BLUE_SOFT, BLUE])\n    disp = list(display_labels or labels)\n\n    cm = np.asarray(cm, dtype=float)\n    row_sum = cm.sum(axis=1, keepdims=True)\n    norm = np.divide(cm, np.clip(row_sum, 1e-9, None))\n\n    n = len(disp)\n    fig, ax = plt.subplots(figsize=(1.55 * n + 2.4, 1.3 * n + 2.0))\n    im = ax.imshow(norm, cmap=cmap, vmin=0, vmax=1)\n    ax.grid(False)\n\n    ax.set_xticks(range(n))\n    ax.set_yticks(range(n))\n    ax.set_xticklabels(disp, rotation=22, ha=\"right\")\n    ax.set_yticklabels(disp)\n    ax.set_xlabel(\"Dự đoán\")\n    ax.set_ylabel(\"Thực tế\")\n    ax.set_title(title)\n\n    for i in range(n):\n        for j in range(n):\n            val = int(cm[i, j])\n            pct = norm[i, j]\n            ax.text(\n                j, i, f\"{val}\\n{pct:.0%}\",\n                ha=\"center\", va=\"center\",\n                color=NAVY if pct < 0.55 else \"#FFFFFF\",\n                fontsize=9, fontweight=\"bold\" if i == j else \"normal\",\n            )\n\n    fig.colorbar(im, ax=ax, fraction=0.042, pad=0.04, label=\"Tỉ lệ theo hàng\")\n    fig.savefig(out_path)\n    plt.close(fig)\n    return Path(out_path)\n\n\n# ---------------------------------------------------------------------------\ndef plot_ablation(rows: Sequence[Dict], out_path: Path) -> Path:\n    \"\"\"Biểu đồ cột so sánh bốn cấu hình của phép phân tích đóng góp.\"\"\"\n    plt = _setup()\n\n    configs = [r[\"Cấu hình\"] for r in rows]\n    labels = [f\"{r['Cấu hình']}. {r['Nội dung']}\" for r in rows]\n    metrics = [\n        (\"Độ nhạy phát hiện\", BLUE),\n        (\"F1 lớp DARK\", BLUSH),\n        (\"F1 vĩ mô 3 trạng thái\", MINT),\n    ]\n\n    x = np.arange(len(configs))\n    width = 0.26\n    fig, ax = plt.subplots(figsize=(9.2, 4.8))\n\n    for k, (name, color) in enumerate(metrics):\n        vals = [float(r.get(name) or 0.0) for r in rows]\n        bars = ax.bar(x + (k - 1) * width, vals, width, label=name, color=color)\n        for b, v in zip(bars, vals):\n            ax.text(\n                b.get_x() + b.get_width() / 2, v + 0.018, f\"{v:.2f}\",\n                ha=\"center\", va=\"bottom\", fontsize=8, color=SLATE,\n            )\n\n    ax.set_xticks(x)\n    ax.set_xticklabels(labels, fontsize=9)\n    ax.set_ylim(0, 1.12)\n    ax.set_ylabel(\"Giá trị chỉ tiêu\")\n    ax.set_title(\"Đóng góp của từng thành phần trong hệ thống\")\n    ax.legend(loc=\"upper left\", fontsize=9, framealpha=0.9)\n    fig.savefig(out_path)\n    plt.close(fig)\n    return Path(out_path)\n\n\n# ---------------------------------------------------------------------------\ndef plot_match_errors(records: Sequence[Dict], out_path: Path, threshold_m: float = 500.0) -> Path:\n    \"\"\"Phân bố sai số ghép cặp giữa phát hiện và vị trí AIS nội suy.\"\"\"\n    plt = _setup()\n\n    errs = np.array(\n        [\n            r[\"match_error_m\"]\n            for r in records\n            if r.get(\"match_error_m\") is not None and np.isfinite(r.get(\"match_error_m\", np.nan))\n        ],\n        dtype=float,\n    )\n    fig, ax = plt.subplots(figsize=(6.4, 4.2))\n    if errs.size:\n        ax.hist(errs, bins=36, color=BLUE_SOFT, edgecolor=BLUE, linewidth=0.9)\n        ax.axvline(\n            float(np.median(errs)), color=NAVY, linestyle=\"--\", linewidth=1.6,\n            label=f\"Trung vị {np.median(errs):.0f} m\",\n        )\n        ax.axvline(\n            threshold_m, color=BLUSH, linestyle=\":\", linewidth=1.6,\n            label=f\"Ngưỡng chấp nhận {threshold_m:.0f} m\",\n        )\n        ax.legend(fontsize=9)\n    ax.set_xlabel(\"Sai số ghép cặp (mét)\")\n    ax.set_ylabel(\"Số trường hợp\")\n    ax.set_title(\"Phân bố sai số ghép cặp ảnh radar và AIS\")\n    fig.savefig(out_path)\n    plt.close(fig)\n    return Path(out_path)\n\n\n# ---------------------------------------------------------------------------\ndef plot_interpolation_comparison(interp: Dict[str, Dict[str, float]], out_path: Path) -> Path:\n    \"\"\"So sánh sai số nội suy giữa bộ lọc Kalman và nội suy tuyến tính.\"\"\"\n    plt = _setup()\n\n    names = {\"kalman\": \"Bộ lọc Kalman\", \"linear\": \"Nội suy tuyến tính\"}\n    keys = [k for k in (\"kalman\", \"linear\") if k in interp]\n    metrics = [\n        (\"mean_error_m\", \"Sai số trung bình\"),\n        (\"median_error_m\", \"Sai số trung vị\"),\n        (\"p90_error_m\", \"Bách phân vị 90\"),\n    ]\n\n    x = np.arange(len(metrics))\n    width = 0.34\n    fig, ax = plt.subplots(figsize=(6.8, 4.3))\n\n    for i, k in enumerate(keys):\n        vals = [float(interp[k].get(m[0], np.nan)) for m in metrics]\n        color = MINT if k == \"kalman\" else GREY\n        bars = ax.bar(x + (i - 0.5) * width, vals, width, label=names[k], color=color)\n        for b, v in zip(bars, vals):\n            if np.isfinite(v):\n                ax.text(\n                    b.get_x() + b.get_width() / 2, v, f\"{v:.0f}\",\n                    ha=\"center\", va=\"bottom\", fontsize=8, color=SLATE,\n                )\n\n    ax.set_xticks(x)\n    ax.set_xticklabels([m[1] for m in metrics])\n    ax.set_ylabel(\"Sai số vị trí (mét)\")\n    ax.set_title(\"Chất lượng nội suy quỹ đạo AIS về thời điểm chụp ảnh\")\n    ax.legend(fontsize=9)\n    fig.savefig(out_path)\n    plt.close(fig)\n    return Path(out_path)\n\n\n# ---------------------------------------------------------------------------\ndef plot_behaviour_tracks(tracks: Sequence, out_path: Path, per_class: int = 2) -> Path:\n    \"\"\"Minh hoạ quỹ đạo tiêu biểu của bốn nhóm hành vi.\"\"\"\n    plt = _setup()\n\n    by_class: Dict[str, List] = {}\n    for tr in tracks:\n        by_class.setdefault(tr.behaviour, []).append(tr)\n\n    labels = {\n        \"qua_canh\": \"Di chuyển quá cảnh\",\n        \"cau\": \"Câu\",\n        \"keo_luoi\": \"Kéo lưới\",\n        \"neo_dau\": \"Neo đậu, tụ tập\",\n    }\n    classes = [c for c in (\"qua_canh\", \"cau\", \"keo_luoi\", \"neo_dau\") if c in by_class]\n\n    fig, axes = plt.subplots(1, len(classes), figsize=(3.4 * len(classes), 3.6))\n    if len(classes) == 1:\n        axes = [axes]\n\n    for ax, cls in zip(axes, classes):\n        for tr in by_class[cls][:per_class]:\n            lon = np.asarray(tr.lons)\n            lat = np.asarray(tr.lats)\n            ax.plot(lon - lon.mean(), lat - lat.mean(), linewidth=1.3, color=NAVY, alpha=0.8)\n            ax.scatter(\n                [lon[0] - lon.mean()], [lat[0] - lat.mean()],\n                s=22, color=MINT, zorder=3,\n            )\n        ax.set_title(labels.get(cls, cls), fontsize=10)\n        ax.set_xlabel(\"Δ kinh độ\")\n        ax.set_ylabel(\"Δ vĩ độ\")\n        ax.ticklabel_format(style=\"sci\", scilimits=(-2, 2), axis=\"both\")\n\n    fig.suptitle(\"Dấu hiệu động học của bốn nhóm hành vi\", fontsize=12,\n                 color=NAVY, fontweight=\"bold\")\n    fig.tight_layout()\n    fig.savefig(out_path)\n    plt.close(fig)\n    return Path(out_path)\n\n\n# ---------------------------------------------------------------------------\ndef plot_feature_importance(importance: Optional[Dict[str, float]], out_path: Path) -> Optional[Path]:\n    \"\"\"Xếp hạng mức đóng góp của các đặc trưng động học.\"\"\"\n    if not importance:\n        return None\n    plt = _setup()\n\n    items = sorted(importance.items(), key=lambda kv: kv[1], reverse=True)[:14]\n    names = [k for k, _ in items][::-1]\n    vals = [v for _, v in items][::-1]\n\n    fig, ax = plt.subplots(figsize=(7.0, 5.0))\n    ax.barh(names, vals, color=BLUE_SOFT, edgecolor=BLUE, linewidth=0.9)\n    ax.set_xlabel(\"Mức đóng góp tương đối\")\n    ax.set_title(\"Đặc trưng động học có ảnh hưởng lớn nhất\")\n    fig.savefig(out_path)\n    plt.close(fig)\n    return Path(out_path)\n"_FILES['sonarnet/viz/dashboard.py'] = "\"\"\"Bảng điều khiển bản đồ giám sát.\n\nSinh một trang HTML độc lập hiển thị toàn bộ phương tiện phát hiện được trên nền\nbản đồ biển, phân biệt theo trạng thái định danh. Tệp kết quả mở được bằng trình\nduyệt bất kỳ và không cần máy chủ, phù hợp để nhúng vào video trình diễn hoặc\ntrình chiếu trực tiếp trước Hội đồng.\n\nKhi thư viện ``folium`` không có sẵn, hệ thống tự động chuyển sang một trang HTML\ntự dựng bằng thư viện bản đồ nguồn mở Leaflet nạp từ mạng phân phối nội dung, và\nnếu vẫn không khả dụng thì kết xuất một biểu đồ phân bố tĩnh.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence\n\nimport numpy as np\n\nfrom ..data.simulator import AIS_MISMATCH, AIS_OK, DARK\nfrom ..utils import get_logger, package_available\n\nLOG = get_logger(\"viz.dashboard\")\n\nSTATE_STYLE = {\n    AIS_OK: {\"color\": \"#2E7D5B\", \"label\": \"Phát tín hiệu trung thực\", \"radius\": 5},\n    AIS_MISMATCH: {\"color\": \"#C79A2E\", \"label\": \"Kích thước khai báo sai lệch\", \"radius\": 6},\n    DARK: {\"color\": \"#C2412C\", \"label\": \"Không phát tín hiệu\", \"radius\": 8},\n}\n\n\ndef build_marker_table(\n    scenes_meta: Sequence[Dict],\n    records: Sequence[Dict],\n    limit_scenes: int = 40,\n) -> List[Dict]:\n    \"\"\"Lập bảng điểm đánh dấu từ kết quả hợp nhất.\"\"\"\n    by_scene: Dict[str, List[Dict]] = {}\n    for r in records:\n        by_scene.setdefault(r[\"scene_id\"], []).append(r)\n\n    markers: List[Dict] = []\n    for meta in list(scenes_meta)[:limit_scenes]:\n        sid = meta[\"scene_id\"]\n        recs = by_scene.get(sid, [])\n        # Ghép theo thứ tự phương tiện trong siêu dữ liệu\n        for v, r in zip(meta[\"vessels\"], recs):\n            state = r.get(\"pred_state\")\n            if state is None:\n                continue\n            markers.append(\n                {\n                    \"scene_id\": sid,\n                    \"lat\": float(v[\"lat\"]),\n                    \"lon\": float(v[\"lon\"]),\n                    \"state\": state,\n                    \"true_state\": r[\"true_state\"],\n                    \"mmsi\": r.get(\"pred_mmsi\"),\n                    \"length_m\": float(v[\"length_m\"]),\n                    \"speed_kn\": float(v[\"speed_kn\"]),\n                    \"behaviour\": v[\"behaviour\"],\n                    \"match_error_m\": (\n                        float(r[\"match_error_m\"])\n                        if r.get(\"match_error_m\") is not None\n                        and np.isfinite(r.get(\"match_error_m\", np.nan))\n                        else None\n                    ),\n                }\n            )\n    return markers\n\n\ndef _folium_map(markers: Sequence[Dict], out_path: Path, center) -> Path:\n    import folium\n\n    m = folium.Map(\n        location=list(center), zoom_start=8, tiles=\"CartoDB positron\",\n        control_scale=True,\n    )\n\n    groups = {\n        s: folium.FeatureGroup(name=STATE_STYLE[s][\"label\"], show=True)\n        for s in STATE_STYLE\n    }\n\n    for mk in markers:\n        style = STATE_STYLE.get(mk[\"state\"], STATE_STYLE[AIS_OK])\n        popup = folium.Popup(\n            html=(\n                f\"<div style='font-family:sans-serif;font-size:12px;min-width:220px'>\"\n                f\"<b style='color:#1C3557'>Phương tiện phát hiện</b><br>\"\n                f\"Trạng thái: <b>{style['label']}</b><br>\"\n                f\"Định danh ghép được: {mk['mmsi'] or 'không có'}<br>\"\n                f\"Chiều dài ước lượng: {mk['length_m']:.0f} m<br>\"\n                f\"Tốc độ: {mk['speed_kn']:.1f} hải lý/giờ<br>\"\n                f\"Hành vi: {mk['behaviour']}<br>\"\n                f\"Sai số ghép cặp: \"\n                f\"{('%.0f m' % mk['match_error_m']) if mk['match_error_m'] is not None else 'không xác định'}<br>\"\n                f\"<span style='color:#6B7A89'>Cảnh {mk['scene_id']}</span>\"\n                f\"</div>\"\n            ),\n            max_width=320,\n        )\n        folium.CircleMarker(\n            location=[mk[\"lat\"], mk[\"lon\"]],\n            radius=style[\"radius\"],\n            color=style[\"color\"],\n            fill=True,\n            fill_color=style[\"color\"],\n            fill_opacity=0.75,\n            weight=1.5,\n            popup=popup,\n        ).add_to(groups[mk[\"state\"]])\n\n    for g in groups.values():\n        g.add_to(m)\n    folium.LayerControl(collapsed=False).add_to(m)\n\n    legend = \"\"\"\n    <div style=\"position: fixed; bottom: 24px; left: 24px; z-index: 9999;\n                background: rgba(255,255,255,0.94); padding: 12px 14px;\n                border: 1px solid #B0BEC5; border-radius: 6px;\n                font-family: sans-serif; font-size: 12px; color: #1C3557;\">\n      <div style=\"font-weight:700; margin-bottom:6px;\">Trạng thái định danh</div>\n      <div><span style=\"display:inline-block;width:11px;height:11px;border-radius:50%;\n           background:#2E7D5B;margin-right:7px;\"></span>Phát tín hiệu trung thực</div>\n      <div><span style=\"display:inline-block;width:11px;height:11px;border-radius:50%;\n           background:#C79A2E;margin-right:7px;\"></span>Kích thước khai báo sai lệch</div>\n      <div><span style=\"display:inline-block;width:11px;height:11px;border-radius:50%;\n           background:#C2412C;margin-right:7px;\"></span>Không phát tín hiệu</div>\n    </div>\n    \"\"\"\n    m.get_root().html.add_child(folium.Element(legend))\n\n    out_path = Path(out_path)\n    m.save(str(out_path))\n    return out_path\n\n\ndef _static_fallback(markers: Sequence[Dict], out_path: Path) -> Path:\n    \"\"\"Biểu đồ phân bố tĩnh khi không dựng được bản đồ tương tác.\"\"\"\n    import matplotlib\n\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n\n    fig, ax = plt.subplots(figsize=(7.4, 6.4))\n    for state, style in STATE_STYLE.items():\n        pts = [m for m in markers if m[\"state\"] == state]\n        if not pts:\n            continue\n        ax.scatter(\n            [p[\"lon\"] for p in pts], [p[\"lat\"] for p in pts],\n            s=26 + 6 * style[\"radius\"], c=style[\"color\"],\n            label=style[\"label\"], alpha=0.75, edgecolors=\"white\", linewidths=0.6,\n        )\n    ax.set_xlabel(\"Kinh độ\")\n    ax.set_ylabel(\"Vĩ độ\")\n    ax.set_title(\"Phân bố phương tiện phát hiện theo trạng thái định danh\")\n    ax.legend(fontsize=9)\n    ax.grid(True, color=\"#E3E9EF\")\n    png = Path(out_path).with_suffix(\".png\")\n    fig.savefig(png, dpi=150, bbox_inches=\"tight\")\n    plt.close(fig)\n    return png\n\n\ndef build_dashboard(\n    scenes_meta: Sequence[Dict],\n    records: Sequence[Dict],\n    out_path: Path,\n    cfg=None,\n) -> Path:\n    \"\"\"Dựng bảng điều khiển bản đồ và trả về đường dẫn tệp kết quả.\"\"\"\n    markers = build_marker_table(scenes_meta, records)\n    if not markers:\n        LOG.warning(\"Không có điểm đánh dấu nào để hiển thị.\")\n        return Path(out_path)\n\n    center = (\n        float(np.mean([m[\"lat\"] for m in markers])),\n        float(np.mean([m[\"lon\"] for m in markers])),\n    )\n\n    # Lưu kèm dữ liệu thô để tiện tái sử dụng\n    Path(out_path).parent.mkdir(parents=True, exist_ok=True)\n    Path(out_path).with_suffix(\".json\").write_text(\n        json.dumps(markers, ensure_ascii=False, indent=1), encoding=\"utf-8\"\n    )\n\n    if package_available(\"folium\"):\n        try:\n            path = _folium_map(markers, out_path, center)\n            LOG.info(\"Đã dựng bảng điều khiển bản đồ: %s (%d điểm)\", path, len(markers))\n            return path\n        except Exception as exc:  # pragma: no cover\n            LOG.warning(\"Không dựng được bản đồ tương tác (%s). Chuyển sang biểu đồ tĩnh.\", exc)\n\n    path = _static_fallback(markers, out_path)\n    LOG.info(\"Đã kết xuất biểu đồ phân bố tĩnh: %s\", path)\n    return path\n"for _rel, _content in _FILES.items():    _p = SRC_DIR / _rel    _p.parent.mkdir(parents=True, exist_ok=True)    _p.write_text(_content, encoding='utf-8')print(f'Đã ghi {len(_FILES)} tệp mã nguồn.')for _rel in _FILES:    print('   ', _rel)

### Điều phối quy trình và báo cáo

In [ ]:
# Ghi mã nguồn ra đĩa. Nội dung giữ nguyên như trong kho mã nguồn._FILES = {}_FILES['sonarnet/pipeline.py'] = "\"\"\"Điều phối toàn bộ quy trình SonarNet-VN.\n\nMỗi hàm ``step_*`` tương ứng với một ô lệnh trong notebook và có thể chạy độc\nlập. Trạng thái trung gian được ghi ra đĩa nên có thể chạy lại từng bước mà\nkhông phải thực hiện lại các bước trước đó.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport subprocess\nimport sys\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom .data.dataset import (\n    build_dataset,\n    read_ais,\n    read_scene_meta,\n    georef_from_dict,\n)\nfrom .data.simulator import AIS_MISMATCH, AIS_OK, DARK\nfrom .evaluation.ablation import format_ablation_table, run_ablation\nfrom .evaluation.detection_metrics import (\n    evaluate_multi_iou,\n    match_predictions_to_truth,\n)\nfrom .evaluation.fusion_metrics import compare_interpolation_methods, evaluate_fusion\nfrom .fusion.matching import Detection, SARAISFusion\nfrom .utils import (\n    format_table,\n    get_logger,\n    gpu_utilisation,\n    probe_devices,\n    save_json,\n    set_seed,\n    timed,\n)\n\nLOG = get_logger(\"pipeline\")\n\n\n# ===========================================================================\n# Bước 1 — Chuẩn bị môi trường\n# ===========================================================================\ndef step_prepare(cfg) -> Dict:\n    \"\"\"Tạo thư mục, đặt hạt giống ngẫu nhiên và kiểm tra phần cứng.\"\"\"\n    cfg.make_dirs()\n    rng = set_seed(cfg.seed)\n\n    info = probe_devices()\n    LOG.info(\"Môi trường tính toán:\\n%s\", info.summary)\n\n    util = gpu_utilisation()\n    if util:\n        LOG.info(\"Trạng thái GPU hiện tại:\\n%s\", util)\n\n    cfg.to_json(cfg.dir_results / \"config.json\")\n    return {\n        \"rng\": rng,\n        \"n_gpu\": info.n_gpu,\n        \"gpu_names\": info.names,\n        \"root\": str(cfg.root),\n    }\n\n\n# ===========================================================================\n# Bước 2 — Sinh dữ liệu\n# ===========================================================================\ndef step_build_data(cfg, rng: np.random.Generator) -> Dict:\n    \"\"\"Sinh toàn bộ ba tập dữ liệu và ghi ra đĩa.\"\"\"\n    with timed(\"Sinh bộ dữ liệu ảnh radar và dòng AIS\"):\n        summary = build_dataset(cfg, rng)\n    save_json(summary, cfg.dir_results / \"dataset_summary.json\")\n\n    LOG.info(\"Tóm tắt bộ dữ liệu:\\n%s\", format_table(summary[\"splits\"]))\n    return summary\n\n\n# ===========================================================================\n# Bước 3 — Huấn luyện mô hình phát hiện\n# ===========================================================================\ndef step_train_detector(cfg, use_subprocess: bool = True) -> Dict:\n    \"\"\"Huấn luyện mô hình phát hiện phương tiện.\n\n    Khi có từ hai GPU trở lên và phương án Ultralytics khả dụng, việc huấn luyện\n    được thực hiện trong một tiến trình con để quá trình khởi tạo phân tán diễn\n    ra ổn định. Đây là cách làm được khuyến nghị khi chạy trong nhân notebook.\n    \"\"\"\n    from .detect.interface import build_detector, select_backend\n\n    data_yaml = cfg.dir_yolo / \"data.yaml\"\n    if not data_yaml.exists():\n        raise FileNotFoundError(\n            f\"Chưa có mô tả dữ liệu tại {data_yaml}. Hãy chạy bước sinh dữ liệu trước.\"\n        )\n\n    backend = select_backend(cfg.detect.backend)\n    n_gpu = probe_devices().n_gpu\n    multi = cfg.multi_gpu and n_gpu >= 2 and backend == \"ultralytics\"\n\n    if multi and use_subprocess:\n        script = Path(__file__).resolve().parents[2] / \"scripts\" / \"train_detector.py\"\n        if script.exists():\n            LOG.info(\n                \"Huấn luyện phân tán trên %d GPU thông qua tiến trình con.\", n_gpu\n            )\n            cmd = [\n                sys.executable, str(script),\n                \"--data\", str(data_yaml),\n                \"--config\", str(cfg.dir_results / \"config.json\"),\n            ]\n            with timed(\"Huấn luyện mô hình phát hiện (phân tán)\"):\n                proc = subprocess.run(cmd, text=True)\n            if proc.returncode != 0:\n                LOG.warning(\n                    \"Tiến trình con kết thúc với mã %d. Chuyển sang huấn luyện \"\n                    \"trong tiến trình hiện tại.\", proc.returncode\n                )\n            else:\n                best = cfg.dir_runs / \"yolo_sar\" / \"weights\" / \"best.pt\"\n                return {\"backend\": backend, \"weights\": str(best),\n                        \"distributed\": True, \"n_gpu\": n_gpu}\n\n    detector = build_detector(cfg)\n    with timed(\"Huấn luyện mô hình phát hiện\"):\n        info = detector.train(data_yaml, cfg)\n    info[\"distributed\"] = False\n    save_json(info, cfg.dir_results / \"train_info.json\")\n    return info\n\n\n# ===========================================================================\n# Bước 4 — Đánh giá mô hình phát hiện\n# ===========================================================================\ndef step_eval_detector(cfg, weights: Optional[Path] = None, split: str = \"test\") -> Dict:\n    \"\"\"Suy luận trên tập kiểm tra và tính các chỉ tiêu phát hiện.\"\"\"\n    from .detect.interface import build_detector\n\n    detector = build_detector(cfg)\n\n    if weights is None:\n        candidates = [\n            cfg.dir_runs / \"yolo_sar\" / \"weights\" / \"best.pt\",\n            cfg.dir_runs / \"frcnn_sar\" / \"weights\" / \"best.pt\",\n        ]\n        weights = next((c for c in candidates if c.exists()), None)\n    if weights is None or not Path(weights).exists():\n        raise FileNotFoundError(\"Không tìm thấy trọng số mô hình đã huấn luyện.\")\n\n    detector.load(Path(weights))\n\n    img_dir = cfg.dir_yolo / \"images\" / split\n    paths = sorted(img_dir.glob(\"*.png\"))\n    LOG.info(\"Suy luận trên %d ảnh của tập %s.\", len(paths), split)\n\n    with timed(f\"Suy luận tập {split}\"):\n        outs = detector.predict_sharded(paths, conf=cfg.detect.conf_threshold)\n\n    predictions = {o.image_id: (o.boxes, o.scores) for o in outs}\n\n    # Khung bao đối chứng lấy từ siêu dữ liệu cảnh\n    meta = read_scene_meta(cfg.dir_yolo, split)\n    ground_truth = {\n        m[\"scene_id\"]: np.array([v[\"bbox\"] for v in m[\"vessels\"]], dtype=np.float32)\n        if m[\"vessels\"] else np.zeros((0, 4), np.float32)\n        for m in meta\n    }\n\n    metrics = evaluate_multi_iou(\n        predictions, ground_truth, score_threshold=cfg.detect.conf_threshold\n    )\n    LOG.info(\"Chỉ tiêu phát hiện: %s\", json.dumps(metrics.to_dict(), ensure_ascii=False))\n\n    save_json(metrics.to_dict(), cfg.dir_results / f\"detection_metrics_{split}.json\")\n\n    # Lưu dự đoán để các bước sau dùng lại mà không phải suy luận lại\n    np.savez_compressed(\n        cfg.dir_results / f\"predictions_{split}.npz\",\n        **{f\"{k}__boxes\": v[0] for k, v in predictions.items()},\n        **{f\"{k}__scores\": v[1] for k, v in predictions.items()},\n    )\n\n    return {\n        \"metrics\": metrics,\n        \"predictions\": predictions,\n        \"ground_truth\": ground_truth,\n        \"scene_meta\": meta,\n        \"weights\": str(weights),\n    }\n\n\ndef load_predictions(cfg, split: str = \"test\") -> Dict[str, Tuple[np.ndarray, np.ndarray]]:\n    \"\"\"Nạp lại dự đoán đã lưu ở bước đánh giá.\"\"\"\n    path = cfg.dir_results / f\"predictions_{split}.npz\"\n    data = np.load(path)\n    out: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}\n    for key in data.files:\n        if key.endswith(\"__boxes\"):\n            sid = key[: -len(\"__boxes\")]\n            out[sid] = (data[key], data[f\"{sid}__scores\"])\n    return out\n\n\n# ===========================================================================\n# Bước 5 — Hợp nhất ảnh radar và AIS\n# ===========================================================================\ndef run_fusion_records(\n    cfg,\n    scenes_meta: Sequence[Dict],\n    ais_by_scene: Dict[str, List[Dict]],\n    detections_by_scene: Dict[str, Tuple[np.ndarray, np.ndarray]],\n    use_kalman: bool = True,\n    use_size_check: bool = True,\n) -> List[Dict]:\n    \"\"\"Chạy tầng hợp nhất và trả về bản ghi chi tiết theo từng phương tiện.\n\n    Mỗi bản ghi tương ứng với một phương tiện đối chứng, gồm trạng thái thật,\n    trạng thái dự đoán, định danh ghép được và sai số ghép cặp. Phương tiện bị\n    mô hình phát hiện bỏ sót mang ``pred_state`` bằng ``None``.\n    \"\"\"\n    import copy as _copy\n\n    local_cfg = _copy.deepcopy(cfg)\n    if not use_size_check:\n        # Đặt ngưỡng vượt quá giá trị tối đa có thể để vô hiệu hoá phép kiểm tra\n        local_cfg.fusion.size_mismatch_ratio = 10.0\n\n    fusion = SARAISFusion(local_cfg, use_kalman=use_kalman)\n    records: List[Dict] = []\n\n    for meta in scenes_meta:\n        sid = meta[\"scene_id\"]\n        vessels = meta[\"vessels\"]\n        if not vessels:\n            continue\n\n        georef = georef_from_dict(meta[\"georef\"])\n        gt_boxes = np.array([v[\"bbox\"] for v in vessels], dtype=np.float32)\n\n        pred_boxes, pred_scores = detections_by_scene.get(\n            sid, (np.zeros((0, 4), np.float32), np.zeros((0,), np.float32))\n        )\n\n        # Ghép phát hiện với phương tiện đối chứng theo IoU\n        det_to_gt = match_predictions_to_truth(\n            pred_boxes, gt_boxes, iou_threshold=cfg.detect.iou_threshold\n        )\n        gt_to_det = {g: d for d, g in det_to_gt.items()}\n\n        # Chỉ đưa vào tầng hợp nhất những phát hiện khớp với đối chứng, để chỉ\n        # tiêu của tầng này không bị pha trộn với sai số của tầng phát hiện\n        det_indices = sorted(det_to_gt.keys())\n        dets = []\n        for new_i, di in enumerate(det_indices):\n            x1, y1, x2, y2 = [float(v) for v in pred_boxes[di]]\n            cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0\n            lon, lat = georef.pixel_to_lonlat(cx, cy)\n            w_px, h_px = abs(x2 - x1), abs(y2 - y1)\n            spacing = cfg.geo.pixel_spacing_m\n            dets.append(\n                Detection(\n                    det_id=new_i, lat=float(lat), lon=float(lon),\n                    length_m=max(w_px, h_px) * spacing,\n                    width_m=min(w_px, h_px) * spacing,\n                    bbox=(x1, y1, x2, y2), score=float(pred_scores[di]),\n                )\n            )\n\n        result = fusion.run(\n            sid, dets, ais_by_scene.get(sid, []), float(meta[\"capture_time_s\"])\n        )\n\n        # Lập bản ghi cho từng phương tiện đối chứng\n        for gi, v in enumerate(vessels):\n            di = gt_to_det.get(gi)\n            if di is None:\n                records.append(\n                    {\n                        \"scene_id\": sid,\n                        \"true_state\": v[\"identity_state\"],\n                        \"pred_state\": None,\n                        \"true_mmsi\": v[\"mmsi\"],\n                        \"pred_mmsi\": None,\n                        \"match_error_m\": float(\"nan\"),\n                        \"detected\": False,\n                    }\n                )\n                continue\n            new_i = det_indices.index(di)\n            records.append(\n                {\n                    \"scene_id\": sid,\n                    \"true_state\": v[\"identity_state\"],\n                    \"pred_state\": result.states[new_i],\n                    \"true_mmsi\": v[\"mmsi\"],\n                    \"pred_mmsi\": result.matched_mmsi[new_i],\n                    \"match_error_m\": result.match_distance_m[new_i],\n                    \"detected\": True,\n                }\n            )\n\n    return records\n\n\ndef step_fusion(cfg, eval_out: Dict, split: str = \"test\") -> Dict:\n    \"\"\"Chạy tầng hợp nhất trên tập kiểm tra và tính các chỉ tiêu.\"\"\"\n    scenes_meta = eval_out[\"scene_meta\"]\n    ais_by_scene = read_ais(cfg.dir_yolo, split)\n    detections = eval_out[\"predictions\"]\n\n    with timed(\"Hợp nhất ảnh radar và tín hiệu AIS\"):\n        records = run_fusion_records(\n            cfg, scenes_meta, ais_by_scene, detections,\n            use_kalman=True, use_size_check=True,\n        )\n\n    visible = [r for r in records if r[\"pred_state\"] is not None]\n    missed_dark = [\n        r for r in records if r[\"pred_state\"] is None and r[\"true_state\"] == DARK\n    ]\n\n    true_states = [r[\"true_state\"] for r in visible] + [DARK] * len(missed_dark)\n    pred_states = [r[\"pred_state\"] for r in visible] + [AIS_OK] * len(missed_dark)\n    true_mmsi = [r[\"true_mmsi\"] for r in visible] + [None] * len(missed_dark)\n    pred_mmsi = [r[\"pred_mmsi\"] for r in visible] + [None] * len(missed_dark)\n    errs = [r[\"match_error_m\"] for r in visible] + [float(\"nan\")] * len(missed_dark)\n\n    metrics = evaluate_fusion(true_states, pred_states, true_mmsi, pred_mmsi, errs)\n    LOG.info(\n        \"Chỉ tiêu hợp nhất: %s\",\n        json.dumps(metrics.to_dict(), ensure_ascii=False),\n    )\n    save_json(metrics.to_dict(), cfg.dir_results / \"fusion_metrics.json\")\n\n    # So sánh riêng hai phương pháp nội suy\n    with timed(\"So sánh bộ lọc Kalman với nội suy tuyến tính\"):\n        interp = compare_interpolation_methods(scenes_meta, ais_by_scene, cfg)\n    save_json(interp, cfg.dir_results / \"interpolation_comparison.json\")\n    LOG.info(\n        \"Sai số nội suy — Kalman %.1f m | Tuyến tính %.1f m (trung vị)\",\n        interp[\"kalman\"][\"median_error_m\"], interp[\"linear\"][\"median_error_m\"],\n    )\n\n    return {\n        \"metrics\": metrics,\n        \"records\": records,\n        \"interpolation\": interp,\n        \"ais_by_scene\": ais_by_scene,\n    }\n\n\n# ===========================================================================\n# Bước 6 — Phân loại hành vi\n# ===========================================================================\ndef step_behavior(cfg, rng: np.random.Generator) -> Dict:\n    \"\"\"Sinh quỹ đạo, trích đặc trưng và huấn luyện bộ phân loại hành vi.\"\"\"\n    from .behavior.classifier import train_and_evaluate\n    from .behavior.features import FEATURE_NAMES, features_matrix\n    from .data.tracks import generate_track_dataset\n\n    with timed(\"Sinh tập quỹ đạo phương tiện\"):\n        tracks = generate_track_dataset(cfg, rng)\n    LOG.info(\"Đã sinh %d quỹ đạo trên %d nhóm hành vi.\",\n             len(tracks), len(cfg.behavior.classes))\n\n    with timed(\"Trích xuất đặc trưng động học\"):\n        X, y = features_matrix(tracks)\n    LOG.info(\"Ma trận đặc trưng: %s\", X.shape)\n\n    with timed(\"Huấn luyện bộ phân loại hành vi\"):\n        clf, report = train_and_evaluate(X, y, FEATURE_NAMES, cfg)\n\n    clf.save(cfg.dir_runs / \"behavior\" / \"model.pkl\")\n    save_json(report.to_dict(), cfg.dir_results / \"behavior_metrics.json\")\n    return {\"classifier\": clf, \"report\": report, \"X\": X, \"y\": y, \"tracks\": tracks}\n\n\n# ===========================================================================\n# Bước 7 — Phân tích đóng góp thành phần\n# ===========================================================================\ndef step_ablation(cfg, eval_out: Dict, fusion_out: Dict, split: str = \"test\") -> Dict:\n    \"\"\"Chạy bốn cấu hình so sánh và kết xuất bảng tổng hợp.\"\"\"\n    with timed(\"Phân tích đóng góp thành phần\"):\n        rows, metrics_map = run_ablation(\n            run_fusion_records,\n            cfg,\n            eval_out[\"scene_meta\"],\n            fusion_out[\"ais_by_scene\"],\n            eval_out[\"predictions\"],\n        )\n\n    table = format_ablation_table(rows)\n    LOG.info(\"Bảng phân tích đóng góp thành phần:\\n%s\", table)\n\n    payload = {\n        \"rows\": [r.to_dict() for r in rows],\n        \"metrics\": {k: v.to_dict() for k, v in metrics_map.items()},\n    }\n    save_json(payload, cfg.dir_results / \"ablation.json\")\n    (cfg.dir_results / \"ablation_table.txt\").write_text(table, encoding=\"utf-8\")\n\n    try:\n        import pandas as pd\n\n        pd.DataFrame([r.to_dict() for r in rows]).to_csv(\n            cfg.dir_results / \"ablation.csv\", index=False, encoding=\"utf-8-sig\"\n        )\n    except Exception:\n        pass\n\n    return {\"rows\": rows, \"metrics\": metrics_map, \"table\": table}\n"_FILES['sonarnet/reporting.py'] = "\"\"\"Kết xuất hình minh hoạ và báo cáo tổng hợp cho một lần chạy.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence\n\nimport numpy as np\n\nfrom .data.dataset import load_image\nfrom .data.simulator import AIS_MISMATCH, AIS_OK, DARK, IDENTITY_STATES\nfrom .utils import get_logger, save_json, timed\nfrom .viz import dashboard as dash\nfrom .viz import figures as fig\n\nLOG = get_logger(\"reporting\")\n\nBEHAVIOUR_LABELS = {\n    \"qua_canh\": \"Quá cảnh\",\n    \"cau\": \"Câu\",\n    \"keo_luoi\": \"Kéo lưới\",\n    \"neo_dau\": \"Neo đậu\",\n}\nSTATE_SHORT = {\n    AIS_OK: \"Trung thực\",\n    AIS_MISMATCH: \"Sai kích thước\",\n    DARK: \"Không phát AIS\",\n}\n\n\ndef make_figures(\n    cfg,\n    eval_out: Dict,\n    fusion_out: Dict,\n    behaviour_out: Dict,\n    ablation_out: Dict,\n    split: str = \"test\",\n) -> Dict[str, str]:\n    \"\"\"Sinh toàn bộ hình minh hoạ và trả về đường dẫn từng tệp.\"\"\"\n    out: Dict[str, str] = {}\n    figs = cfg.dir_figures\n    figs.mkdir(parents=True, exist_ok=True)\n\n    # 1. Một cảnh ảnh tiêu biểu, ưu tiên cảnh có phương tiện ngắt định danh\n    meta_list = eval_out[\"scene_meta\"]\n    chosen = None\n    for m in meta_list:\n        if any(v[\"identity_state\"] == DARK for v in m[\"vessels\"]) and len(m[\"vessels\"]) >= 5:\n            chosen = m\n            break\n    chosen = chosen or (meta_list[0] if meta_list else None)\n\n    if chosen is not None:\n        img_path = cfg.dir_yolo / \"images\" / split / f\"{chosen['scene_id']}.png\"\n        if img_path.exists():\n            image = load_image(img_path)\n            gt_boxes = np.array([v[\"bbox\"] for v in chosen[\"vessels\"]], dtype=np.float32)\n            gt_states = [v[\"identity_state\"] for v in chosen[\"vessels\"]]\n            pred = eval_out[\"predictions\"].get(chosen[\"scene_id\"])\n            out[\"scene\"] = str(\n                fig.plot_scene_with_boxes(\n                    image, gt_boxes, gt_states,\n                    pred[0] if pred is not None else None,\n                    figs / \"01_canh_anh_radar.png\",\n                    title=f\"Cảnh {chosen['scene_id']} — đối chứng và phát hiện\",\n                )\n            )\n\n    # 2. Đường cong chính xác – độ nhạy\n    dm = eval_out[\"metrics\"]\n    recall, precision = dm.pr_curve\n    if recall.size:\n        out[\"pr_curve\"] = str(\n            fig.plot_pr_curve(recall, precision, dm.ap50, figs / \"02_duong_cong_pr.png\")\n        )\n\n    # 3. Ma trận nhầm lẫn của tầng hợp nhất\n    fm = fusion_out[\"metrics\"]\n    out[\"fusion_confusion\"] = str(\n        fig.plot_confusion(\n            fm.confusion_matrix, list(IDENTITY_STATES),\n            figs / \"03_ma_tran_hop_nhat.png\",\n            title=\"Phân loại ba trạng thái định danh\",\n            display_labels=[STATE_SHORT[s] for s in IDENTITY_STATES],\n        )\n    )\n\n    # 4. Phân bố sai số ghép cặp\n    out[\"match_errors\"] = str(\n        fig.plot_match_errors(\n            fusion_out[\"records\"], figs / \"04_sai_so_ghep_cap.png\",\n            threshold_m=cfg.fusion.max_match_distance_m,\n        )\n    )\n\n    # 5. So sánh hai phương pháp nội suy\n    out[\"interpolation\"] = str(\n        fig.plot_interpolation_comparison(\n            fusion_out[\"interpolation\"], figs / \"05_so_sanh_noi_suy.png\"\n        )\n    )\n\n    # 6. Ma trận nhầm lẫn của bộ phân loại hành vi\n    br = behaviour_out[\"report\"]\n    out[\"behaviour_confusion\"] = str(\n        fig.plot_confusion(\n            br.confusion_matrix, br.classes,\n            figs / \"06_ma_tran_hanh_vi.png\",\n            title=\"Phân loại bốn nhóm hành vi hoạt động\",\n            display_labels=[BEHAVIOUR_LABELS.get(c, c) for c in br.classes],\n        )\n    )\n\n    # 7. Quỹ đạo tiêu biểu\n    out[\"tracks\"] = str(\n        fig.plot_behaviour_tracks(behaviour_out[\"tracks\"], figs / \"07_quy_dao_hanh_vi.png\")\n    )\n\n    # 8. Độ quan trọng đặc trưng\n    p = fig.plot_feature_importance(br.feature_importance, figs / \"08_dac_trung_quan_trong.png\")\n    if p:\n        out[\"feature_importance\"] = str(p)\n\n    # 9. Bảng phân tích đóng góp thành phần\n    out[\"ablation\"] = str(\n        fig.plot_ablation(\n            [r.to_dict() for r in ablation_out[\"rows\"]], figs / \"09_dong_gop_thanh_phan.png\"\n        )\n    )\n\n    LOG.info(\"Đã kết xuất %d hình minh hoạ vào %s\", len(out), figs)\n    return out\n\n\ndef make_dashboard(cfg, eval_out: Dict, fusion_out: Dict) -> str:\n    \"\"\"Dựng bảng điều khiển bản đồ giám sát.\"\"\"\n    path = dash.build_dashboard(\n        eval_out[\"scene_meta\"], fusion_out[\"records\"],\n        cfg.dir_results / \"ban_do_giam_sat.html\", cfg,\n    )\n    return str(path)\n\n\ndef _fmt(v, nd: int = 4) -> str:\n    if v is None:\n        return \"—\"\n    if isinstance(v, float):\n        if not np.isfinite(v):\n            return \"—\"\n        return f\"{v:.{nd}f}\"\n    return str(v)\n\n\ndef write_report(\n    cfg,\n    dataset_summary: Dict,\n    eval_out: Dict,\n    fusion_out: Dict,\n    behaviour_out: Dict,\n    ablation_out: Dict,\n    figure_paths: Dict[str, str],\n    dashboard_path: str,\n    train_info: Optional[Dict] = None,\n) -> Path:\n    \"\"\"Ghi báo cáo tổng hợp dạng Markdown.\"\"\"\n    dm = eval_out[\"metrics\"].to_dict()\n    fm = fusion_out[\"metrics\"].to_dict()\n    br = behaviour_out[\"report\"].to_dict()\n    interp = fusion_out[\"interpolation\"]\n\n    lines: List[str] = []\n    A = lines.append\n\n    A(\"# SonarNet-VN — Báo cáo kết quả thực nghiệm\")\n    A(\"\")\n    A(f\"*Thời điểm kết xuất: {datetime.now().strftime('%d/%m/%Y %H:%M')}*\")\n    A(\"\")\n    A(\"Hệ thống giám sát tuân thủ quy định chống khai thác thuỷ sản bất hợp pháp, \"\n      \"không khai báo và không theo quy định, trên cơ sở hợp nhất ảnh vệ tinh radar \"\n      \"khẩu độ tổng hợp với tín hiệu giám sát hành trình tàu cá.\")\n    A(\"\")\n\n    # ---- Cấu hình --------------------------------------------------------\n    A(\"## 1. Cấu hình lần chạy\")\n    A(\"\")\n    A(\"| Tham số | Giá trị |\")\n    A(\"|---|---|\")\n    A(f\"| Chế độ | {'Rút gọn' if cfg.quick_mode else 'Đầy đủ'} |\")\n    A(f\"| Hạt giống ngẫu nhiên | {cfg.seed} |\")\n    A(f\"| Kích thước cảnh | {cfg.data.scene_size} × {cfg.data.scene_size} điểm ảnh |\")\n    A(f\"| Độ phân giải mặt đất | {cfg.geo.pixel_spacing_m:.0f} m/điểm ảnh |\")\n    A(f\"| Tỉ lệ phương tiện ngắt định danh | {cfg.data.dark_vessel_ratio:.0%} |\")\n    A(f\"| Số chu kỳ huấn luyện | {cfg.detect.epochs} |\")\n    if train_info:\n        A(f\"| Phương án phát hiện | {train_info.get('backend', 'không rõ')} |\")\n        A(f\"| Huấn luyện phân tán | {'có' if train_info.get('distributed') else 'không'} |\")\n    A(f\"| Ngưỡng ghép cặp | {cfg.fusion.max_match_distance_m:.0f} m |\")\n    A(\"\")\n\n    # ---- Dữ liệu ---------------------------------------------------------\n    A(\"## 2. Bộ dữ liệu\")\n    A(\"\")\n    A(\"| Tập | Số cảnh | Số phương tiện | Số khung bao | Số bản ghi AIS |\")\n    A(\"|---|---:|---:|---:|---:|\")\n    for s in dataset_summary[\"splits\"]:\n        A(f\"| {s['split']} | {s['n_scenes']} | {s['n_vessels']} | \"\n          f\"{s['n_boxes']} | {s['n_ais_records']} |\")\n    A(\"\")\n\n    # ---- Phát hiện -------------------------------------------------------\n    A(\"## 3. Tầng phát hiện phương tiện\")\n    A(\"\")\n    A(\"| Chỉ tiêu | Giá trị | Mục tiêu đề ra |\")\n    A(\"|---|---:|---:|\")\n    A(f\"| mAP@0.5 | {dm['mAP@0.5']:.4f} | ≥ 0,70 |\")\n    A(f\"| mAP@0.5:0.95 | {dm['mAP@0.5:0.95']:.4f} | — |\")\n    A(f\"| Độ chính xác | {dm['precision']:.4f} | — |\")\n    A(f\"| Độ nhạy | {dm['recall']:.4f} | ≥ 0,80 |\")\n    A(f\"| F1 | {dm['f1']:.4f} | — |\")\n    A(f\"| Dương tính thật / giả / bỏ sót | {dm['true_positive']} / \"\n      f\"{dm['false_positive']} / {dm['false_negative']} | — |\")\n    A(\"\")\n    if \"pr_curve\" in figure_paths:\n        A(f\"![Đường cong chính xác – độ nhạy]({Path(figure_paths['pr_curve']).name})\")\n        A(\"\")\n\n    # ---- Hợp nhất --------------------------------------------------------\n    A(\"## 4. Tầng hợp nhất ảnh radar và AIS\")\n    A(\"\")\n    A(\"| Chỉ tiêu | Giá trị | Mục tiêu đề ra |\")\n    A(\"|---|---:|---:|\")\n    A(f\"| Tỉ lệ ghép cặp chính xác | {fm['match_accuracy']:.4f} | ≥ 0,85 |\")\n    A(f\"| Sai số ghép cặp trung bình | {_fmt(fm['mean_match_error_m'], 1)} m | ≤ 200 m |\")\n    A(f\"| Sai số ghép cặp trung vị | {_fmt(fm['median_match_error_m'], 1)} m | — |\")\n    A(f\"| Độ chính xác lớp không phát AIS | {fm['dark_precision']:.4f} | ≥ 0,75 |\")\n    A(f\"| Độ nhạy lớp không phát AIS | {fm['dark_recall']:.4f} | ≥ 0,70 |\")\n    A(f\"| F1 lớp không phát AIS | {fm['dark_f1']:.4f} | — |\")\n    A(f\"| Độ chính xác ba trạng thái | {fm['state_accuracy']:.4f} | — |\")\n    A(f\"| F1 vĩ mô ba trạng thái | {fm['state_macro_f1']:.4f} | — |\")\n    A(\"\")\n    A(\"### So sánh phương pháp nội suy quỹ đạo\")\n    A(\"\")\n    A(\"| Phương pháp | Sai số trung bình | Sai số trung vị | Bách phân vị 90 |\")\n    A(\"|---|---:|---:|---:|\")\n    for key, name in ((\"kalman\", \"Bộ lọc Kalman\"), (\"linear\", \"Nội suy tuyến tính\")):\n        if key in interp:\n            d = interp[key]\n            A(f\"| {name} | {_fmt(d['mean_error_m'], 1)} m | \"\n              f\"{_fmt(d['median_error_m'], 1)} m | {_fmt(d['p90_error_m'], 1)} m |\")\n    A(\"\")\n\n    # ---- Hành vi ---------------------------------------------------------\n    A(\"## 5. Tầng phân loại hành vi hoạt động\")\n    A(\"\")\n    A(f\"Thuật toán sử dụng: **{br['backend']}**. \"\n      f\"Huấn luyện trên {br['n_train']} quỹ đạo, kiểm tra trên {br['n_test']} quỹ đạo.\")\n    A(\"\")\n    A(\"| Chỉ tiêu | Giá trị | Mục tiêu đề ra |\")\n    A(\"|---|---:|---:|\")\n    A(f\"| Độ chính xác | {br['accuracy']:.4f} | — |\")\n    A(f\"| F1 vĩ mô | {br['macro_f1']:.4f} | ≥ 0,70 |\")\n    A(\"\")\n    A(\"| Nhóm hành vi | F1 |\")\n    A(\"|---|---:|\")\n    for k, v in br[\"per_class_f1\"].items():\n        A(f\"| {BEHAVIOUR_LABELS.get(k, k)} | {v:.4f} |\")\n    A(\"\")\n\n    # ---- Ablation --------------------------------------------------------\n    A(\"## 6. Phân tích đóng góp của từng thành phần\")\n    A(\"\")\n    rows = [r.to_dict() for r in ablation_out[\"rows\"]]\n    if rows:\n        headers = list(rows[0].keys())\n        A(\"| \" + \" | \".join(headers) + \" |\")\n        A(\"|\" + \"|\".join([\"---\"] * len(headers)) + \"|\")\n        for r in rows:\n            A(\"| \" + \" | \".join(_fmt(r[h]) if isinstance(r[h], float) else str(r[h] if r[h] is not None else \"—\")\n                                for h in headers) + \" |\")\n    A(\"\")\n    A(\"Bảng trên cho thấy giá trị cốt lõi của việc hợp nhất hai nguồn dữ liệu. \"\n      \"Cấu hình chỉ dùng ảnh radar phát hiện được phương tiện nhưng không có căn cứ \"\n      \"để xác định trạng thái định danh. Cấu hình chỉ dùng AIS bỏ sót hoàn toàn các \"\n      \"phương tiện chủ động ngắt tín hiệu — đúng những trường hợp cần phát hiện nhất. \"\n      \"Chỉ khi hợp nhất cả hai nguồn, hệ thống mới đồng thời đạt độ nhạy phát hiện cao \"\n      \"và khả năng nhận diện phương tiện ngắt định danh.\")\n    A(\"\")\n\n    # ---- Hình và sản phẩm ------------------------------------------------\n    A(\"## 7. Sản phẩm kết xuất\")\n    A(\"\")\n    A(\"| Tệp | Nội dung |\")\n    A(\"|---|---|\")\n    names = {\n        \"scene\": \"Cảnh ảnh radar kèm đối chứng và phát hiện\",\n        \"pr_curve\": \"Đường cong chính xác – độ nhạy\",\n        \"fusion_confusion\": \"Ma trận nhầm lẫn ba trạng thái định danh\",\n        \"match_errors\": \"Phân bố sai số ghép cặp\",\n        \"interpolation\": \"So sánh hai phương pháp nội suy\",\n        \"behaviour_confusion\": \"Ma trận nhầm lẫn bốn nhóm hành vi\",\n        \"tracks\": \"Quỹ đạo tiêu biểu của từng nhóm hành vi\",\n        \"feature_importance\": \"Đặc trưng động học quan trọng nhất\",\n        \"ablation\": \"Biểu đồ đóng góp của từng thành phần\",\n    }\n    for key, desc in names.items():\n        if key in figure_paths:\n            A(f\"| `{Path(figure_paths[key]).name}` | {desc} |\")\n    A(f\"| `{Path(dashboard_path).name}` | Bảng điều khiển bản đồ giám sát |\")\n    A(\"\")\n\n    A(\"## 8. Ghi chú về dữ liệu\")\n    A(\"\")\n    A(\"Kết quả trong báo cáo này được tạo trên bộ dữ liệu mô phỏng có nhãn đối chứng \"\n      \"đầy đủ. Bộ mô phỏng tái tạo các đặc trưng vật lý chính của ảnh radar khẩu độ \"\n      \"tổng hợp trên biển: tán xạ nền Rayleigh, nhiễu đốm nhân tính, điều biến do gió, \"\n      \"vệt nước sau tàu và bóng ma phương vị. Mục đích là kiểm chứng tính đúng đắn của \"\n      \"toàn bộ kiến trúc xử lý và thiết lập mức tham chiếu cho từng tầng.\")\n    A(\"\")\n    A(\"Để chuyển sang dữ liệu thật, thay thế bước sinh dữ liệu bằng ảnh Sentinel-1 tải \"\n      \"từ Copernicus Data Space và dòng AIS từ Global Fishing Watch; toàn bộ các tầng \"\n      \"phía sau giữ nguyên không đổi. Hướng dẫn chi tiết nằm trong tệp `README.md`.\")\n    A(\"\")\n\n    path = cfg.dir_results / \"BAO_CAO_KET_QUA.md\"\n    path.write_text(\"\\n\".join(lines), encoding=\"utf-8\")\n    LOG.info(\"Đã ghi báo cáo tổng hợp: %s\", path)\n\n    # Đồng thời lưu bản tổng hợp dạng máy đọc được\n    save_json(\n        {\n            \"detection\": dm,\n            \"fusion\": fm,\n            \"behaviour\": br,\n            \"interpolation\": interp,\n            \"ablation\": rows,\n            \"figures\": figure_paths,\n            \"dashboard\": dashboard_path,\n        },\n        cfg.dir_results / \"tong_hop_ket_qua.json\",\n    )\n    return path\n\n\ndef step_report(\n    cfg,\n    dataset_summary: Dict,\n    eval_out: Dict,\n    fusion_out: Dict,\n    behaviour_out: Dict,\n    ablation_out: Dict,\n    train_info: Optional[Dict] = None,\n    split: str = \"test\",\n) -> Dict:\n    \"\"\"Bước cuối: sinh hình, bảng điều khiển và báo cáo.\"\"\"\n    with timed(\"Kết xuất hình minh hoạ\"):\n        figure_paths = make_figures(\n            cfg, eval_out, fusion_out, behaviour_out, ablation_out, split\n        )\n    with timed(\"Dựng bảng điều khiển bản đồ\"):\n        dashboard_path = make_dashboard(cfg, eval_out, fusion_out)\n    with timed(\"Ghi báo cáo tổng hợp\"):\n        report_path = write_report(\n            cfg, dataset_summary, eval_out, fusion_out, behaviour_out,\n            ablation_out, figure_paths, dashboard_path, train_info,\n        )\n    return {\n        \"figures\": figure_paths,\n        \"dashboard\": dashboard_path,\n        \"report\": str(report_path),\n    }\n"for _rel, _content in _FILES.items():    _p = SRC_DIR / _rel    _p.parent.mkdir(parents=True, exist_ok=True)    _p.write_text(_content, encoding='utf-8')print(f'Đã ghi {len(_FILES)} tệp mã nguồn.')for _rel in _FILES:    print('   ', _rel)

---# Bước 2 — Nạp gói và thiết lập cấu hìnhMọi tham số điều khiển hệ thống nằm trong đối tượng `CFG`. Sửa ở đây là thay đổi toàn bộ quy trình.

In [ ]:
import importlibimport sonarnetimportlib.reload(sonarnet)from sonarnet.config import CFG, describefrom sonarnet import pipeline as Pfrom sonarnet.reporting import step_reportfrom sonarnet.utils import timed# ---------------------------------------------------------------------------# CHẾ ĐỘ RÚT GỌN# Bỏ dấu chú thích ở dòng dưới để chạy thử toàn tuyến trong khoảng sáu phút.# Sau khi xác nhận mọi bước chạy thông, đặt lại dấu chú thích và chạy lại# để có kết quả đầy đủ.# ---------------------------------------------------------------------------# CFG.apply_quick_mode()# Chọn phương án phát hiện theo kết quả cài đặt ở Bước 0CFG.detect.backend = "ultralytics" if HAS_ULTRALYTICS else "torchvision"print(describe(CFG))print(f"\nPhương án phát hiện   : {CFG.detect.backend}")

---# Bước 3 — Chuẩn bịTạo thư mục làm việc, đặt hạt giống ngẫu nhiên cho toàn bộ thư viện để kết quả tái lập được, và ghi lại cấu hình.

In [ ]:
prep = P.step_prepare(CFG)rng = prep["rng"]print(f"\nSố GPU khả dụng: {prep['n_gpu']}")print(f"Thư mục kết quả: {CFG.dir_results}")

---# Bước 4 — Sinh bộ dữ liệuHệ thống sinh ra các cảnh ảnh radar mô phỏng cùng dòng tín hiệu AIS gắn chặt với chúng.**Vì sao dùng dữ liệu mô phỏng.** Có hai lý do. Thứ nhất, nó cho phép toàn bộ hệ thống chạy được từ đầu đến cuối mà không cần khoá truy cập Copernicus hay Global Fishing Watch — điều kiện cần để người đọc tự kiểm chứng kết quả chỉ bằng một lần bấm. Thứ hai, và quan trọng hơn, bộ mô phỏng cung cấp **nhãn đối chứng chính xác tuyệt đối cho tầng hợp nhất**: trên dữ liệu thật, việc biết chắc một phương tiện có thực sự ngắt AIS hay chỉ mất sóng tạm thời là rất khó.Bộ dựng ảnh tái tạo các đặc trưng vật lý chính của ảnh radar biển: tán xạ nền Rayleigh, nhiễu đốm nhân tính theo mô hình đa nhìn, điều biến quy mô lớn do gió, tán xạ tử điểm trên thượng tầng phương tiện, vệt nước sau tàu, bóng ma phương vị và vùng đất liền.Quỹ đạo AIS được tích phân từ chuỗi gia tốc nhiễu trắng biên độ nhỏ thay vì giả định vận tốc không đổi tuyệt đối — điểm này quan trọng, vì một mô phỏng tuyến tính hoàn hảo sẽ khiến mọi phép so sánh giữa các phương pháp nội suy mất ý nghĩa.Hướng dẫn chuyển sang dữ liệu vệ tinh thật nằm trong `docs/ADAPT_NEW_DATA.md`.

In [ ]:
dataset_summary = P.step_build_data(CFG, rng)

### Xem thử một cảnh ảnhKhung màu thể hiện trạng thái định danh đối chứng của từng phương tiện.

In [ ]:
import matplotlib.pyplot as pltfrom sonarnet.data.dataset import load_image, read_scene_metafrom sonarnet.viz import figures as Fmeta_preview = read_scene_meta(CFG.dir_yolo, "test")sample = next(    (m for m in meta_preview     if any(v["identity_state"] == "DARK" for v in m["vessels"])     and len(m["vessels"]) >= 5),    meta_preview[0] if meta_preview else None,)if sample is not None:    import numpy as np    img = load_image(CFG.dir_yolo / "images" / "test" / f"{sample['scene_id']}.png")    boxes = np.array([v["bbox"] for v in sample["vessels"]], dtype=np.float32)    states = [v["identity_state"] for v in sample["vessels"]]    out = CFG.dir_figures / "00_xem_truoc.png"    F.plot_scene_with_boxes(img, boxes, states, None, out,                            title=f"Cảnh {sample['scene_id']}")    from IPython.display import Image, display    display(Image(filename=str(out)))    from collections import Counter    c = Counter(states)    print(f"Cảnh {sample['scene_id']}: {len(states)} phương tiện")    for k, v in c.items():        print(f"   {k:<14}: {v}")

---# Bước 5 — Huấn luyện mô hình phát hiệnĐây là bước tốn thời gian nhất, khoảng 14 đến 22 phút ở chế độ đầy đủ trên hai card T4.Khi có từ hai GPU trở lên và Ultralytics khả dụng, việc huấn luyện được thực hiện **phân tán trên cả hai card** thông qua một tiến trình con độc lập. Cách này ổn định hơn so với gọi trực tiếp trong nhân notebook, vì quá trình khởi tạo môi trường phân tán của Ultralytics cần một tiến trình Python riêng.Các phép tăng cường dữ liệu được điều chỉnh cho đặc thù ảnh radar: tắt hoàn toàn biến đổi màu sắc vì ảnh là đơn kênh cường độ, và cho phép xoay toàn dải 360 độ vì hướng mũi tàu là bất kỳ.

In [ ]:
train_info = P.step_train_detector(CFG)print("\nThông tin huấn luyện:")for k, v in train_info.items():    print(f"   {k}: {v}")from sonarnet.utils import gpu_utilisationutil = gpu_utilisation()if util:    print("\nTrạng thái GPU sau huấn luyện:")    print(util)

---# Bước 6 — Đánh giá tầng phát hiệnSuy luận trên tập kiểm tra rồi tính các chỉ tiêu. Khi có hai GPU, khối lượng suy luận được chia đôi cho hai card để rút ngắn thời gian.Chỉ tiêu mAP được cài đặt trực tiếp theo chuẩn PASCAL VOC, không phụ thuộc `pycocotools`, nên chạy được trong mọi môi trường.

In [ ]:
eval_out = P.step_eval_detector(CFG, split="test")dm = eval_out["metrics"]print("\n" + "=" * 62)print("CHỈ TIÊU TẦNG PHÁT HIỆN")print("=" * 62)rows = [    ("mAP@0.5",       dm.ap50,      "≥ 0,70"),    ("mAP@0.5:0.95",  dm.ap50_95,   "—"),    ("Độ chính xác",  dm.precision, "—"),    ("Độ nhạy",       dm.recall,    "≥ 0,80"),    ("F1",            dm.f1,        "—"),]for name, val, target in rows:    mark = ""    if target != "—":        thr = float(target.split()[1].replace(",", "."))        mark = "  ✔" if val >= thr else "  (chưa đạt)"    print(f"   {name:<16}: {val:.4f}   mục tiêu {target}{mark}")print(f"\n   Dương tính thật : {dm.n_tp}")print(f"   Dương tính giả  : {dm.n_fp}")print(f"   Bỏ sót          : {dm.n_fn}")

---# Bước 7 — Hợp nhất ảnh radar và tín hiệu AISĐây là tầng mang lại giá trị cốt lõi của đề tài. Quy trình gồm ba bước.**Bước một — nội suy quỹ đạo AIS về đúng thời điểm chụp ảnh.** Điểm mấu chốt về phương pháp: thời điểm chụp nằm *ở giữa* chuỗi bản ghi AIS, nên bài toán là *làm trơn* chứ không phải *lọc*. Một bộ lọc Kalman tiến thuần tuý chỉ dùng quan trắc quá khứ rồi ngoại suy, bỏ phí toàn bộ thông tin phía sau — trong thực nghiệm nó còn kém hơn cả nội suy tuyến tính. Hệ thống dùng bộ làm trơn **Rauch–Tung–Striebel**: chạy lọc tiến qua toàn bộ chuỗi, sau đó truy hồi ngược để phân bổ lại thông tin từ tương lai về quá khứ.**Bước hai — ghép cặp tối ưu toàn cục** bằng thuật toán Hungarian, với ma trận chi phí là khoảng cách địa lý và ngưỡng chặn trên 500 mét.**Bước ba — phân loại ba trạng thái định danh.** Phương tiện xuất hiện rõ trên ảnh radar mà không có bản ghi AIS tương ứng chính là dấu hiệu chủ động ngắt định danh.

In [ ]:
fusion_out = P.step_fusion(CFG, eval_out, split="test")fm = fusion_out["metrics"]print("\n" + "=" * 62)print("CHỈ TIÊU TẦNG HỢP NHẤT")print("=" * 62)print(f"   Tỉ lệ ghép cặp chính xác : {fm.match_accuracy:.4f}   mục tiêu ≥ 0,85")print(f"   Sai số ghép cặp trung vị : {fm.median_match_error_m:.1f} m")print(f"   Sai số ghép cặp trung bình: {fm.mean_match_error_m:.1f} m   mục tiêu ≤ 200 m")print()print(f"   Độ chính xác lớp DARK    : {fm.dark_precision:.4f}   mục tiêu ≥ 0,75")print(f"   Độ nhạy lớp DARK         : {fm.dark_recall:.4f}   mục tiêu ≥ 0,70")print(f"   F1 lớp DARK              : {fm.dark_f1:.4f}")print()print(f"   Độ chính xác ba trạng thái: {fm.state_accuracy:.4f}")print(f"   F1 vĩ mô ba trạng thái   : {fm.state_macro_f1:.4f}")interp = fusion_out["interpolation"]print("\n" + "-" * 62)print("SO SÁNH PHƯƠNG PHÁP NỘI SUY QUỸ ĐẠO")print("-" * 62)for key, name in (("kalman", "Làm trơn Kalman"), ("linear", "Nội suy tuyến tính")):    d = interp[key]    print(f"   {name:<20}: trung vị {d['median_error_m']:6.1f} m | "          f"trung bình {d['mean_error_m']:6.1f} m")gain = ((interp["linear"]["median_error_m"] - interp["kalman"]["median_error_m"])        / max(interp["linear"]["median_error_m"], 1e-9) * 100)print(f"\n   Bộ làm trơn cải thiện {gain:+.1f}% so với nội suy tuyến tính")

---# Bước 8 — Phân loại hành vi hoạt độngBốn nhóm hành vi được phân biệt dựa trên dấu hiệu động học bền vững, không phụ thuộc vị trí địa lý tuyệt đối.| Nhóm | Dấu hiệu động học ||---|---|| Quá cảnh | Tốc độ cao, hướng gần như không đổi || Câu | Tốc độ thấp, hướng đổi liên tục kiểu bước ngẫu nhiên, nhiều lần dừng || Kéo lưới | Tốc độ trung bình ổn định, các đoạn thẳng dài rồi quay đầu gấp || Neo đậu | Gần như đứng yên, trôi quanh một điểm theo dòng triều |Hai mươi sáu đặc trưng được trích xuất, gồm thống kê tốc độ, thống kê đổi hướng, tỉ lệ dừng, độ thẳng đường đi, bán kính hồi chuyển, và phân tích phổ chuỗi hướng — đặc trưng cuối nắm bắt tính tuần hoàn của mẫu răng lược đặc trưng cho hoạt động kéo lưới.

In [ ]:
behaviour_out = P.step_behavior(CFG, rng)br = behaviour_out["report"]print("\n" + "=" * 62)print("CHỈ TIÊU TẦNG PHÂN LOẠI HÀNH VI")print("=" * 62)print(f"   Thuật toán    : {br.backend}")print(f"   Độ chính xác  : {br.accuracy:.4f}")print(f"   F1 vĩ mô      : {br.macro_f1:.4f}   mục tiêu ≥ 0,70")print("\n   F1 theo từng nhóm:")labels = {"qua_canh": "Quá cảnh", "cau": "Câu",          "keo_luoi": "Kéo lưới", "neo_dau": "Neo đậu"}for k, v in br.per_class_f1.items():    print(f"      {labels.get(k, k):<12}: {v:.4f}")

---# Bước 9 — Phân tích đóng góp của từng thành phầnBốn cấu hình được so sánh để trả lời câu hỏi trọng tâm của đề tài: **mỗi thành phần đóng góp bao nhiêu vào năng lực phát hiện phương tiện chủ động ngắt định danh?**| Cấu hình | Nội dung ||---|---|| A | Chỉ ảnh radar — phát hiện được phương tiện nhưng không có căn cứ xác định định danh || B | Chỉ dữ liệu AIS — phương tiện ngắt tín hiệu hoàn toàn vô hình || C | Hợp nhất với nội suy tuyến tính, chưa kiểm tra kích thước khai báo || D | Hệ thống đầy đủ — làm trơn Kalman và có kiểm tra kích thước |

In [ ]:
ablation_out = P.step_ablation(CFG, eval_out, fusion_out, split="test")print(ablation_out["table"])

---# Bước 10 — Kết xuất hình minh hoạ, bản đồ và báo cáoSinh chín hình minh hoạ cho báo cáo kỹ thuật và video trình diễn, dựng bản đồ giám sát tương tác, và ghi báo cáo tổng hợp.

In [ ]:
report_out = step_report(    CFG, dataset_summary, eval_out, fusion_out,    behaviour_out, ablation_out, train_info, split="test",)print("\nSản phẩm đã kết xuất:")print(f"   Báo cáo tổng hợp : {report_out['report']}")print(f"   Bản đồ giám sát  : {report_out['dashboard']}")print(f"   Số hình minh hoạ : {len(report_out['figures'])}")

### Hiển thị các hình chính

In [ ]:
from IPython.display import Image, display, Markdownorder = [    ("scene",               "Cảnh ảnh radar kèm đối chứng và phát hiện"),    ("pr_curve",            "Đường cong chính xác – độ nhạy"),    ("fusion_confusion",    "Ma trận nhầm lẫn ba trạng thái định danh"),    ("match_errors",        "Phân bố sai số ghép cặp"),    ("interpolation",       "So sánh hai phương pháp nội suy"),    ("ablation",            "Đóng góp của từng thành phần"),    ("behaviour_confusion", "Ma trận nhầm lẫn bốn nhóm hành vi"),    ("tracks",              "Quỹ đạo tiêu biểu của từng nhóm hành vi"),    ("feature_importance",  "Đặc trưng động học quan trọng nhất"),]for key, caption in order:    path = report_out["figures"].get(key)    if path and Path(path).exists():        display(Markdown(f"**{caption}**"))        display(Image(filename=path))

### Bản đồ giám sát tương tác

In [ ]:
from IPython.display import IFrame, display, HTMLdash = Path(report_out["dashboard"])if dash.suffix == ".html" and dash.exists():    display(HTML(dash.read_text(encoding="utf-8")))elif dash.exists():    from IPython.display import Image    display(Image(filename=str(dash)))

### Báo cáo tổng hợp

In [ ]:
from IPython.display import Markdown, displaydisplay(Markdown(Path(report_out["report"]).read_text(encoding="utf-8")))

---# Bước 11 — Đóng gói kết quảGộp toàn bộ sản phẩm vào một tệp nén để tải về bằng một lần bấm. Tệp xuất hiện ở mục **Output** bên phải.

In [ ]:
import shutil, zipfilebundle = WORK_DIR / "sonarnet_ket_qua.zip"with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:    # Toàn bộ thư mục kết quả    for p in CFG.dir_results.rglob("*"):        if p.is_file():            z.write(p, f"ket_qua/{p.relative_to(CFG.dir_results)}")    # Trọng số mô hình    for w in CFG.dir_runs.rglob("*.pt"):        z.write(w, f"trong_so/{w.name}")    # Mã nguồn để nộp kèm hồ sơ    for p in SRC_DIR.rglob("*.py"):        z.write(p, f"ma_nguon/{p.relative_to(SRC_DIR)}")size_mb = bundle.stat().st_size / 1024**2print(f"Đã đóng gói: {bundle}")print(f"Dung lượng : {size_mb:.1f} MB")print("\nTệp nén nằm ở mục Output bên phải, tải về bằng một lần bấm.")

---# Tổng kếtToàn bộ quy trình đã chạy xong. Thư mục `results/` chứa:| Tệp | Nội dung ||---|---|| `BAO_CAO_KET_QUA.md` | Báo cáo tổng hợp toàn bộ chỉ tiêu || `tong_hop_ket_qua.json` | Kết quả dạng máy đọc được || `ban_do_giam_sat.html` | Bản đồ giám sát tương tác || `ablation.csv` | Bảng phân tích đóng góp thành phần || `figures/*.png` | Chín hình minh hoạ |## Bước tiếp theo**Chuyển sang dữ liệu vệ tinh thật.** Xem `docs/ADAPT_NEW_DATA.md`. Kiến trúc được thiết kế để việc thay nguồn dữ liệu chỉ tác động đến tầng thu nhận; ba tầng còn lại giữ nguyên không đổi.**Chuẩn bị hồ sơ dự thi.** Theo Thể lệ, hồ sơ Bảng C gồm tài liệu dự án PDF tối đa 20 trang, video thuyết trình, video trình diễn sản phẩm, đường dẫn kho mã nguồn kèm lịch sử thay đổi đầy đủ, Lịch sử câu lệnh, và Bản kê khai công cụ trí tuệ nhân tạo. Hai tệp `KE_KHAI_CONG_CU_AI.md` và `docs/PROMPT_LOG.md` trong kho mã nguồn hỗ trợ hai mục cuối.**Lưu ý về lịch sử thay đổi mã nguồn.** Thể lệ yêu cầu kho mã nguồn có lịch sử thay đổi đầy đủ và nghiêm cấm việc dựng lại lịch sử này về sau. Hãy tạo kho mã và ghi nhận thay đổi thường xuyên ngay từ đầu.---## Ghi chú về phạm vi sử dụngSản phẩm này là công cụ hỗ trợ nghiên cứu khoa học. Toàn bộ dữ liệu sử dụng đều là dữ liệu mở, được cung cấp công khai cho mục đích nghiên cứu và học thuật.Hệ thống không tham gia vào bất kỳ quy trình ra quyết định thực thi pháp luật nào, không thực hiện theo dõi cá nhân, và không đưa ra kết luận pháp lý về hành vi của bất kỳ phương tiện hay tổ chức nào. Mọi kết quả phát hiện đều mang tính chất tín hiệu cảnh báo kỹ thuật, cần được cơ quan có thẩm quyền kiểm chứng độc lập.